# Notebook 5 — Lightweight CNN Optimisation

## Lightweight ECG Signal Classification Using Machine Learning and Deep Learning Models

This notebook continues the experimental workflow established in Notebook 4 and focuses exclusively on the optimisation of the selected Lightweight Convolutional Neural Network (CNN).

Notebook 4 evaluated the baseline machine-learning and deep-learning models and selected the Lightweight CNN based on validation performance. The selected CNN is therefore treated as the frozen baseline for the experiments conducted in this notebook.

The objective of Notebook 5 is to investigate whether the selected CNN can be made more computationally efficient while retaining acceptable predictive performance. The optimisation stage will investigate software-based techniques including model pruning and quantisation, where technically feasible and methodologically justified.

All optimisation decisions will be made using the training and validation data only. The participant-level test set established in Notebook 4 will remain frozen and will not be used for optimisation, hyperparameter selection, or repeated model comparison. The test set will be evaluated only after the final optimised CNN has been selected.

The central research question addressed by this notebook is:

> **How effectively can the selected Lightweight CNN be optimised to improve computational efficiency while retaining acceptable predictive performance?**

The results of this notebook will provide the final optimised CNN and the performance-efficiency evidence required for the subsequent analysis in Notebook 6.


In [ ]:
#  Cell 1
# Environment and reproducibility setup

import os
import random
import time
import numpy as np
import pandas as pd
import tensorflow as tf

# Reproducibility
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Reduce unnecessary TensorFlow logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Random seed:", SEED)

TensorFlow version: 2.20.0
NumPy version: 2.0.2
Pandas version: 2.2.2
Random seed: 42


In [ ]:
#  Cell 2
# Load frozen Notebook 3 data exports

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Notebook 3 export locations
# ------------------------------------------------------------

EXPORT_DIR = (
    "/content/drive/MyDrive/"
    "QMUL_MSc_Dissertation/"
    "Notebook3_Exports"
)

ML_FILE = os.path.join(
    EXPORT_DIR,
    "ML_feature_dataset.csv"
)

DL_FILE = os.path.join(
    EXPORT_DIR,
    "DL_window_metadata.csv"
)

FEATURE_FILE = os.path.join(
    EXPORT_DIR,
    "feature_columns.txt"
)

# ------------------------------------------------------------
# Verify required files
# ------------------------------------------------------------

required_files = {
    "ML feature dataset": ML_FILE,
    "DL window metadata": DL_FILE,
    "Feature list": FEATURE_FILE
}

print("=" * 70)
print("CHECKING NOTEBOOK 3 EXPORTS")
print("=" * 70)

for name, path in required_files.items():
    status = "FOUND" if os.path.exists(path) else "NOT FOUND"
    print(f"{name:<25}: {status}")

missing_files = [
    name
    for name, path in required_files.items()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "Required Notebook 3 export(s) missing:\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------------------
# Load exported datasets
# ------------------------------------------------------------

ml_df = pd.read_csv(ML_FILE)
dl_df = pd.read_csv(DL_FILE)

# Load feature names
with open(FEATURE_FILE, "r") as f:
    feature_columns = [
        line.strip()
        for line in f
        if line.strip()
    ]

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NOTEBOOK 3 EXPORTS LOADED")
print("=" * 70)

print(f"ML dataset shape:       {ml_df.shape}")
print(f"DL metadata shape:      {dl_df.shape}")
print(f"Number of ML features:  {len(feature_columns)}")
print(f"DL participants:        {dl_df['ParticipantID'].nunique()}")
print(f"DL windows:             {len(dl_df):,}")

print("\nFeature columns:")
print(feature_columns)

print("\nDL metadata columns:")
print(dl_df.columns.tolist())

Mounted at /content/drive
CHECKING NOTEBOOK 3 EXPORTS
ML feature dataset       : FOUND
DL window metadata       : FOUND
Feature list             : FOUND

NOTEBOOK 3 EXPORTS LOADED
ML dataset shape:       (421166, 15)
DL metadata shape:      (421166, 10)
Number of ML features:  9
DL participants:        30
DL windows:             421,166

Feature columns:
['Mean', 'Std', 'Min', 'Max', 'Range', 'Median', 'IQR', 'RMS', 'Energy']

DL metadata columns:
['WindowID', 'ParticipantID', 'ECGFile', 'StartTime', 'EndTime', 'WakeSleepLabel', 'Path', 'ActualFirstTimestamp', 'StartSample', 'EndSample']


## 2. Frozen CNN Baseline

Notebook 4 established the Lightweight CNN as the selected deep-learning model based on validation performance. This section reconstructs that model as the fixed reference point for all subsequent optimisation experiments.

The baseline architecture is retained without modification. It consists of two one-dimensional convolutional layers followed by global average pooling and a compact fully connected classification head. The model accepts 2,500-sample ECG waveform segments corresponding to 10-second windows recorded at 250 Hz.

The frozen CNN contains **3,809 trainable parameters**.

Before applying any optimisation technique, the baseline architecture, parameter count, input shape, and computational characteristics will be verified. The baseline will then provide the reference against which pruning and quantisation experiments are evaluated.

All optimisation decisions in this notebook will be based on the training and validation data. The held-out test participants from Notebook 4 remain untouched until the final optimised model has been selected.


In [ ]:
# Cell 3
# Reconstruct and load the frozen Notebook 4 CNN baseline

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

# ------------------------------------------------------------
# Frozen CNN configuration from Notebook 4
# ------------------------------------------------------------

CNN_INPUT_SAMPLES = 2500

CNN_CHECKPOINT_DIR = (
    "/content/drive/MyDrive/"
    "QMUL_MSc_Dissertation/"
    "Notebook4_CNN_Checkpoints"
)

CNN_CHECKPOINT_PATH = os.path.join(
    CNN_CHECKPOINT_DIR,
    "cnn_best.weights.h5"
)

print("=" * 70)
print("LOADING FROZEN NOTEBOOK 4 CNN")
print("=" * 70)

print(f"\nCheckpoint path:")
print(CNN_CHECKPOINT_PATH)

if not os.path.exists(CNN_CHECKPOINT_PATH):
    raise FileNotFoundError(
        "Frozen Notebook 4 CNN checkpoint was not found.\n"
        f"Expected location:\n{CNN_CHECKPOINT_PATH}\n\n"
        "Do NOT retrain Notebook 4. Verify the checkpoint location instead."
    )

# ------------------------------------------------------------
# Reconstruct the exact Notebook 4 architecture
# ------------------------------------------------------------

baseline_cnn = Sequential([
    Input(
        shape=(CNN_INPUT_SAMPLES, 1)
    ),

    Conv1D(
        filters=16,
        kernel_size=7,
        activation="relu",
        padding="same"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Conv1D(
        filters=32,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),

    GlobalAveragePooling1D(),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(
        0.3
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

# ------------------------------------------------------------
# Compile using the same Notebook 4 configuration
# ------------------------------------------------------------

baseline_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

# ------------------------------------------------------------
# Load the frozen Notebook 4 weights
# ------------------------------------------------------------

baseline_cnn.load_weights(
    CNN_CHECKPOINT_PATH
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FROZEN CNN VERIFICATION")
print("=" * 70)

print(
    f"\nTrainable parameters: "
    f"{baseline_cnn.count_params():,}"
)

print(
    f"Input shape: "
    f"{baseline_cnn.input_shape}"
)

print(
    f"Checkpoint loaded: "
    f"{os.path.basename(CNN_CHECKPOINT_PATH)}"
)

# Strict architecture checks
if baseline_cnn.count_params() != 3809:
    raise ValueError(
        f"Unexpected parameter count: "
        f"{baseline_cnn.count_params():,}. "
        "Expected 3,809."
    )

if baseline_cnn.input_shape != (None, 2500, 1):
    raise ValueError(
        f"Unexpected input shape: "
        f"{baseline_cnn.input_shape}. "
        "Expected (None, 2500, 1)."
    )

print("\nParameter count check: PASSED")
print("Input shape check: PASSED")
print("Frozen CNN weights: LOADED")
print("\n" + "=" * 70)
print("FROZEN NOTEBOOK 4 CNN IS READY FOR NOTEBOOK 5")
print("=" * 70)

LOADING FROZEN NOTEBOOK 4 CNN

Checkpoint path:
/content/drive/MyDrive/QMUL_MSc_Dissertation/Notebook4_CNN_Checkpoints/cnn_best.weights.h5

FROZEN CNN VERIFICATION

Trainable parameters: 3,809
Input shape: (None, 2500, 1)
Checkpoint loaded: cnn_best.weights.h5

Parameter count check: PASSED
Input shape check: PASSED
Frozen CNN weights: LOADED

FROZEN NOTEBOOK 4 CNN IS READY FOR NOTEBOOK 5


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
#  4
# Record frozen CNN baseline configuration

BASELINE_CNN_INFO = {
    "Model": "Lightweight CNN",
    "Input samples": 2500,
    "Trainable parameters": baseline_cnn.count_params(),
    "Conv1D filters": "16 -> 32",
    "Pooling": "MaxPooling1D",
    "Global pooling": "GlobalAveragePooling1D",
    "Dense units": 32,
    "Dropout": 0.3,
    "Optimizer": "Adam",
    "Learning rate": 0.001,
    "Loss": "Binary Crossentropy",
    "Checkpoint": CNN_CHECKPOINT_PATH,
    "Status": "FROZEN BASELINE"
}

baseline_cnn_info_df = pd.DataFrame(
    [BASELINE_CNN_INFO]
)

print("=" * 80)
print("FROZEN CNN BASELINE — NOTEBOOK 5")
print("=" * 80)

display(baseline_cnn_info_df)

print("\nBaseline model is frozen.")
print("No retraining or architecture modification will be performed.")
print("=" * 80)

FROZEN CNN BASELINE — NOTEBOOK 5


,Model,Input samples,Trainable parameters,Conv1D filters,Pooling,Global pooling,Dense units,Dropout,Optimizer,Learning rate,Loss,Checkpoint,Status
0,Lightweight CNN,2500,3809,16 -> 32,MaxPooling1D,GlobalAveragePooling1D,32,0.3,Adam,0.001,Binary Crossentropy,/content/drive/MyDrive/QMUL_MSc_Dissertation/N...,FROZEN BASELINE



Baseline model is frozen.
No retraining or architecture modification will be performed.


In [ ]:
# Cell 5
# Baseline efficiency measurement

print("=" * 80)
print("BASELINE CNN — EFFICIENCY MEASUREMENT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Parameter count
# ------------------------------------------------------------

baseline_parameters = baseline_cnn.count_params()

# ------------------------------------------------------------
# 2. Model weight storage size
# ------------------------------------------------------------

baseline_weights_bytes = sum(
    weight.numpy().nbytes
    for weight in baseline_cnn.weights
)

baseline_weights_kb = baseline_weights_bytes / 1024
baseline_weights_mb = baseline_weights_kb / 1024

# ------------------------------------------------------------
# 3. Non-zero parameter count
# ------------------------------------------------------------

baseline_nonzero_parameters = sum(
    np.count_nonzero(weight.numpy())
    for weight in baseline_cnn.weights
)

baseline_sparsity = (
    1 -
    baseline_nonzero_parameters / baseline_parameters
)

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

baseline_efficiency = pd.DataFrame({
    "Metric": [
        "Trainable parameters",
        "Non-zero parameters",
        "Sparsity",
        "Weight storage (KB)",
        "Weight storage (MB)"
    ],
    "Baseline CNN": [
        baseline_parameters,
        baseline_nonzero_parameters,
        f"{baseline_sparsity:.4%}",
        f"{baseline_weights_kb:.2f}",
        f"{baseline_weights_mb:.4f}"
    ]
})

display(baseline_efficiency)

print("\nBaseline efficiency measurement completed.")
print("No model weights were modified.")

BASELINE CNN — EFFICIENCY MEASUREMENT


,Metric,Baseline CNN
0,Trainable parameters,3809
1,Non-zero parameters,3808
2,Sparsity,0.0263%
3,Weight storage (KB),14.88
4,Weight storage (MB),0.0145



Baseline efficiency measurement completed.
No model weights were modified.


## 3. Pruning Methodology

Model pruning is investigated as the first optimisation technique in this notebook. The objective is to determine whether unnecessary model weights can be removed while maintaining acceptable predictive performance.

The frozen Lightweight CNN from Notebook 4 is used as the reference model. Pruning will be applied systematically at predefined sparsity levels rather than selecting a pruning level after observing the results. Each candidate will be trained or fine-tuned using the training data and evaluated using the validation data.

The pruning experiments will consider the relationship between increasing sparsity and predictive performance. For each candidate model, validation ROC-AUC, F1-score, precision, recall, accuracy, parameter sparsity, and model size will be recorded where applicable.

The held-out test set will remain completely unused during pruning and candidate selection. The test set will only be evaluated once the final optimised model has been selected.

The purpose of this experiment is therefore not simply to maximise sparsity, but to identify a useful trade-off between computational efficiency and predictive performance.

In [ ]:
#  Cell 6
# Verify pruning optimisation environment

print("=" * 80)
print("PRUNING ENVIRONMENT CHECK")
print("=" * 80)

try:
    import tensorflow_model_optimization as tfmot

    print("\nTensorFlow Model Optimization: AVAILABLE")
    print("TF-MOT version:", tfmot.__version__)

    PRUNING_AVAILABLE = True

except ImportError:
    print("\nTensorFlow Model Optimization: NOT AVAILABLE")
    print(
        "Pruning experiments cannot be started until "
        "the required package is available."
    )

    PRUNING_AVAILABLE = False

print("\nTensorFlow version:", tf.__version__)
print("Pruning available:", PRUNING_AVAILABLE)

print("\n" + "=" * 80)

PRUNING ENVIRONMENT CHECK

TensorFlow Model Optimization: NOT AVAILABLE
Pruning experiments cannot be started until the required package is available.

TensorFlow version: 2.20.0
Pruning available: False



In [ ]:
# Cell 7

# Prepare TensorFlow Model Optimization Toolkit

if not PRUNING_AVAILABLE:
    print("Installing TensorFlow Model Optimization Toolkit...")

    %pip install -q tensorflow-model-optimization

    import tensorflow_model_optimization as tfmot

    PRUNING_AVAILABLE = True

    print("\nTensorFlow Model Optimization installed successfully.")
    print("TF-MOT version:", tfmot.__version__)

else:
    print("TensorFlow Model Optimization is already available.")
    print("TF-MOT version:", tfmot.__version__)

print("\nPruning toolkit ready:", PRUNING_AVAILABLE)

Installing TensorFlow Model Optimization Toolkit...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 15.7 MB/s eta 0:00:00

TensorFlow Model Optimization installed successfully.
TF-MOT version: 0.8.1

Pruning toolkit ready: True


In [ ]:
# Cell 8

# Define pruning experiment configuration

PRUNING_SPARSE_LEVELS = [
    0.20,
    0.40,
    0.60,
    0.80
]

PRUNING_BEGIN_STEP = 0
PRUNING_END_STEP = 1000

PRUNING_BATCH_SIZE = 64
PRUNING_EPOCHS = 5

PRUNING_METRICS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

print("=" * 80)
print("PRUNING EXPERIMENT CONFIGURATION")
print("=" * 80)

print("\nPruning method:")
print("Magnitude-based unstructured weight pruning")

print("\nCandidate sparsity levels:")
for level in PRUNING_SPARSE_LEVELS:
    print(f"  {level:.0%}")

print(f"\nBatch size:       {PRUNING_BATCH_SIZE}")
print(f"Maximum epochs:   {PRUNING_EPOCHS}")
print(f"Begin step:       {PRUNING_BEGIN_STEP}")
print(f"End step:         {PRUNING_END_STEP}")

print("\nEvaluation metrics:")
for metric in PRUNING_METRICS:
    print(f"  - {metric}")

print("\nFrozen baseline parameters:")
print(f"  {baseline_cnn.count_params():,}")

print("\nNo model weights have been modified.")
print("=" * 80)

PRUNING EXPERIMENT CONFIGURATION

Pruning method:
Magnitude-based unstructured weight pruning

Candidate sparsity levels:
  20%
  40%
  60%
  80%

Batch size:       64
Maximum epochs:   5
Begin step:       0
End step:         1000

Evaluation metrics:
  - accuracy
  - precision
  - recall
  - f1
  - roc_auc

Frozen baseline parameters:
  3,809

No model weights have been modified.


In [ ]:
# Cell 9

# Recreate the frozen participant-level split from Notebook 4

print("=" * 80)
print("RECREATING FROZEN PARTICIPANT-LEVEL SPLIT")
print("=" * 80)

# ------------------------------------------------------------
# Get participant-level information
# ------------------------------------------------------------

participant_ids = (
    ml_df["ParticipantID"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print(f"\nTotal participants: {len(participant_ids)}")

# ------------------------------------------------------------
# Reproduce the exact Notebook 4 participant shuffle
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

shuffled_participants = participant_ids.copy()
rng.shuffle(shuffled_participants)

# ------------------------------------------------------------
# Reproduce the exact Notebook 4 split proportions
# ------------------------------------------------------------

n_participants = len(shuffled_participants)

n_train = round(
    n_participants * 0.70
)

n_validation = round(
    n_participants * 0.15
)

n_test = (
    n_participants
    - n_train
    - n_validation
)

train_participants = sorted(
    shuffled_participants[
        :n_train
    ]
)

validation_participants = sorted(
    shuffled_participants[
        n_train:n_train + n_validation
    ]
)

test_participants = sorted(
    shuffled_participants[
        n_train + n_validation:
    ]
)

# ------------------------------------------------------------
# Assign split to every ML and DL window
# ------------------------------------------------------------

def assign_split(participant_id):

    if participant_id in train_participants:
        return "train"

    elif participant_id in validation_participants:
        return "validation"

    elif participant_id in test_participants:
        return "test"

    else:
        return "unassigned"


ml_df["Split"] = (
    ml_df["ParticipantID"]
    .apply(assign_split)
)

dl_df["Split"] = (
    dl_df["ParticipantID"]
    .apply(assign_split)
)

# ------------------------------------------------------------
# Participant-level verification
# ------------------------------------------------------------

train_set = set(
    train_participants
)

validation_set = set(
    validation_participants
)

test_set = set(
    test_participants
)

participant_overlap = {
    "Train ∩ Validation":
        train_set.intersection(validation_set),

    "Train ∩ Test":
        train_set.intersection(test_set),

    "Validation ∩ Test":
        validation_set.intersection(test_set)
}

# ------------------------------------------------------------
# Window-level verification
# ------------------------------------------------------------

train_window_ids = set(
    ml_df.loc[
        ml_df["Split"] == "train",
        "WindowID"
    ]
)

validation_window_ids = set(
    ml_df.loc[
        ml_df["Split"] == "validation",
        "WindowID"
    ]
)

test_window_ids = set(
    ml_df.loc[
        ml_df["Split"] == "test",
        "WindowID"
    ]
)

window_overlap = {
    "Train ∩ Validation":
        train_window_ids.intersection(
            validation_window_ids
        ),

    "Train ∩ Test":
        train_window_ids.intersection(
            test_window_ids
        ),

    "Validation ∩ Test":
        validation_window_ids.intersection(
            test_window_ids
        )
}

# ------------------------------------------------------------
# Assignment verification
# ------------------------------------------------------------

unassigned_ml = (
    ml_df["Split"] == "unassigned"
).sum()

unassigned_dl = (
    dl_df["Split"] == "unassigned"
).sum()

total_assigned = (
    len(train_window_ids)
    + len(validation_window_ids)
    + len(test_window_ids)
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nParticipant allocation")
print("-" * 80)

print(
    f"Training participants:   "
    f"{len(train_participants)}"
)

print(
    f"Validation participants: "
    f"{len(validation_participants)}"
)

print(
    f"Test participants:       "
    f"{len(test_participants)}"
)

print("\nTraining participants:")
print(train_participants)

print("\nValidation participants:")
print(validation_participants)

print("\nTest participants:")
print(test_participants)

print("\nWindow allocation")
print("-" * 80)

print(
    f"Training windows:   "
    f"{len(train_window_ids):,}"
)

print(
    f"Validation windows: "
    f"{len(validation_window_ids):,}"
)

print(
    f"Test windows:       "
    f"{len(test_window_ids):,}"
)

print(
    f"Total assigned:     "
    f"{total_assigned:,}"
)

print(
    f"Total ML windows:   "
    f"{len(ml_df):,}"
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

participant_separation_passed = all(
    len(overlap) == 0
    for overlap in participant_overlap.values()
)

window_separation_passed = all(
    len(overlap) == 0
    for overlap in window_overlap.values()
)

all_windows_assigned = (
    total_assigned == len(ml_df)
)

no_unassigned_windows = (
    unassigned_ml == 0
    and unassigned_dl == 0
)

split_reconstruction_passed = (
    participant_separation_passed
    and window_separation_passed
    and all_windows_assigned
    and no_unassigned_windows
)

print("\n" + "=" * 80)

print(
    "Participant separation:",
    "PASSED"
    if participant_separation_passed
    else "FAILED"
)

print(
    "Window separation:",
    "PASSED"
    if window_separation_passed
    else "FAILED"
)

print(
    "All windows assigned:",
    "PASSED"
    if all_windows_assigned
    else "FAILED"
)

print(
    "No unassigned windows:",
    "PASSED"
    if no_unassigned_windows
    else "FAILED"
)

print("-" * 80)

print(
    "FROZEN SPLIT RECONSTRUCTION:",
    "PASSED"
    if split_reconstruction_passed
    else "FAILED"
)

print("=" * 80)

RECREATING FROZEN PARTICIPANT-LEVEL SPLIT

Total participants: 30

Participant allocation
--------------------------------------------------------------------------------
Training participants:   21
Validation participants: 4
Test participants:       5

Training participants:
[1, 4, 6, 7, 8, 10, 11, 13, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 30]

Validation participants:
[3, 12, 15, 29]

Test participants:
[2, 5, 9, 14, 28]

Window allocation
--------------------------------------------------------------------------------
Training windows:   297,867
Validation windows: 57,718
Test windows:       65,581
Total assigned:     421,166
Total ML windows:   421,166

Participant separation: PASSED
Window separation: PASSED
All windows assigned: PASSED
No unassigned windows: PASSED
--------------------------------------------------------------------------------
FROZEN SPLIT RECONSTRUCTION: PASSED


In [ ]:
# Cell 10

# Create split-specific datasets and build the ECG batch pipeline

print("=" * 80)
print("BUILDING ECG BATCH PIPELINE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects exist
# ------------------------------------------------------------

required_names = [
    "dl_df",
    "SEED",
    "PRUNING_BATCH_SIZE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
        + "."
    )

# ------------------------------------------------------------
# 2. Create split-specific DL metadata
# ------------------------------------------------------------

required_columns = {
    "ParticipantID",
    "WindowID",
    "WakeSleepLabel",
    "Path",
    "StartSample",
    "EndSample",
    "Split"
}

missing_columns = (
    required_columns
    - set(dl_df.columns)
)

if missing_columns:
    raise ValueError(
        "Required DL metadata columns are missing: "
        + ", ".join(sorted(missing_columns))
    )

dl_train = (
    dl_df.loc[
        dl_df["Split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

dl_validation = (
    dl_df.loc[
        dl_df["Split"] == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

dl_test = (
    dl_df.loc[
        dl_df["Split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Validate split sizes
# ------------------------------------------------------------

for split_name, split_df in {
    "training": dl_train,
    "validation": dl_validation,
    "test": dl_test
}.items():

    if split_df.empty:
        raise ValueError(
            f"The {split_name} split is empty."
        )

# ------------------------------------------------------------
# 4. Fixed ECG configuration
# ------------------------------------------------------------

WINDOW_SAMPLES = 2500

if WINDOW_SAMPLES != CNN_INPUT_SAMPLES:
    raise ValueError(
        "Window size does not match the frozen CNN "
        "input size."
    )

if PRUNING_BATCH_SIZE <= 0:
    raise ValueError(
        "PRUNING_BATCH_SIZE must be greater than zero."
    )

# Original Notebook 4 class weights
CNN_CLASS_WEIGHTS = {
    0: 2.3905,
    1: 0.6322
}

# ------------------------------------------------------------
# 5. Load waveform cache index
# ------------------------------------------------------------

WAVEFORM_CACHE_INDEX = (
    "/content/drive/MyDrive/"
    "QMUL_MSc_Dissertation/"
    "Notebook4_Waveform_Cache/"
    "waveform_cache_index.csv"
)

if not os.path.isfile(WAVEFORM_CACHE_INDEX):
    raise FileNotFoundError(
        "Waveform cache index was not found:\n"
        f"{WAVEFORM_CACHE_INDEX}"
    )

waveform_cache_index = pd.read_csv(
    WAVEFORM_CACHE_INDEX
)

required_cache_columns = {
    "Path",
    "CachePath"
}

missing_cache_columns = (
    required_cache_columns
    - set(waveform_cache_index.columns)
)

if missing_cache_columns:
    raise ValueError(
        "Required cache-index columns are missing: "
        + ", ".join(sorted(missing_cache_columns))
    )

if waveform_cache_index.empty:
    raise ValueError(
        "Waveform cache index is empty."
    )

# ------------------------------------------------------------
# 6. Build cache lookup
# ------------------------------------------------------------

cache_path_lookup = dict(
    zip(
        waveform_cache_index["Path"],
        waveform_cache_index["CachePath"]
    )
)

required_ecg_paths = set(
    dl_df["Path"].unique()
)

missing_cache_entries = (
    required_ecg_paths
    - set(cache_path_lookup.keys())
)

if missing_cache_entries:
    raise FileNotFoundError(
        "Some ECG recordings are missing from "
        "the waveform cache index."
    )

# ------------------------------------------------------------
# 7. Open cached waveforms as read-only memory maps
# ------------------------------------------------------------

waveform_memmaps = {}

for ecg_path in sorted(required_ecg_paths):

    cache_path = cache_path_lookup[ecg_path]

    if not os.path.isfile(cache_path):
        raise FileNotFoundError(
            f"Cached waveform file not found:\n"
            f"{cache_path}"
        )

    waveform = np.load(
        cache_path,
        mmap_mode="r"
    )

    if waveform.ndim != 1:
        raise ValueError(
            f"Expected one-dimensional ECG waveform "
            f"for {ecg_path}, got {waveform.shape}."
        )

    if len(waveform) == 0:
        raise ValueError(
            f"ECG waveform is empty: {ecg_path}"
        )

    waveform_memmaps[ecg_path] = waveform

# ------------------------------------------------------------
# 8. Validate metadata sample boundaries
# ------------------------------------------------------------

for split_name, split_df in {
    "train": dl_train,
    "validation": dl_validation,
    "test": dl_test
}.items():

    for row in split_df[
        ["Path", "StartSample", "EndSample"]
    ].itertuples(index=False):

        start_sample = int(row.StartSample)
        end_sample = int(row.EndSample)

        waveform_length = len(
            waveform_memmaps[row.Path]
        )

        if start_sample < 0:
            raise ValueError(
                f"Negative StartSample in {split_name}."
            )

        if end_sample <= start_sample:
            raise ValueError(
                f"Invalid sample range in {split_name}."
            )

        if end_sample > waveform_length:
            raise ValueError(
                f"EndSample exceeds waveform length "
                f"in {split_name}."
            )

        if (
            end_sample - start_sample
            != WINDOW_SAMPLES
        ):
            raise ValueError(
                f"Unexpected ECG window length in "
                f"{split_name}."
            )

# ------------------------------------------------------------
# 9. ECG batch generator
# ------------------------------------------------------------

def generate_ecg_batches_fast(
    metadata_df,
    batch_size=64,
    shuffle=False
):
    """
    Generate standardised ECG windows.

    Each ECG window is independently standardised using
    its own mean and standard deviation.
    """

    if metadata_df.empty:
        raise ValueError(
            "Cannot generate batches from an empty dataset."
        )

    if batch_size <= 0:
        raise ValueError(
            "batch_size must be greater than zero."
        )

    indices = np.arange(
        len(metadata_df)
    )

    rng = np.random.default_rng(
        SEED
    )

    while True:

        if shuffle:
            rng.shuffle(indices)

        for start in range(
            0,
            len(indices),
            batch_size
        ):

            batch_indices = indices[
                start:start + batch_size
            ]

            batch_rows = metadata_df.iloc[
                batch_indices
            ]

            batch_size_actual = len(
                batch_rows
            )

            X_batch = np.empty(
                (
                    batch_size_actual,
                    WINDOW_SAMPLES,
                    1
                ),
                dtype=np.float32
            )

            y_batch = (
                batch_rows[
                    "WakeSleepLabel"
                ]
                .to_numpy(
                    dtype=np.int32
                )
            )

            for i, row in enumerate(
                batch_rows.itertuples(
                    index=False
                )
            ):

                waveform = waveform_memmaps[
                    row.Path
                ]

                start_sample = int(
                    row.StartSample
                )

                end_sample = int(
                    row.EndSample
                )

                window = np.asarray(
                    waveform[
                        start_sample:end_sample
                    ],
                    dtype=np.float32
                )

                if len(window) != WINDOW_SAMPLES:
                    raise ValueError(
                        "Unexpected ECG window length."
                    )

                if not np.isfinite(window).all():
                    raise ValueError(
                        "Non-finite ECG values detected."
                    )

                window_mean = window.mean()
                window_std = window.std()

                if window_std > 0.0:
                    window = (
                        window - window_mean
                    ) / window_std
                else:
                    window = (
                        window - window_mean
                    )

                X_batch[
                    i, :, 0
                ] = window

            yield X_batch, y_batch


# ------------------------------------------------------------
# 10. Weighted training generator
# ------------------------------------------------------------

def generate_weighted_cnn_batches(
    metadata_df,
    batch_size=64,
    shuffle=True
):

    base_generator = generate_ecg_batches_fast(
        metadata_df=metadata_df,
        batch_size=batch_size,
        shuffle=shuffle
    )

    while True:

        X_batch, y_batch = next(
            base_generator
        )

        sample_weights = np.where(
            y_batch == 0,
            CNN_CLASS_WEIGHTS[0],
            CNN_CLASS_WEIGHTS[1]
        ).astype(np.float32)

        yield (
            X_batch,
            y_batch,
            sample_weights
        )


# ------------------------------------------------------------
# 11. Create generators
# ------------------------------------------------------------

pruning_train_generator = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

pruning_validation_generator = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 12. Verify one real training batch
# ------------------------------------------------------------

X_train_check, y_train_check, weights_check = next(
    pruning_train_generator
)

# ------------------------------------------------------------
# 13. Verify one real validation batch
# ------------------------------------------------------------

X_validation_check, y_validation_check = next(
    pruning_validation_generator
)

# ------------------------------------------------------------
# 14. Validate batch outputs
# ------------------------------------------------------------

train_batch_valid = (
    X_train_check.ndim == 3
    and
    X_train_check.shape[1:] == (
        WINDOW_SAMPLES,
        1
    )
    and
    y_train_check.shape[0]
    == X_train_check.shape[0]
    and
    weights_check.shape[0]
    == X_train_check.shape[0]
    and
    np.isfinite(X_train_check).all()
    and
    np.isfinite(weights_check).all()
)

validation_batch_valid = (
    X_validation_check.ndim == 3
    and
    X_validation_check.shape[1:] == (
        WINDOW_SAMPLES,
        1
    )
    and
    y_validation_check.shape[0]
    == X_validation_check.shape[0]
    and
    np.isfinite(X_validation_check).all()
)

labels_valid = (
    set(
        np.unique(y_train_check)
    ).issubset({0, 1})
    and
    set(
        np.unique(y_validation_check)
    ).issubset({0, 1})
)

# ------------------------------------------------------------
# 15. Final pipeline status
# ------------------------------------------------------------

pipeline_ready = (
    train_batch_valid
    and
    validation_batch_valid
    and
    labels_valid
)

print("\nSplit sizes")
print("-" * 80)

print(
    f"Training windows:   {len(dl_train):,}"
)

print(
    f"Validation windows: {len(dl_validation):,}"
)

print(
    f"Test windows:       {len(dl_test):,}"
)

print(
    f"Cached recordings:  {len(waveform_memmaps):,}"
)

print("\nTraining batch")
print("-" * 80)

print(
    f"X shape:       {X_train_check.shape}"
)

print(
    f"y shape:       {y_train_check.shape}"
)

print(
    f"weights shape: {weights_check.shape}"
)

print(
    f"X mean:        {X_train_check.mean():.6f}"
)

print(
    f"X std:         {X_train_check.std():.6f}"
)

print("\nValidation batch")
print("-" * 80)

print(
    f"X shape: {X_validation_check.shape}"
)

print(
    f"y shape: {y_validation_check.shape}"
)

print("\n" + "=" * 80)

print(
    "Training batch:",
    "PASSED"
    if train_batch_valid
    else "FAILED"
)

print(
    "Validation batch:",
    "PASSED"
    if validation_batch_valid
    else "FAILED"
)

print(
    "Binary labels:",
    "PASSED"
    if labels_valid
    else "FAILED"
)

print(
    "ECG BATCH PIPELINE:",
    "READY"
    if pipeline_ready
    else "FAILED"
)

print("=" * 80)

if not pipeline_ready:
    raise RuntimeError(
        "ECG batch pipeline validation failed."
    )

BUILDING ECG BATCH PIPELINE

Split sizes
--------------------------------------------------------------------------------
Training windows:   297,867
Validation windows: 57,718
Test windows:       65,581
Cached recordings:  34

Training batch
--------------------------------------------------------------------------------
X shape:       (64, 2500, 1)
y shape:       (64,)
weights shape: (64,)
X mean:        0.000001
X std:         1.000000

Validation batch
--------------------------------------------------------------------------------
X shape: (64, 2500, 1)
y shape: (64,)

Training batch: PASSED
Validation batch: PASSED
Binary labels: PASSED
ECG BATCH PIPELINE: READY


In [ ]:
# Cell 11

# Evaluate the frozen CNN on the validation split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 80)
print("FROZEN CNN — VALIDATION BASELINE EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects exist
# ------------------------------------------------------------

required_names = [
    "baseline_cnn",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE",
    "pipeline_ready"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pipeline_ready:
    raise RuntimeError(
        "Cell 10 did not pass pipeline validation."
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

# ------------------------------------------------------------
# 2. Create a FRESH validation generator
# ------------------------------------------------------------

validation_generator = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 3. Determine exact validation size
# ------------------------------------------------------------

validation_samples = len(
    dl_validation
)

validation_steps = int(
    np.ceil(
        validation_samples
        / PRUNING_BATCH_SIZE
    )
)

if validation_steps <= 0:
    raise ValueError(
        "Validation step count must be positive."
    )

# ------------------------------------------------------------
# 4. Generate predictions
# ------------------------------------------------------------

validation_start_time = time.perf_counter()

validation_probabilities = (
    baseline_cnn.predict(
        validation_generator,
        steps=validation_steps,
        verbose=0
    )
)

validation_prediction_time = (
    time.perf_counter()
    - validation_start_time
)

# ------------------------------------------------------------
# 5. Convert predictions to one-dimensional array
# ------------------------------------------------------------

validation_probabilities = np.asarray(
    validation_probabilities,
    dtype=np.float64
).reshape(-1)

# Because the final batch may contain fewer than
# PRUNING_BATCH_SIZE samples, retain exactly the
# number of validation windows.
validation_probabilities = (
    validation_probabilities[
        :validation_samples
    ]
)

# ------------------------------------------------------------
# 6. Validate prediction count and values
# ------------------------------------------------------------

if len(validation_probabilities) != validation_samples:
    raise RuntimeError(
        "Prediction count does not match validation "
        "sample count."
    )

if not np.isfinite(
    validation_probabilities
).all():
    raise ValueError(
        "Non-finite prediction probabilities detected."
    )

if (
    (validation_probabilities < 0.0)
    |
    (validation_probabilities > 1.0)
).any():
    raise ValueError(
        "Prediction probabilities outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 7. Obtain validation labels
# ------------------------------------------------------------

validation_labels = (
    dl_validation[
        "WakeSleepLabel"
    ]
    .to_numpy(
        dtype=np.int32
    )
)

if len(validation_labels) != validation_samples:
    raise RuntimeError(
        "Validation label count does not match "
        "validation sample count."
    )

if not set(
    np.unique(validation_labels)
).issubset({0, 1}):
    raise ValueError(
        "Validation labels are not binary."
    )

if len(
    np.unique(validation_labels)
) < 2:
    raise ValueError(
        "ROC-AUC cannot be calculated because "
        "the validation set contains only one class."
    )

# ------------------------------------------------------------
# 8. Convert probabilities to class predictions
# ------------------------------------------------------------

VALIDATION_THRESHOLD = 0.50

validation_predictions = (
    validation_probabilities
    >= VALIDATION_THRESHOLD
).astype(np.int32)

# ------------------------------------------------------------
# 9. Calculate validation metrics
# ------------------------------------------------------------

validation_accuracy = accuracy_score(
    validation_labels,
    validation_predictions
)

validation_precision = precision_score(
    validation_labels,
    validation_predictions,
    zero_division=0
)

validation_recall = recall_score(
    validation_labels,
    validation_predictions,
    zero_division=0
)

validation_f1 = f1_score(
    validation_labels,
    validation_predictions,
    zero_division=0
)

validation_roc_auc = roc_auc_score(
    validation_labels,
    validation_probabilities
)

# ------------------------------------------------------------
# 10. Store actual measured results
# ------------------------------------------------------------

baseline_validation_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Validation samples",
        "Validation participants",
        "Prediction time (seconds)",
        "Decision threshold"
    ],
    "Measured value": [
        validation_accuracy,
        validation_precision,
        validation_recall,
        validation_f1,
        validation_roc_auc,
        validation_samples,
        dl_validation[
            "ParticipantID"
        ].nunique(),
        validation_prediction_time,
        VALIDATION_THRESHOLD
    ]
})

# ------------------------------------------------------------
# 11. Display results
# ------------------------------------------------------------

print("\nValidation results")
print("-" * 80)

display(
    baseline_validation_results
)

print("\n" + "=" * 80)
print("VALIDATION BASELINE EVALUATION COMPLETED")
print("=" * 80)

print(
    f"\nValidation samples: "
    f"{validation_samples:,}"
)

print(
    f"Validation participants: "
    f"{dl_validation['ParticipantID'].nunique()}"
)

print(
    f"Prediction time: "
    f"{validation_prediction_time:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

FROZEN CNN — VALIDATION BASELINE EVALUATION

Validation results
--------------------------------------------------------------------------------


,Metric,Measured value
0,Accuracy,0.697027
1,Precision,0.905643
2,Recall,0.705661
3,F1,0.793242
4,ROC-AUC,0.746448
5,Validation samples,57718.000000
6,Validation participants,4.000000
7,Prediction time (seconds),49.515059
8,Decision threshold,0.500000



VALIDATION BASELINE EVALUATION COMPLETED

Validation samples: 57,718
Validation participants: 4
Prediction time: 49.5151 seconds

Test data was NOT accessed.


## 4. Pruning Experiment

Following establishment of the frozen CNN validation baseline, magnitude-based unstructured weight pruning is investigated as the first model-efficiency optimisation technique.

The objective is to evaluate whether progressively increasing parameter sparsity can reduce the model's storage and computational requirements while retaining acceptable predictive performance.

Four predefined target sparsity levels are investigated: 20%, 40%, 60%, and 80%. These levels are specified before evaluating the pruning outcomes to avoid selecting a pruning level based on observed validation performance.

For each pruning configuration, the model will be derived from the same frozen baseline architecture and evaluated using the validation split. The test split will remain completely isolated throughout pruning and candidate selection.

The principal comparison will consider predictive performance alongside parameter sparsity and model storage requirements. The final pruning level will therefore be selected using the validation results and predefined experimental criteria rather than by maximising sparsity alone.

No pruning experiment will modify the original frozen baseline model. Each candidate will be treated as an independent experimental model so that the baseline remains available for direct comparison.

In [ ]:
# Cell 13

# Construct and validate the first pruning candidate
# using the TensorFlow Model Optimization Keras compatibility layer

print("=" * 80)
print("CONSTRUCTING 20% PRUNING CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "baseline_cnn",
    "dl_train",
    "dl_validation",
    "PRUNING_BATCH_SIZE",
    "PRUNING_EPOCHS",
    "PRUNING_AVAILABLE",
    "tfmot"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
        + ". Run the preceding cells successfully."
    )

if not PRUNING_AVAILABLE:
    raise RuntimeError(
        "TensorFlow Model Optimization Toolkit "
        "is not available."
    )

# ------------------------------------------------------------
# 2. Verify the frozen Notebook 4 baseline
# ------------------------------------------------------------

baseline_parameter_count = (
    baseline_cnn.count_params()
)

if baseline_parameter_count != 3809:
    raise ValueError(
        "Unexpected frozen CNN parameter count. "
        f"Observed {baseline_parameter_count:,}; "
        "expected 3,809."
    )

if baseline_cnn.input_shape != (None, 2500, 1):
    raise ValueError(
        "Unexpected frozen CNN input shape. "
        f"Observed {baseline_cnn.input_shape}; "
        "expected (None, 2500, 1)."
    )

if dl_train.empty:
    raise ValueError(
        "Training metadata is empty."
    )

# ------------------------------------------------------------
# 3. Define pruning configuration
# ------------------------------------------------------------

PRUNING_TARGET_SPARSITY = 0.20

if not (
    0.0 < PRUNING_TARGET_SPARSITY < 1.0
):
    raise ValueError(
        "PRUNING_TARGET_SPARSITY must be strictly "
        "between 0 and 1."
    )

# ------------------------------------------------------------
# 4. Calculate the actual pruning schedule
# ------------------------------------------------------------

PRUNING_STEPS_PER_EPOCH = int(
    np.ceil(
        len(dl_train)
        / PRUNING_BATCH_SIZE
    )
)

if PRUNING_STEPS_PER_EPOCH <= 0:
    raise ValueError(
        "PRUNING_STEPS_PER_EPOCH must be positive."
    )

PRUNING_TOTAL_STEPS = (
    PRUNING_STEPS_PER_EPOCH
    * PRUNING_EPOCHS
)

if PRUNING_TOTAL_STEPS <= 0:
    raise ValueError(
        "PRUNING_TOTAL_STEPS must be positive."
    )

PRUNING_BEGIN_STEP = 0
PRUNING_END_STEP = PRUNING_TOTAL_STEPS

# ------------------------------------------------------------
# 5. Import TF-MOT's Keras compatibility API
# ------------------------------------------------------------

try:

    from tensorflow_model_optimization.python.core.keras.compat import (
        keras as pruning_keras
    )

except Exception as exc:

    raise RuntimeError(
        "Could not load the TensorFlow Model Optimization "
        "Keras compatibility API."
    ) from exc



pruning_base_cnn = pruning_keras.Sequential([
    pruning_keras.layers.Input(
        shape=(2500, 1)
    ),

    pruning_keras.layers.Conv1D(
        filters=16,
        kernel_size=7,
        activation="relu",
        padding="same"
    ),

    pruning_keras.layers.MaxPooling1D(
        pool_size=2
    ),

    pruning_keras.layers.Conv1D(
        filters=32,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),

    pruning_keras.layers.GlobalAveragePooling1D(),

    pruning_keras.layers.Dense(
        32,
        activation="relu"
    ),

    pruning_keras.layers.Dropout(
        0.3
    ),

    pruning_keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

# ------------------------------------------------------------
# 7. Verify the reconstructed architecture
# ------------------------------------------------------------

reconstructed_parameter_count = (
    pruning_base_cnn.count_params()
)

if reconstructed_parameter_count != 3809:
    raise ValueError(
        "The pruning-compatible CNN does not match "
        "the frozen architecture. "
        f"Observed {reconstructed_parameter_count:,} "
        "parameters; expected 3,809."
    )

if pruning_base_cnn.input_shape != (None, 2500, 1):
    raise ValueError(
        "The pruning-compatible CNN has an unexpected "
        f"input shape: {pruning_base_cnn.input_shape}"
    )

# ------------------------------------------------------------
# 8. Copy the frozen Notebook 4 weights
# ------------------------------------------------------------

baseline_weights = baseline_cnn.get_weights()

pruning_base_cnn.set_weights(
    baseline_weights
)

# ------------------------------------------------------------
# 9. Verify weight transfer
# ------------------------------------------------------------

transferred_weights = (
    pruning_base_cnn.get_weights()
)

if len(baseline_weights) != len(transferred_weights):
    raise RuntimeError(
        "The number of weight tensors changed "
        "during weight transfer."
    )

weights_match = all(
    np.array_equal(
        original,
        transferred
    )
    for original, transferred in zip(
        baseline_weights,
        transferred_weights
    )
)

if not weights_match:
    raise RuntimeError(
        "The pruning-compatible CNN does not contain "
        "the exact frozen Notebook 4 weights."
    )

# ------------------------------------------------------------
# 10. Preserve a separate copy of the baseline weights
# ------------------------------------------------------------

baseline_weights_before_pruning = [
    weight.copy()
    for weight in baseline_cnn.get_weights()
]

# ------------------------------------------------------------
# 11. Define the pruning schedule
# ------------------------------------------------------------

pruning_schedule = (
    tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=PRUNING_TARGET_SPARSITY,
        begin_step=PRUNING_BEGIN_STEP,
        end_step=PRUNING_END_STEP
    )
)

# ------------------------------------------------------------
# 12. Apply magnitude-based pruning
# ------------------------------------------------------------

pruned_cnn_20 = (
    tfmot.sparsity.keras.prune_low_magnitude(
        pruning_base_cnn,
        pruning_schedule=pruning_schedule
    )
)

# ------------------------------------------------------------
# 13. Verify pruning wrapper construction
# ------------------------------------------------------------

pruning_wrapper_count = sum(
    1
    for layer in pruned_cnn_20.layers
    if "PruneLowMagnitude"
    in layer.__class__.__name__
)

if pruning_wrapper_count <= 0:
    raise RuntimeError(
        "No pruning wrappers were detected."
    )

# ------------------------------------------------------------
# 14. Verify frozen baseline was not modified
# ------------------------------------------------------------

baseline_weights_after_pruning = (
    baseline_cnn.get_weights()
)

baseline_weights_unchanged = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_before_pruning,
        baseline_weights_after_pruning
    )
)

if not baseline_weights_unchanged:
    raise RuntimeError(
        "The frozen Notebook 4 baseline weights "
        "were modified."
    )

if baseline_cnn.count_params() != 3809:
    raise RuntimeError(
        "The frozen Notebook 4 parameter count changed."
    )

# ------------------------------------------------------------
# 15. Validate pruning schedule
# ------------------------------------------------------------

schedule_config_valid = (
    PRUNING_BEGIN_STEP == 0
    and
    PRUNING_END_STEP
    == (
        PRUNING_STEPS_PER_EPOCH
        * PRUNING_EPOCHS
    )
    and
    PRUNING_END_STEP > PRUNING_BEGIN_STEP
)

if not schedule_config_valid:
    raise RuntimeError(
        "Pruning schedule configuration is invalid."
    )

# ------------------------------------------------------------
# 16. Final validation status
# ------------------------------------------------------------

pruning_candidate_ready = (
    reconstructed_parameter_count == 3809
    and
    pruning_base_cnn.input_shape
    == (None, 2500, 1)
    and
    weights_match
    and
    pruning_wrapper_count > 0
    and
    baseline_weights_unchanged
    and
    schedule_config_valid
)

# ------------------------------------------------------------
# 17. Report verified configuration
# ------------------------------------------------------------

print("\nPruning configuration")
print("-" * 80)

print(
    f"Target sparsity:       "
    f"{PRUNING_TARGET_SPARSITY:.0%}"
)

print(
    f"Training samples:      "
    f"{len(dl_train):,}"
)

print(
    f"Batch size:            "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Steps per epoch:       "
    f"{PRUNING_STEPS_PER_EPOCH:,}"
)

print(
    f"Pruning epochs:        "
    f"{PRUNING_EPOCHS}"
)

print(
    f"Pruning begin step:    "
    f"{PRUNING_BEGIN_STEP:,}"
)

print(
    f"Pruning end step:      "
    f"{PRUNING_END_STEP:,}"
)

print(
    f"Baseline parameters:   "
    f"{baseline_parameter_count:,}"
)

print(
    f"Reconstructed params:  "
    f"{reconstructed_parameter_count:,}"
)

print(
    f"Pruning wrappers:      "
    f"{pruning_wrapper_count}"
)

# ------------------------------------------------------------
# 18. Verification summary
# ------------------------------------------------------------

print("\n" + "=" * 80)

print(
    "Architecture match:",
    "PASSED"
    if reconstructed_parameter_count == 3809
    else "FAILED"
)

print(
    "Input shape match:",
    "PASSED"
    if pruning_base_cnn.input_shape
    == (None, 2500, 1)
    else "FAILED"
)

print(
    "Frozen weight transfer:",
    "PASSED"
    if weights_match
    else "FAILED"
)

print(
    "Pruning wrapper construction:",
    "PASSED"
    if pruning_wrapper_count > 0
    else "FAILED"
)

print(
    "Frozen baseline unchanged:",
    "PASSED"
    if baseline_weights_unchanged
    else "FAILED"
)

print(
    "Pruning schedule:",
    "PASSED"
    if schedule_config_valid
    else "FAILED"
)

print("-" * 80)

print(
    "20% PRUNING CANDIDATE:",
    "READY"
    if pruning_candidate_ready
    else "FAILED"
)

print("=" * 80)

if not pruning_candidate_ready:
    raise RuntimeError(
        "Pruning candidate validation failed. "
        "No pruning training should proceed."
    )

CONSTRUCTING 20% PRUNING CANDIDATE

Pruning configuration
--------------------------------------------------------------------------------
Target sparsity:       20%
Training samples:      297,867
Batch size:            64
Steps per epoch:       4,655
Pruning epochs:        5
Pruning begin step:    0
Pruning end step:      23,275
Baseline parameters:   3,809
Reconstructed params:  3,809
Pruning wrappers:      7

Architecture match: PASSED
Input shape match: PASSED
Frozen weight transfer: PASSED
Pruning wrapper construction: PASSED
Frozen baseline unchanged: PASSED
Pruning schedule: PASSED
--------------------------------------------------------------------------------
20% PRUNING CANDIDATE: READY


In [ ]:
# Cell 14

# Train the 20% magnitude-pruned CNN candidate

print("=" * 80)
print("TRAINING 20% PRUNED CNN CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_20",
    "pruning_candidate_ready",
    "pruning_train_generator",
    "PRUNING_STEPS_PER_EPOCH",
    "PRUNING_EPOCHS",
    "CNN_CLASS_WEIGHTS",
    "baseline_cnn"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready:
    raise RuntimeError(
        "The 20% pruning candidate did not pass "
        "Cell 13 validation."
    )

# ------------------------------------------------------------
# 2. Verify the frozen baseline before training
# ------------------------------------------------------------

baseline_weights_before_training = [
    weight.copy()
    for weight in baseline_cnn.get_weights()
]

baseline_parameter_count_before_training = (
    baseline_cnn.count_params()
)

if baseline_parameter_count_before_training != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809."
    )

# ------------------------------------------------------------
# 3. Inspect the frozen baseline optimizer
# ------------------------------------------------------------

if not hasattr(
    baseline_cnn,
    "optimizer"
):
    raise RuntimeError(
        "The frozen baseline does not contain "
        "a compiled optimizer."
    )

if baseline_cnn.optimizer is None:
    raise RuntimeError(
        "The frozen baseline optimizer is None."
    )

baseline_optimizer_config = (
    baseline_cnn.optimizer.get_config()
)

baseline_optimizer_name = (
    baseline_cnn.optimizer.__class__.__name__
)

# ------------------------------------------------------------
# 4. Recreate the baseline optimizer in the
#    TF-MOT-compatible Keras environment
# ------------------------------------------------------------

optimizer_learning_rate = (
    baseline_optimizer_config.get(
        "learning_rate"
    )
)

if optimizer_learning_rate is None:
    raise RuntimeError(
        "Could not determine the frozen baseline "
        "optimizer learning rate."
    )

try:

    if baseline_optimizer_name == "Adam":

        pruning_optimizer = (
            pruning_keras.optimizers.Adam(
                learning_rate=optimizer_learning_rate
            )
        )

    elif baseline_optimizer_name == "RMSprop":

        pruning_optimizer = (
            pruning_keras.optimizers.RMSprop(
                learning_rate=optimizer_learning_rate
            )
        )

    elif baseline_optimizer_name == "SGD":

        pruning_optimizer = (
            pruning_keras.optimizers.SGD(
                learning_rate=optimizer_learning_rate
            )
        )

    else:

        raise RuntimeError(
            "Unsupported frozen baseline optimizer: "
            f"{baseline_optimizer_name}"
        )

except Exception as exc:

    raise RuntimeError(
        "Could not recreate the frozen baseline "
        "optimizer in the TF-MOT-compatible Keras "
        "environment."
    ) from exc

# ------------------------------------------------------------
# 5. Verify the optimizer learning rate
# ------------------------------------------------------------

if not np.isclose(
    float(
        pruning_optimizer.learning_rate.numpy()
    ),
    float(
        optimizer_learning_rate
    )
):
    raise RuntimeError(
        "The pruning optimizer learning rate does not "
        "match the frozen baseline optimizer."
    )

# ------------------------------------------------------------
# 6. Compile the pruning candidate
# ------------------------------------------------------------

pruned_cnn_20.compile(
    optimizer=pruning_optimizer,
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)

# ------------------------------------------------------------
# 7. Create the pruning callback
# ------------------------------------------------------------

pruning_callback = (
    tfmot.sparsity.keras.UpdatePruningStep()
)

# ------------------------------------------------------------
# 8. Calculate training configuration
# ------------------------------------------------------------

training_steps = (
    PRUNING_STEPS_PER_EPOCH
)

training_epochs = (
    PRUNING_EPOCHS
)

if training_steps <= 0:
    raise RuntimeError(
        "Training steps must be positive."
    )

if training_epochs <= 0:
    raise RuntimeError(
        "Training epochs must be positive."
    )

# ------------------------------------------------------------
# 9. Train the 20% pruning candidate
# ------------------------------------------------------------

training_start_time = (
    time.perf_counter()
)

pruning_history_20 = (
    pruned_cnn_20.fit(
        pruning_train_generator,
        steps_per_epoch=training_steps,
        epochs=training_epochs,
        callbacks=[
            pruning_callback
        ],
        verbose=1
    )
)

pruning_training_time_20 = (
    time.perf_counter()
    - training_start_time
)

# ------------------------------------------------------------
# 10. Verify training history
# ------------------------------------------------------------

if not hasattr(
    pruning_history_20,
    "history"
):
    raise RuntimeError(
        "Training did not return a valid history object."
    )

history_keys = set(
    pruning_history_20.history.keys()
)

if "loss" not in history_keys:
    raise RuntimeError(
        "Training history does not contain loss values."
    )

if "accuracy" not in history_keys:
    raise RuntimeError(
        "Training history does not contain accuracy values."
    )

if len(
    pruning_history_20.history["loss"]
) != training_epochs:
    raise RuntimeError(
        "Training history contains an unexpected "
        "number of epochs."
    )

# ------------------------------------------------------------
# 11. Verify the frozen baseline remained unchanged
# ------------------------------------------------------------

baseline_weights_after_training = (
    baseline_cnn.get_weights()
)

baseline_weights_unchanged = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_before_training,
        baseline_weights_after_training
    )
)

if not baseline_weights_unchanged:
    raise RuntimeError(
        "The frozen baseline CNN was modified during "
        "pruning training."
    )

if (
    baseline_cnn.count_params()
    != baseline_parameter_count_before_training
):
    raise RuntimeError(
        "The frozen baseline parameter count changed "
        "during pruning training."
    )

# ------------------------------------------------------------
# 12. Report training results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("20% PRUNING TRAINING COMPLETED")
print("=" * 80)

print(
    f"\nOptimizer: "
    f"{baseline_optimizer_name}"
)

print(
    f"Learning rate: "
    f"{float(optimizer_learning_rate):.10g}"
)

print(
    f"Training samples per epoch: "
    f"{len(dl_train):,}"
)

print(
    f"Steps per epoch: "
    f"{training_steps:,}"
)

print(
    f"Epochs completed: "
    f"{len(pruning_history_20.history['loss'])}"
)

print(
    f"Training time: "
    f"{pruning_training_time_20:.4f} seconds"
)

print("\nFinal training metrics")
print("-" * 80)

print(
    f"Loss: "
    f"{pruning_history_20.history['loss'][-1]:.6f}"
)

print(
    f"Accuracy: "
    f"{pruning_history_20.history['accuracy'][-1]:.6f}"
)

print("\nFrozen baseline verification")
print("-" * 80)

print(
    "Baseline weights unchanged:",
    "PASSED"
    if baseline_weights_unchanged
    else "FAILED"
)

print(
    "Baseline parameter count unchanged:",
    "PASSED"
    if (
        baseline_cnn.count_params()
        == baseline_parameter_count_before_training
    )
    else "FAILED"
)

print(
    "\nTest data was NOT used."
)

print("=" * 80)

if not baseline_weights_unchanged:
    raise RuntimeError(
        "Frozen baseline integrity check failed."
    )

TRAINING 20% PRUNED CNN CANDIDATE
Epoch 1/5
4655/4655 [==============================] - 726s 155ms/step - loss: 0.4155 - accuracy: 0.7947
Epoch 2/5
4655/4655 [==============================] - 651s 140ms/step - loss: 0.3887 - accuracy: 0.8076
Epoch 3/5
4655/4655 [==============================] - 636s 137ms/step - loss: 0.3743 - accuracy: 0.8152
Epoch 4/5
4655/4655 [==============================] - 626s 135ms/step - loss: 0.3659 - accuracy: 0.8209
Epoch 5/5
4526/4655 [============================>.] - ETA: 17s - loss: 0.3597 - accuracy: 0.8246

In [ ]:
# Cell 15

# Evaluate the trained 20% pruned CNN on the validation split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 80)
print("20% PRUNED CNN — VALIDATION EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_20",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE",
    "pruning_candidate_ready"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready:
    raise RuntimeError(
        "The 20% pruning candidate did not pass "
        "Cell 13 validation."
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

# ------------------------------------------------------------
# 2. Create a fresh validation generator
# ------------------------------------------------------------

pruned_validation_generator_20 = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 3. Determine exact validation steps
# ------------------------------------------------------------

validation_samples_20 = len(
    dl_validation
)

validation_steps_20 = int(
    np.ceil(
        validation_samples_20
        / PRUNING_BATCH_SIZE
    )
)

if validation_steps_20 <= 0:
    raise RuntimeError(
        "Validation step count must be positive."
    )

# ------------------------------------------------------------
# 4. Generate predictions
# ------------------------------------------------------------

evaluation_start_time_20 = (
    time.perf_counter()
)

validation_probabilities_20 = (
    pruned_cnn_20.predict(
        pruned_validation_generator_20,
        steps=validation_steps_20,
        verbose=0
    )
)

evaluation_time_20 = (
    time.perf_counter()
    - evaluation_start_time_20
)

# ------------------------------------------------------------
# 5. Convert predictions to one-dimensional array
# ------------------------------------------------------------

validation_probabilities_20 = np.asarray(
    validation_probabilities_20,
    dtype=np.float64
).reshape(-1)

validation_probabilities_20 = (
    validation_probabilities_20[
        :validation_samples_20
    ]
)

if len(
    validation_probabilities_20
) != validation_samples_20:
    raise RuntimeError(
        "Prediction count does not match the "
        "validation sample count."
    )

# ------------------------------------------------------------
# 6. Validate prediction values
# ------------------------------------------------------------

if not np.isfinite(
    validation_probabilities_20
).all():
    raise ValueError(
        "Non-finite prediction probabilities detected."
    )

if (
    (validation_probabilities_20 < 0.0)
    |
    (validation_probabilities_20 > 1.0)
).any():
    raise ValueError(
        "Prediction probabilities outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 7. Obtain validation labels
# ------------------------------------------------------------

validation_labels_20 = (
    dl_validation[
        "WakeSleepLabel"
    ]
    .to_numpy(
        dtype=np.int32
    )
)

if len(
    validation_labels_20
) != validation_samples_20:
    raise RuntimeError(
        "Validation label count does not match "
        "validation sample count."
    )

if not set(
    np.unique(validation_labels_20)
).issubset({0, 1}):
    raise ValueError(
        "Validation labels are not binary."
    )

if len(
    np.unique(validation_labels_20)
) < 2:
    raise ValueError(
        "ROC-AUC cannot be calculated because "
        "the validation set contains only one class."
    )

# ------------------------------------------------------------
# 8. Convert probabilities to class predictions
# ------------------------------------------------------------

VALIDATION_THRESHOLD_20 = 0.50

validation_predictions_20 = (
    validation_probabilities_20
    >= VALIDATION_THRESHOLD_20
).astype(np.int32)

# ------------------------------------------------------------
# 9. Calculate validation metrics
# ------------------------------------------------------------

pruned_validation_accuracy_20 = (
    accuracy_score(
        validation_labels_20,
        validation_predictions_20
    )
)

pruned_validation_precision_20 = (
    precision_score(
        validation_labels_20,
        validation_predictions_20,
        zero_division=0
    )
)

pruned_validation_recall_20 = (
    recall_score(
        validation_labels_20,
        validation_predictions_20,
        zero_division=0
    )
)

pruned_validation_f1_20 = (
    f1_score(
        validation_labels_20,
        validation_predictions_20,
        zero_division=0
    )
)

pruned_validation_roc_auc_20 = (
    roc_auc_score(
        validation_labels_20,
        validation_probabilities_20
    )
)

# ------------------------------------------------------------
# 10. Store measured results
# ------------------------------------------------------------

pruned_validation_results_20 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Validation samples",
        "Validation participants",
        "Evaluation time (seconds)",
        "Decision threshold",
        "Target sparsity"
    ],
    "Measured value": [
        pruned_validation_accuracy_20,
        pruned_validation_precision_20,
        pruned_validation_recall_20,
        pruned_validation_f1_20,
        pruned_validation_roc_auc_20,
        validation_samples_20,
        dl_validation[
            "ParticipantID"
        ].nunique(),
        evaluation_time_20,
        VALIDATION_THRESHOLD_20,
        PRUNING_TARGET_SPARSITY
    ]
})

# ------------------------------------------------------------
# 11. Display results
# ------------------------------------------------------------

print("\nValidation results")
print("-" * 80)

display(
    pruned_validation_results_20
)

print("\n" + "=" * 80)
print("20% PRUNED CNN — VALIDATION EVALUATION COMPLETED")
print("=" * 80)

print(
    f"\nValidation samples: "
    f"{validation_samples_20:,}"
)

print(
    f"Validation participants: "
    f"{dl_validation['ParticipantID'].nunique()}"
)

print(
    f"Evaluation time: "
    f"{evaluation_time_20:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

20% PRUNED CNN — VALIDATION EVALUATION

Validation results
--------------------------------------------------------------------------------


,Metric,Measured value
0,Accuracy,0.714387
1,Precision,0.868789
2,Recall,0.769422
3,F1,0.816092
4,ROC-AUC,0.745819
5,Validation samples,57718.000000
6,Validation participants,4.000000
7,Evaluation time (seconds),82.899292
8,Decision threshold,0.500000
9,Target sparsity,0.200000



20% PRUNED CNN — VALIDATION EVALUATION COMPLETED

Validation samples: 57,718
Validation participants: 4
Evaluation time: 82.8993 seconds

Test data was NOT accessed.


In [ ]:
# Cell 16


print("=" * 80)
print("20% PRUNED CNN — SPARSITY AND BASELINE COMPARISON")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_20",
    "baseline_cnn",
    "baseline_validation_results",
    "pruned_validation_results_20",
    "PRUNING_TARGET_SPARSITY"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

# ------------------------------------------------------------
# 2. Create a stripped copy of the trained pruning model
#
# Stripping removes the pruning wrappers while retaining
# the trained masked weights.
# ------------------------------------------------------------

pruned_cnn_20_stripped = (
    tfmot.sparsity.keras.strip_pruning(
        pruned_cnn_20
    )
)

# ------------------------------------------------------------
# 3. Collect trainable weight tensors
# ------------------------------------------------------------

pruned_weight_arrays = []

for layer in pruned_cnn_20_stripped.layers:

    for weight in layer.weights:

        weight_array = (
            weight.numpy()
        )

        if weight_array.size == 0:
            continue

        pruned_weight_arrays.append(
            weight_array
        )

# ------------------------------------------------------------
# 4. Calculate actual non-zero parameter count
# ------------------------------------------------------------

total_weight_values_20 = sum(
    array.size
    for array in pruned_weight_arrays
)

nonzero_weight_values_20 = sum(
    np.count_nonzero(array)
    for array in pruned_weight_arrays
)

zero_weight_values_20 = (
    total_weight_values_20
    - nonzero_weight_values_20
)

if total_weight_values_20 <= 0:
    raise RuntimeError(
        "No trainable weight values were found."
    )

actual_sparsity_20 = (
    zero_weight_values_20
    / total_weight_values_20
)

# ------------------------------------------------------------
# 5. Verify parameter count against the frozen architecture
# ------------------------------------------------------------

stripped_parameter_count_20 = (
    pruned_cnn_20_stripped.count_params()
)

if stripped_parameter_count_20 != (
    baseline_cnn.count_params()
):
    raise RuntimeError(
        "The stripped pruning candidate has a different "
        "dense parameter count from the frozen baseline."
    )

# ------------------------------------------------------------
# 6. Calculate dense float32 storage
# ------------------------------------------------------------

dense_storage_bytes_20 = (
    total_weight_values_20
    * np.dtype(
        np.float32
    ).itemsize
)

dense_storage_kb_20 = (
    dense_storage_bytes_20
    / 1024
)

dense_storage_mb_20 = (
    dense_storage_bytes_20
    / (1024 ** 2)
)

# ------------------------------------------------------------
# 7. Calculate validation metric changes
# ------------------------------------------------------------

baseline_accuracy = (
    float(
        baseline_validation_results.loc[
            baseline_validation_results["Metric"]
            == "Accuracy",
            "Measured value"
        ].iloc[0]
    )
)

baseline_precision = (
    float(
        baseline_validation_results.loc[
            baseline_validation_results["Metric"]
            == "Precision",
            "Measured value"
        ].iloc[0]
    )
)

baseline_recall = (
    float(
        baseline_validation_results.loc[
            baseline_validation_results["Metric"]
            == "Recall",
            "Measured value"
        ].iloc[0]
    )
)

baseline_f1 = (
    float(
        baseline_validation_results.loc[
            baseline_validation_results["Metric"]
            == "F1",
            "Measured value"
        ].iloc[0]
    )
)

baseline_roc_auc = (
    float(
        baseline_validation_results.loc[
            baseline_validation_results["Metric"]
            == "ROC-AUC",
            "Measured value"
        ].iloc[0]
    )
)

accuracy_change_20 = (
    pruned_validation_accuracy_20
    - baseline_accuracy
)

precision_change_20 = (
    pruned_validation_precision_20
    - baseline_precision
)

recall_change_20 = (
    pruned_validation_recall_20
    - baseline_recall
)

f1_change_20 = (
    pruned_validation_f1_20
    - baseline_f1
)

roc_auc_change_20 = (
    pruned_validation_roc_auc_20
    - baseline_roc_auc
)

# ------------------------------------------------------------
# 8. Create comparison table
# ------------------------------------------------------------

pruning_comparison_20 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],
    "Frozen baseline": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_roc_auc
    ],
    "20% pruning": [
        pruned_validation_accuracy_20,
        pruned_validation_precision_20,
        pruned_validation_recall_20,
        pruned_validation_f1_20,
        pruned_validation_roc_auc_20
    ],
    "Change": [
        accuracy_change_20,
        precision_change_20,
        recall_change_20,
        f1_change_20,
        roc_auc_change_20
    ]
})

# ------------------------------------------------------------
# 9. Create efficiency summary
# ------------------------------------------------------------

pruning_efficiency_20 = pd.DataFrame({
    "Metric": [
        "Dense parameter count",
        "Zero weight values",
        "Non-zero weight values",
        "Actual sparsity",
        "Target sparsity",
        "Dense float32 storage (KB)",
        "Dense float32 storage (MB)"
    ],
    "20% pruning": [
        stripped_parameter_count_20,
        zero_weight_values_20,
        nonzero_weight_values_20,
        actual_sparsity_20,
        PRUNING_TARGET_SPARSITY,
        dense_storage_kb_20,
        dense_storage_mb_20
    ]
})

# ------------------------------------------------------------
# 10. Display measured results
# ------------------------------------------------------------

print("\nValidation comparison")
print("-" * 80)

display(
    pruning_comparison_20
)

print("\nEfficiency measurements")
print("-" * 80)

display(
    pruning_efficiency_20
)

print("\n" + "=" * 80)
print("20% PRUNING ANALYSIS COMPLETED")
print("=" * 80)

print(
    f"\nActual sparsity: "
    f"{actual_sparsity_20:.4%}"
)

print(
    f"Target sparsity: "
    f"{PRUNING_TARGET_SPARSITY:.2%}"
)

print(
    f"Zero weights: "
    f"{zero_weight_values_20:,}"
)

print(
    f"Non-zero weights: "
    f"{nonzero_weight_values_20:,}"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

20% PRUNED CNN — SPARSITY AND BASELINE COMPARISON

Validation comparison
--------------------------------------------------------------------------------


,Metric,Frozen baseline,20% pruning,Change
0,Accuracy,0.697027,0.714387,0.017360
1,Precision,0.905643,0.868789,-0.036854
2,Recall,0.705661,0.769422,0.063761
3,F1,0.793242,0.816092,0.022850
4,ROC-AUC,0.746448,0.745819,-0.000629



Efficiency measurements
--------------------------------------------------------------------------------


,Metric,20% pruning
0,Dense parameter count,3809.000000
1,Zero weight values,746.000000
2,Non-zero weight values,3063.000000
3,Actual sparsity,0.195852
4,Target sparsity,0.200000
5,Dense float32 storage (KB),14.878906
6,Dense float32 storage (MB),0.014530



20% PRUNING ANALYSIS COMPLETED

Actual sparsity: 19.5852%
Target sparsity: 20.00%
Zero weights: 746
Non-zero weights: 3,063

Test data was NOT accessed.


In [ ]:
# Cell 17

# Construct and validate the 40% pruning candidate
# directly from the frozen Notebook 4 CNN

print("=" * 80)
print("CONSTRUCTING 40% PRUNING CANDIDATE")
print("=" * 80)



required_names = [
    "baseline_cnn",
    "dl_train",
    "PRUNING_BATCH_SIZE",
    "PRUNING_EPOCHS",
    "PRUNING_STEPS_PER_EPOCH",
    "tfmot",
    "pruning_keras",
    "PRUNING_AVAILABLE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not PRUNING_AVAILABLE:
    raise RuntimeError(
        "TensorFlow Model Optimization Toolkit "
        "is not available."
    )

# ------------------------------------------------------------
# 2. Verify the frozen baseline
# ------------------------------------------------------------

baseline_parameter_count_40 = (
    baseline_cnn.count_params()
)

if baseline_parameter_count_40 != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809. "
        f"Observed: {baseline_parameter_count_40:,}"
    )

if baseline_cnn.input_shape != (
    None,
    2500,
    1
):
    raise RuntimeError(
        "Unexpected frozen baseline input shape: "
        f"{baseline_cnn.input_shape}"
    )



baseline_weights_40 = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

baseline_weight_shapes_40 = [
    weight.shape
    for weight in baseline_weights_40
]

# ------------------------------------------------------------
# 4. Define target sparsity
# ------------------------------------------------------------

PRUNING_TARGET_SPARSITY_40 = 0.40

if not (
    0.0 < PRUNING_TARGET_SPARSITY_40 < 1.0
):
    raise ValueError(
        "Target sparsity must be strictly between "
        "0 and 1."
    )

# ------------------------------------------------------------
# 5. Reconstruct a FRESH TF-MOT-compatible CNN
#
# This is deliberately independent of:
#     pruning_base_cnn
#     pruned_cnn_20
#
# Therefore the previously trained 20% model cannot
# contaminate the 40% candidate.
# ------------------------------------------------------------

pruning_base_cnn_40 = (
    pruning_keras.Sequential([
        pruning_keras.layers.Input(
            shape=(2500, 1)
        ),

        pruning_keras.layers.Conv1D(
            filters=16,
            kernel_size=7,
            activation="relu",
            padding="same"
        ),

        pruning_keras.layers.MaxPooling1D(
            pool_size=2
        ),

        pruning_keras.layers.Conv1D(
            filters=32,
            kernel_size=5,
            activation="relu",
            padding="same"
        ),

        pruning_keras.layers.GlobalAveragePooling1D(),

        pruning_keras.layers.Dense(
            32,
            activation="relu"
        ),

        pruning_keras.layers.Dropout(
            0.3
        ),

        pruning_keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])
)

# ------------------------------------------------------------
# 6. Verify architecture
# ------------------------------------------------------------

if (
    pruning_base_cnn_40.count_params()
    != baseline_parameter_count_40
):
    raise RuntimeError(
        "The fresh 40% pruning base model does not "
        "match the frozen baseline parameter count."
    )

if (
    pruning_base_cnn_40.input_shape
    != baseline_cnn.input_shape
):
    raise RuntimeError(
        "The fresh 40% pruning base model does not "
        "match the frozen baseline input shape."
    )

# ------------------------------------------------------------
# 7. Transfer ONLY the frozen baseline weights
# ------------------------------------------------------------

pruning_base_cnn_40.set_weights(
    baseline_weights_40
)

# ------------------------------------------------------------
# 8. Verify exact tensor-by-tensor equality
# ------------------------------------------------------------

candidate_weights_40 = [
    np.array(
        weight,
        copy=True
    )
    for weight in pruning_base_cnn_40.get_weights()
]

if len(
    candidate_weights_40
) != len(
    baseline_weights_40
):
    raise RuntimeError(
        "Baseline and candidate contain different "
        "numbers of weight tensors."
    )

weight_mismatches_40 = []

for index, (
    baseline_weight,
    candidate_weight
) in enumerate(
    zip(
        baseline_weights_40,
        candidate_weights_40
    )
):

    if not np.array_equal(
        baseline_weight,
        candidate_weight
    ):

        weight_mismatches_40.append({
            "tensor_index": index,
            "shape": baseline_weight.shape,
            "max_absolute_difference": float(
                np.max(
                    np.abs(
                        baseline_weight
                        - candidate_weight
                    )
                )
            )
        })

if weight_mismatches_40:

    print(
        "\nWeight mismatches detected:"
    )

    for mismatch in weight_mismatches_40:
        print(mismatch)

    raise RuntimeError(
        "The fresh 40% candidate does not contain "
        "an exact copy of the frozen baseline weights."
    )

weights_match_40 = True

# ------------------------------------------------------------
# 9. Calculate the actual pruning schedule
# ------------------------------------------------------------

PRUNING_BEGIN_STEP_40 = 0

PRUNING_END_STEP_40 = (
    PRUNING_STEPS_PER_EPOCH
    * PRUNING_EPOCHS
)

if PRUNING_END_STEP_40 <= 0:
    raise RuntimeError(
        "Invalid pruning end step."
    )

# ------------------------------------------------------------
# 10. Define the 40% pruning schedule
# ------------------------------------------------------------

pruning_schedule_40 = (
    tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=PRUNING_TARGET_SPARSITY_40,
        begin_step=PRUNING_BEGIN_STEP_40,
        end_step=PRUNING_END_STEP_40
    )
)

# ------------------------------------------------------------
# 11. Apply magnitude-based pruning
# ------------------------------------------------------------

pruned_cnn_40 = (
    tfmot.sparsity.keras.prune_low_magnitude(
        pruning_base_cnn_40,
        pruning_schedule=pruning_schedule_40
    )
)

# ------------------------------------------------------------
# 12. Verify pruning wrappers
# ------------------------------------------------------------

pruning_wrapper_count_40 = sum(
    1
    for layer in pruned_cnn_40.layers
    if "PruneLowMagnitude"
    in layer.__class__.__name__
)

if pruning_wrapper_count_40 <= 0:
    raise RuntimeError(
        "No pruning wrappers were detected "
        "in the 40% candidate."
    )

# ------------------------------------------------------------
# 13. Verify frozen baseline was NOT modified
# ------------------------------------------------------------

baseline_weights_after_40 = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

baseline_unchanged_40 = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_40,
        baseline_weights_after_40
    )
)

if not baseline_unchanged_40:
    raise RuntimeError(
        "The frozen Notebook 4 baseline was modified."
    )

# ------------------------------------------------------------
# 14. Final candidate status
# ------------------------------------------------------------

pruning_candidate_ready_40 = (
    weights_match_40
    and
    baseline_unchanged_40
    and
    pruning_base_cnn_40.count_params()
    == 3809
    and
    pruning_base_cnn_40.input_shape
    == (None, 2500, 1)
    and
    pruning_wrapper_count_40 > 0
)

# ------------------------------------------------------------
# 15. Report verification
# ------------------------------------------------------------

print("\nPruning configuration")
print("-" * 80)

print(
    f"Target sparsity:       "
    f"{PRUNING_TARGET_SPARSITY_40:.0%}"
)

print(
    f"Training samples:      "
    f"{len(dl_train):,}"
)

print(
    f"Batch size:            "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Steps per epoch:       "
    f"{PRUNING_STEPS_PER_EPOCH:,}"
)

print(
    f"Pruning epochs:        "
    f"{PRUNING_EPOCHS}"
)

print(
    f"Pruning end step:      "
    f"{PRUNING_END_STEP_40:,}"
)

print(
    f"Baseline parameters:   "
    f"{baseline_parameter_count_40:,}"
)

print(
    f"Candidate parameters:  "
    f"{pruning_base_cnn_40.count_params():,}"
)

print(
    f"Pruning wrappers:      "
    f"{pruning_wrapper_count_40}"
)

print("\n" + "=" * 80)

print(
    "Frozen weight transfer:",
    "PASSED"
    if weights_match_40
    else "FAILED"
)

print(
    "Architecture match:",
    "PASSED"
    if pruning_base_cnn_40.count_params()
    == 3809
    else "FAILED"
)

print(
    "Input shape match:",
    "PASSED"
    if pruning_base_cnn_40.input_shape
    == (None, 2500, 1)
    else "FAILED"
)

print(
    "Pruning wrapper construction:",
    "PASSED"
    if pruning_wrapper_count_40 > 0
    else "FAILED"
)

print(
    "Frozen baseline unchanged:",
    "PASSED"
    if baseline_unchanged_40
    else "FAILED"
)

print("-" * 80)

print(
    "40% PRUNING CANDIDATE:",
    "READY"
    if pruning_candidate_ready_40
    else "FAILED"
)

print("=" * 80)

if not pruning_candidate_ready_40:
    raise RuntimeError(
        "40% pruning candidate validation failed."
    )

CONSTRUCTING 40% PRUNING CANDIDATE

Pruning configuration
--------------------------------------------------------------------------------
Target sparsity:       40%
Training samples:      297,867
Batch size:            64
Steps per epoch:       4,655
Pruning epochs:        5
Pruning end step:      23,275
Baseline parameters:   3,809
Candidate parameters:  3,809
Pruning wrappers:      7

Frozen weight transfer: PASSED
Architecture match: PASSED
Input shape match: PASSED
Pruning wrapper construction: PASSED
Frozen baseline unchanged: PASSED
--------------------------------------------------------------------------------
40% PRUNING CANDIDATE: READY


In [ ]:
# Cell 18

# Train the 40% magnitude-pruned CNN candidate

print("=" * 80)
print("TRAINING 40% PRUNED CNN CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_40",
    "pruning_candidate_ready_40",
    "baseline_cnn",
    "dl_train",
    "PRUNING_BATCH_SIZE",
    "PRUNING_EPOCHS",
    "PRUNING_STEPS_PER_EPOCH",
    "tfmot",
    "pruning_keras",
    "PRUNING_AVAILABLE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_40:
    raise RuntimeError(
        "The 40% pruning candidate did not pass "
        "Cell 17 validation."
    )

if dl_train.empty:
    raise ValueError(
        "Training metadata is empty."
    )

# ------------------------------------------------------------
# 2. Preserve the frozen baseline before training
# ------------------------------------------------------------

baseline_weights_before_40_training = [
    weight.copy()
    for weight in baseline_cnn.get_weights()
]

baseline_parameter_count_before_40_training = (
    baseline_cnn.count_params()
)

# ------------------------------------------------------------
# 3. Verify the frozen baseline parameter count
# ------------------------------------------------------------

if (
    baseline_parameter_count_before_40_training
    != 3809
):
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809."
    )



if not hasattr(
    baseline_cnn,
    "optimizer"
):
    raise RuntimeError(
        "Frozen baseline optimizer is unavailable."
    )

if baseline_cnn.optimizer is None:
    raise RuntimeError(
        "Frozen baseline optimizer is None."
    )

baseline_optimizer_name_40 = (
    baseline_cnn.optimizer.__class__.__name__
)

baseline_optimizer_config_40 = (
    baseline_cnn.optimizer.get_config()
)

optimizer_learning_rate_40 = (
    baseline_optimizer_config_40.get(
        "learning_rate"
    )
)

if optimizer_learning_rate_40 is None:
    raise RuntimeError(
        "Could not determine the frozen baseline "
        "learning rate."
    )

# ------------------------------------------------------------
# 5. Create a FRESH optimizer
# ------------------------------------------------------------

if baseline_optimizer_name_40 == "Adam":

    pruning_optimizer_40 = (
        pruning_keras.optimizers.Adam(
            learning_rate=optimizer_learning_rate_40
        )
    )

elif baseline_optimizer_name_40 == "RMSprop":

    pruning_optimizer_40 = (
        pruning_keras.optimizers.RMSprop(
            learning_rate=optimizer_learning_rate_40
        )
    )

elif baseline_optimizer_name_40 == "SGD":

    pruning_optimizer_40 = (
        pruning_keras.optimizers.SGD(
            learning_rate=optimizer_learning_rate_40
        )
    )

else:

    raise RuntimeError(
        "Unsupported frozen baseline optimizer: "
        f"{baseline_optimizer_name_40}"
    )

# ------------------------------------------------------------
# 6. Verify the fresh optimizer learning rate
# ------------------------------------------------------------

fresh_learning_rate_40 = float(
    pruning_optimizer_40.learning_rate.numpy()
)

if not np.isclose(
    fresh_learning_rate_40,
    float(optimizer_learning_rate_40)
):
    raise RuntimeError(
        "Fresh optimizer learning rate does not "
        "match the frozen baseline learning rate."
    )

# ------------------------------------------------------------
# 7. Compile the 40% pruning candidate
# ------------------------------------------------------------

pruned_cnn_40.compile(
    optimizer=pruning_optimizer_40,
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)

# ------------------------------------------------------------
# 8. Create a FRESH real-data training generator
# ------------------------------------------------------------

pruning_train_generator_40 = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

# ------------------------------------------------------------
# 9. Verify one real training batch
# ------------------------------------------------------------

X_train_check_40, y_train_check_40, weights_check_40 = (
    next(
        pruning_train_generator_40
    )
)

if X_train_check_40.ndim != 3:
    raise RuntimeError(
        "Training ECG batch must be 3-dimensional."
    )

if X_train_check_40.shape[1:] != (
    2500,
    1
):
    raise RuntimeError(
        "Unexpected ECG batch shape: "
        f"{X_train_check_40.shape}"
    )

if y_train_check_40.ndim != 1:
    raise RuntimeError(
        "Training labels must be one-dimensional."
    )

if len(X_train_check_40) != len(
    y_train_check_40
):
    raise RuntimeError(
        "Training samples and labels have "
        "different lengths."
    )

if len(X_train_check_40) != len(
    weights_check_40
):
    raise RuntimeError(
        "Training samples and sample weights have "
        "different lengths."
    )

if not np.isfinite(
    X_train_check_40
).all():
    raise ValueError(
        "Non-finite values detected in training ECG batch."
    )

if not np.isfinite(
    y_train_check_40
).all():
    raise ValueError(
        "Non-finite values detected in training labels."
    )

if not np.isfinite(
    weights_check_40
).all():
    raise ValueError(
        "Non-finite values detected in training weights."
    )

# ------------------------------------------------------------
# 10. Recreate the generator because the verification call
#     consumed one batch
# ------------------------------------------------------------

pruning_train_generator_40 = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

# ------------------------------------------------------------
# 11. Create a fresh pruning callback
# ------------------------------------------------------------

pruning_callback_40 = (
    tfmot.sparsity.keras.UpdatePruningStep()
)

# ------------------------------------------------------------
# 12. Verify training configuration
# ------------------------------------------------------------

training_steps_40 = (
    PRUNING_STEPS_PER_EPOCH
)

training_epochs_40 = (
    PRUNING_EPOCHS
)

if training_steps_40 <= 0:
    raise RuntimeError(
        "Training steps must be positive."
    )

if training_epochs_40 <= 0:
    raise RuntimeError(
        "Training epochs must be positive."
    )

# ------------------------------------------------------------
# 13. Train the 40% candidate
# ------------------------------------------------------------

training_start_time_40 = (
    time.perf_counter()
)

pruning_history_40 = (
    pruned_cnn_40.fit(
        pruning_train_generator_40,
        steps_per_epoch=training_steps_40,
        epochs=training_epochs_40,
        callbacks=[
            pruning_callback_40
        ],
        verbose=1
    )
)

pruning_training_time_40 = (
    time.perf_counter()
    - training_start_time_40
)

# ------------------------------------------------------------
# 14. Verify training history
# ------------------------------------------------------------

if not hasattr(
    pruning_history_40,
    "history"
):
    raise RuntimeError(
        "Training did not return a valid history object."
    )

history_keys_40 = set(
    pruning_history_40.history.keys()
)

if "loss" not in history_keys_40:
    raise RuntimeError(
        "Training history does not contain loss."
    )

if "accuracy" not in history_keys_40:
    raise RuntimeError(
        "Training history does not contain accuracy."
    )

completed_epochs_40 = len(
    pruning_history_40.history["loss"]
)

if completed_epochs_40 != training_epochs_40:
    raise RuntimeError(
        "Unexpected number of completed epochs: "
        f"{completed_epochs_40}; "
        f"expected {training_epochs_40}."
    )

# ------------------------------------------------------------
# 15. Verify frozen baseline integrity
# ------------------------------------------------------------

baseline_weights_after_40_training = (
    baseline_cnn.get_weights()
)

baseline_unchanged_40_training = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_before_40_training,
        baseline_weights_after_40_training
    )
)

if not baseline_unchanged_40_training:
    raise RuntimeError(
        "Frozen baseline weights were modified "
        "during 40% pruning training."
    )

if (
    baseline_cnn.count_params()
    != baseline_parameter_count_before_40_training
):
    raise RuntimeError(
        "Frozen baseline parameter count changed "
        "during 40% pruning training."
    )

# ------------------------------------------------------------
# 16. Report training results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("40% PRUNING TRAINING COMPLETED")
print("=" * 80)

print(
    f"\nOptimizer: "
    f"{baseline_optimizer_name_40}"
)

print(
    f"Learning rate: "
    f"{fresh_learning_rate_40:.10g}"
)

print(
    f"Training samples: "
    f"{len(dl_train):,}"
)

print(
    f"Batch size: "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Steps per epoch: "
    f"{training_steps_40:,}"
)

print(
    f"Epochs completed: "
    f"{completed_epochs_40}"
)

print(
    f"Training time: "
    f"{pruning_training_time_40:.4f} seconds"
)

print("\nFinal training metrics")
print("-" * 80)

print(
    f"Loss: "
    f"{pruning_history_40.history['loss'][-1]:.6f}"
)

print(
    f"Accuracy: "
    f"{pruning_history_40.history['accuracy'][-1]:.6f}"
)

print("\nFrozen baseline verification")
print("-" * 80)

print(
    "Baseline weights unchanged:",
    "PASSED"
    if baseline_unchanged_40_training
    else "FAILED"
)

print(
    "Baseline parameter count unchanged:",
    "PASSED"
    if (
        baseline_cnn.count_params()
        == baseline_parameter_count_before_40_training
    )
    else "FAILED"
)

print(
    "\nTest data was NOT used."
)

print("=" * 80)

if not baseline_unchanged_40_training:
    raise RuntimeError(
        "Frozen baseline integrity check failed."
    )

TRAINING 40% PRUNED CNN CANDIDATE
Epoch 1/5
4655/4655 [==============================] - 633s 134ms/step - loss: 0.4158 - accuracy: 0.7946
Epoch 2/5
4655/4655 [==============================] - 630s 135ms/step - loss: 0.3884 - accuracy: 0.8088
Epoch 3/5
4655/4655 [==============================] - 625s 134ms/step - loss: 0.3760 - accuracy: 0.8121
Epoch 4/5
4655/4655 [==============================] - 636s 137ms/step - loss: 0.3669 - accuracy: 0.8155
Epoch 5/5
4655/4655 [==============================] - 634s 136ms/step - loss: 0.3598 - accuracy: 0.8212

40% PRUNING TRAINING COMPLETED

Optimizer: Adam
Learning rate: 0.001000000047
Training samples: 297,867
Batch size: 64
Steps per epoch: 4,655
Epochs completed: 5
Training time: 3158.5017 seconds

Final training metrics
--------------------------------------------------------------------------------
Loss: 0.359794
Accuracy: 0.821182

Frozen baseline verification
----------------------------------------------------------------------------

In [ ]:
# Cell 19

# Evaluate the trained 40% pruned CNN on the frozen validation split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 80)
print("40% PRUNED CNN — VALIDATION EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_40",
    "pruning_candidate_ready_40",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_40:
    raise RuntimeError(
        "The 40% pruning candidate did not pass "
        "Cell 17 validation."
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

# ------------------------------------------------------------
# 2. Verify validation metadata
# ------------------------------------------------------------

required_validation_columns = [
    "WakeSleepLabel",
    "ParticipantID"
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in dl_validation.columns
]

if missing_validation_columns:
    raise RuntimeError(
        "Required validation columns are missing: "
        + ", ".join(missing_validation_columns)
    )

validation_samples_40 = len(
    dl_validation
)

validation_participants_40 = (
    dl_validation[
        "ParticipantID"
    ].nunique()
)

if validation_samples_40 <= 0:
    raise RuntimeError(
        "Validation sample count must be positive."
    )

if validation_participants_40 <= 0:
    raise RuntimeError(
        "Validation participant count must be positive."
    )

# ------------------------------------------------------------
# 3. Create a fresh validation generator
# ------------------------------------------------------------

pruned_validation_generator_40 = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 4. Determine exact validation steps
# ------------------------------------------------------------

validation_steps_40 = int(
    np.ceil(
        validation_samples_40
        / PRUNING_BATCH_SIZE
    )
)

if validation_steps_40 <= 0:
    raise RuntimeError(
        "Validation step count must be positive."
    )

# ------------------------------------------------------------
# 5. Generate validation predictions
# ------------------------------------------------------------

evaluation_start_time_40 = (
    time.perf_counter()
)

validation_probabilities_40 = (
    pruned_cnn_40.predict(
        pruned_validation_generator_40,
        steps=validation_steps_40,
        verbose=0
    )
)

evaluation_time_40 = (
    time.perf_counter()
    - evaluation_start_time_40
)

# ------------------------------------------------------------
# 6. Convert predictions to one-dimensional array
# ------------------------------------------------------------

validation_probabilities_40 = np.asarray(
    validation_probabilities_40,
    dtype=np.float64
).reshape(-1)

# The final batch may contain padding depending on
# the generator implementation, so retain exactly the
# number of real validation samples.

validation_probabilities_40 = (
    validation_probabilities_40[
        :validation_samples_40
    ]
)

if len(
    validation_probabilities_40
) != validation_samples_40:
    raise RuntimeError(
        "Prediction count does not match the "
        "validation sample count."
    )

# ------------------------------------------------------------
# 7. Validate prediction values
# ------------------------------------------------------------

if not np.isfinite(
    validation_probabilities_40
).all():
    raise ValueError(
        "Non-finite validation predictions detected."
    )

if (
    (validation_probabilities_40 < 0.0)
    |
    (validation_probabilities_40 > 1.0)
).any():
    raise ValueError(
        "Prediction probabilities outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 8. Obtain validation labels
# ------------------------------------------------------------

validation_labels_40 = (
    dl_validation[
        "WakeSleepLabel"
    ].to_numpy(
        dtype=np.int32
    )
)

if len(
    validation_labels_40
) != validation_samples_40:
    raise RuntimeError(
        "Validation label count does not match "
        "validation sample count."
    )

unique_validation_labels_40 = np.unique(
    validation_labels_40
)

if not set(
    unique_validation_labels_40
).issubset({0, 1}):
    raise ValueError(
        "Validation labels are not binary. "
        f"Observed labels: "
        f"{unique_validation_labels_40}"
    )

if len(
    unique_validation_labels_40
) < 2:
    raise ValueError(
        "ROC-AUC cannot be calculated because "
        "the validation set contains only one class."
    )

# ------------------------------------------------------------
# 9. Apply the predefined decision threshold
# ------------------------------------------------------------

VALIDATION_THRESHOLD_40 = 0.50

validation_predictions_40 = (
    validation_probabilities_40
    >= VALIDATION_THRESHOLD_40
).astype(np.int32)

# ------------------------------------------------------------
# 10. Calculate validation metrics
# ------------------------------------------------------------

pruned_validation_accuracy_40 = (
    accuracy_score(
        validation_labels_40,
        validation_predictions_40
    )
)

pruned_validation_precision_40 = (
    precision_score(
        validation_labels_40,
        validation_predictions_40,
        zero_division=0
    )
)

pruned_validation_recall_40 = (
    recall_score(
        validation_labels_40,
        validation_predictions_40,
        zero_division=0
    )
)

pruned_validation_f1_40 = (
    f1_score(
        validation_labels_40,
        validation_predictions_40,
        zero_division=0
    )
)

pruned_validation_roc_auc_40 = (
    roc_auc_score(
        validation_labels_40,
        validation_probabilities_40
    )
)

# ------------------------------------------------------------
# 11. Store measured results
# ------------------------------------------------------------

pruned_validation_results_40 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Validation samples",
        "Validation participants",
        "Evaluation time (seconds)",
        "Decision threshold",
        "Target sparsity"
    ],
    "Measured value": [
        pruned_validation_accuracy_40,
        pruned_validation_precision_40,
        pruned_validation_recall_40,
        pruned_validation_f1_40,
        pruned_validation_roc_auc_40,
        validation_samples_40,
        validation_participants_40,
        evaluation_time_40,
        VALIDATION_THRESHOLD_40,
        PRUNING_TARGET_SPARSITY_40
    ]
})

# ------------------------------------------------------------
# 12. Display results
# ------------------------------------------------------------

print("\nValidation results")
print("-" * 80)

display(
    pruned_validation_results_40
)

print("\n" + "=" * 80)
print("40% PRUNED CNN — VALIDATION EVALUATION COMPLETED")
print("=" * 80)

print(
    f"\nValidation samples: "
    f"{validation_samples_40:,}"
)

print(
    f"Validation participants: "
    f"{validation_participants_40}"
)

print(
    f"Prediction/evaluation time: "
    f"{evaluation_time_40:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

40% PRUNED CNN — VALIDATION EVALUATION

Validation results
--------------------------------------------------------------------------------


,Metric,Measured value
0,Accuracy,0.732007
1,Precision,0.878663
2,Recall,0.782696
3,F1,0.827908
4,ROC-AUC,0.764577
5,Validation samples,57718.000000
6,Validation participants,4.000000
7,Evaluation time (seconds),82.298652
8,Decision threshold,0.500000
9,Target sparsity,0.400000



40% PRUNED CNN — VALIDATION EVALUATION COMPLETED

Validation samples: 57,718
Validation participants: 4
Prediction/evaluation time: 82.2987 seconds

Test data was NOT accessed.


In [ ]:
# Cell 20

# Analyse sparsity and efficiency of the trained 40% pruned CNN

print("=" * 80)
print("40% PRUNED CNN — SPARSITY AND EFFICIENCY ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_40",
    "pruning_candidate_ready_40",
    "pruned_validation_results_40",
    "evaluation_time_40",
    "PRUNING_TARGET_SPARSITY_40"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_40:
    raise RuntimeError(
        "The 40% pruning candidate did not pass "
        "its construction checks."
    )



stripped_cnn_40 = (
    tfmot.sparsity.keras.strip_pruning(
        pruned_cnn_40
    )
)

# ------------------------------------------------------------
# 3. Collect trainable/effective weight tensors
# ------------------------------------------------------------

effective_weight_arrays_40 = [
    np.asarray(
        weight
    )
    for weight in stripped_cnn_40.get_weights()
]

if not effective_weight_arrays_40:
    raise RuntimeError(
        "No weights were found in the stripped "
        "40% pruned model."
    )

# ------------------------------------------------------------
# 4. Calculate parameter statistics
# ------------------------------------------------------------

dense_parameter_count_40 = int(
    sum(
        weight.size
        for weight in effective_weight_arrays_40
    )
)

if dense_parameter_count_40 != 3809:
    raise RuntimeError(
        "Unexpected parameter count after stripping "
        f"pruning wrappers: {dense_parameter_count_40:,}"
    )

zero_weight_count_40 = int(
    sum(
        np.count_nonzero(
            weight == 0
        )
        for weight in effective_weight_arrays_40
    )
)

nonzero_weight_count_40 = (
    dense_parameter_count_40
    - zero_weight_count_40
)

# ------------------------------------------------------------
# 5. Calculate actual sparsity
# ------------------------------------------------------------

actual_sparsity_40 = (
    zero_weight_count_40
    / dense_parameter_count_40
)

# ------------------------------------------------------------
# 6. Verify sparsity is mathematically valid
# ------------------------------------------------------------

if not (
    0.0 <= actual_sparsity_40 <= 1.0
):
    raise RuntimeError(
        "Calculated sparsity is outside [0, 1]."
    )

if (
    zero_weight_count_40
    + nonzero_weight_count_40
    != dense_parameter_count_40
):
    raise RuntimeError(
        "Zero and non-zero parameter counts do not "
        "sum to the dense parameter count."
    )




FLOAT32_BYTES_PER_PARAMETER = 4

dense_float32_storage_bytes_40 = (
    dense_parameter_count_40
    * FLOAT32_BYTES_PER_PARAMETER
)

dense_float32_storage_kb_40 = (
    dense_float32_storage_bytes_40
    / 1024
)

dense_float32_storage_mb_40 = (
    dense_float32_storage_kb_40
    / 1024
)

# ------------------------------------------------------------
# 8. Calculate the theoretical proportion of retained
#     non-zero weights
# ------------------------------------------------------------

nonzero_weight_fraction_40 = (
    nonzero_weight_count_40
    / dense_parameter_count_40
)

# ------------------------------------------------------------
# 9. Extract the measured validation metrics
# ------------------------------------------------------------

validation_metric_map_40 = dict(
    zip(
        pruned_validation_results_40["Metric"],
        pruned_validation_results_40["Measured value"]
    )
)

required_metrics_40 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

missing_metrics_40 = [
    metric
    for metric in required_metrics_40
    if metric not in validation_metric_map_40
]

if missing_metrics_40:
    raise RuntimeError(
        "Validation metrics are missing: "
        + ", ".join(missing_metrics_40)
    )

# ------------------------------------------------------------
# 10. Create efficiency summary
# ------------------------------------------------------------

pruning_efficiency_40 = pd.DataFrame({
    "Metric": [
        "Dense parameter count",
        "Zero weight values",
        "Non-zero weight values",
        "Actual sparsity",
        "Target sparsity",
        "Non-zero weight fraction",
        "Dense Float32 storage (KB)",
        "Dense Float32 storage (MB)",
        "Validation accuracy",
        "Validation precision",
        "Validation recall",
        "Validation F1",
        "Validation ROC-AUC",
        "Validation evaluation time (seconds)"
    ],
    "40% pruning": [
        dense_parameter_count_40,
        zero_weight_count_40,
        nonzero_weight_count_40,
        actual_sparsity_40,
        PRUNING_TARGET_SPARSITY_40,
        nonzero_weight_fraction_40,
        dense_float32_storage_kb_40,
        dense_float32_storage_mb_40,
        validation_metric_map_40["Accuracy"],
        validation_metric_map_40["Precision"],
        validation_metric_map_40["Recall"],
        validation_metric_map_40["F1"],
        validation_metric_map_40["ROC-AUC"],
        evaluation_time_40
    ]
})

# ------------------------------------------------------------
# 11. Display results
# ------------------------------------------------------------

display(
    pruning_efficiency_40
)

print("\n" + "=" * 80)
print("40% PRUNING ANALYSIS COMPLETED")
print("=" * 80)

print(
    f"\nDense parameters: "
    f"{dense_parameter_count_40:,}"
)

print(
    f"Zero weights: "
    f"{zero_weight_count_40:,}"
)

print(
    f"Non-zero weights: "
    f"{nonzero_weight_count_40:,}"
)

print(
    f"Actual sparsity: "
    f"{actual_sparsity_40:.4%}"
)

print(
    f"Target sparsity: "
    f"{PRUNING_TARGET_SPARSITY_40:.2%}"
)

print(
    f"Dense Float32 storage: "
    f"{dense_float32_storage_kb_40:.6f} KB"
)

print(
    f"Dense Float32 storage: "
    f"{dense_float32_storage_mb_40:.8f} MB"
)

print(
    f"Validation F1: "
    f"{validation_metric_map_40['F1']:.6f}"
)

print(
    f"Validation ROC-AUC: "
    f"{validation_metric_map_40['ROC-AUC']:.6f}"
)

print(
    f"Validation evaluation time: "
    f"{evaluation_time_40:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

40% PRUNED CNN — SPARSITY AND EFFICIENCY ANALYSIS


,Metric,40% pruning
0,Dense parameter count,3809.000000
1,Zero weight values,1493.000000
2,Non-zero weight values,2316.000000
3,Actual sparsity,0.391966
4,Target sparsity,0.400000
5,Non-zero weight fraction,0.608034
6,Dense Float32 storage (KB),14.878906
7,Dense Float32 storage (MB),0.014530
8,Validation accuracy,0.732007
9,Validation precision,0.878663



40% PRUNING ANALYSIS COMPLETED

Dense parameters: 3,809
Zero weights: 1,493
Non-zero weights: 2,316
Actual sparsity: 39.1966%
Target sparsity: 40.00%
Dense Float32 storage: 14.878906 KB
Dense Float32 storage: 0.01453018 MB
Validation F1: 0.827908
Validation ROC-AUC: 0.764577
Validation evaluation time: 82.2987 seconds

Test data was NOT accessed.


In [ ]:
# Cell 21

# Construct and validate the 60% pruning candidate
# directly from the frozen baseline CNN

print("=" * 80)
print("CONSTRUCTING 60% PRUNING CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "baseline_cnn",
    "dl_train",
    "PRUNING_BATCH_SIZE",
    "PRUNING_EPOCHS",
    "PRUNING_STEPS_PER_EPOCH",
    "tfmot",
    "pruning_keras",
    "PRUNING_AVAILABLE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not PRUNING_AVAILABLE:
    raise RuntimeError(
        "TensorFlow Model Optimization Toolkit "
        "is not available."
    )

# ------------------------------------------------------------
# 2. Define target sparsity
# ------------------------------------------------------------

PRUNING_TARGET_SPARSITY_60 = 0.60

if not (
    0.0 < PRUNING_TARGET_SPARSITY_60 < 1.0
):
    raise ValueError(
        "Target sparsity must be strictly between 0 and 1."
    )

# ------------------------------------------------------------
# 3. Verify frozen baseline
# ------------------------------------------------------------

baseline_parameter_count_60 = (
    baseline_cnn.count_params()
)

if baseline_parameter_count_60 != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809. "
        f"Observed: {baseline_parameter_count_60:,}"
    )

if baseline_cnn.input_shape != (
    None,
    2500,
    1
):
    raise RuntimeError(
        "Unexpected frozen baseline input shape: "
        f"{baseline_cnn.input_shape}"
    )

# ------------------------------------------------------------
# 4. Capture an exact copy of the frozen baseline weights
# ------------------------------------------------------------

baseline_weights_60 = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

# ------------------------------------------------------------
# 5. Construct a completely fresh CNN
#
# This model is intentionally independent of:
#     pruned_cnn_20
#     pruned_cnn_40
#     pruning_base_cnn
#     pruning_base_cnn_40
#
# Therefore no previous pruning experiment can contaminate
# the 60% candidate.
# ------------------------------------------------------------

pruning_base_cnn_60 = (
    pruning_keras.Sequential([
        pruning_keras.layers.Input(
            shape=(2500, 1)
        ),

        pruning_keras.layers.Conv1D(
            filters=16,
            kernel_size=7,
            activation="relu",
            padding="same"
        ),

        pruning_keras.layers.MaxPooling1D(
            pool_size=2
        ),

        pruning_keras.layers.Conv1D(
            filters=32,
            kernel_size=5,
            activation="relu",
            padding="same"
        ),

        pruning_keras.layers.GlobalAveragePooling1D(),

        pruning_keras.layers.Dense(
            32,
            activation="relu"
        ),

        pruning_keras.layers.Dropout(
            0.3
        ),

        pruning_keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])
)

# ------------------------------------------------------------
# 6. Verify architecture
# ------------------------------------------------------------

if (
    pruning_base_cnn_60.count_params()
    != baseline_parameter_count_60
):
    raise RuntimeError(
        "The fresh 60% pruning base model does not "
        "match the frozen baseline parameter count."
    )

if (
    pruning_base_cnn_60.input_shape
    != baseline_cnn.input_shape
):
    raise RuntimeError(
        "The fresh 60% pruning base model does not "
        "match the frozen baseline input shape."
    )

# ------------------------------------------------------------
# 7. Transfer ONLY the frozen baseline weights
# ------------------------------------------------------------

pruning_base_cnn_60.set_weights(
    baseline_weights_60
)

# ------------------------------------------------------------
# 8. Verify exact weight equality
# ------------------------------------------------------------

candidate_weights_60 = [
    np.array(
        weight,
        copy=True
    )
    for weight in pruning_base_cnn_60.get_weights()
]

if len(candidate_weights_60) != len(
    baseline_weights_60
):
    raise RuntimeError(
        "Baseline and 60% candidate contain different "
        "numbers of weight tensors."
    )

weight_mismatches_60 = []

for index, (
    baseline_weight,
    candidate_weight
) in enumerate(
    zip(
        baseline_weights_60,
        candidate_weights_60
    )
):

    if not np.array_equal(
        baseline_weight,
        candidate_weight
    ):
        weight_mismatches_60.append({
            "tensor_index": index,
            "shape": baseline_weight.shape,
            "max_absolute_difference": float(
                np.max(
                    np.abs(
                        baseline_weight
                        - candidate_weight
                    )
                )
            )
        })

if weight_mismatches_60:

    print("\nWeight mismatches detected:")

    for mismatch in weight_mismatches_60:
        print(mismatch)

    raise RuntimeError(
        "The 60% candidate does not contain an exact "
        "copy of the frozen baseline weights."
    )

weights_match_60 = True

# ------------------------------------------------------------
# 9. Define 60% pruning schedule
# ------------------------------------------------------------

PRUNING_BEGIN_STEP_60 = 0

PRUNING_END_STEP_60 = (
    PRUNING_STEPS_PER_EPOCH
    * PRUNING_EPOCHS
)

if PRUNING_END_STEP_60 <= 0:
    raise RuntimeError(
        "Invalid pruning end step."
    )

pruning_schedule_60 = (
    tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=PRUNING_TARGET_SPARSITY_60,
        begin_step=PRUNING_BEGIN_STEP_60,
        end_step=PRUNING_END_STEP_60
    )
)

# ------------------------------------------------------------
# 10. Apply magnitude-based pruning
# ------------------------------------------------------------

pruned_cnn_60 = (
    tfmot.sparsity.keras.prune_low_magnitude(
        pruning_base_cnn_60,
        pruning_schedule=pruning_schedule_60
    )
)

# ------------------------------------------------------------
# 11. Verify pruning wrappers
# ------------------------------------------------------------

pruning_wrapper_count_60 = sum(
    1
    for layer in pruned_cnn_60.layers
    if "PruneLowMagnitude"
    in layer.__class__.__name__
)

if pruning_wrapper_count_60 <= 0:
    raise RuntimeError(
        "No pruning wrappers were detected "
        "in the 60% candidate."
    )

# ------------------------------------------------------------
# 12. Verify frozen baseline was not modified
# ------------------------------------------------------------

baseline_weights_after_60 = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

baseline_unchanged_60 = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_60,
        baseline_weights_after_60
    )
)

if not baseline_unchanged_60:
    raise RuntimeError(
        "The frozen baseline was modified while "
        "constructing the 60% candidate."
    )

# ------------------------------------------------------------
# 13. Final readiness check
# ------------------------------------------------------------

pruning_candidate_ready_60 = (
    weights_match_60
    and
    baseline_unchanged_60
    and
    pruning_base_cnn_60.count_params()
    == 3809
    and
    pruning_base_cnn_60.input_shape
    == (None, 2500, 1)
    and
    pruning_wrapper_count_60 > 0
)

# ------------------------------------------------------------
# 14. Report
# ------------------------------------------------------------

print("\nPruning configuration")
print("-" * 80)

print(
    f"Target sparsity:       "
    f"{PRUNING_TARGET_SPARSITY_60:.0%}"
)

print(
    f"Training samples:      "
    f"{len(dl_train):,}"
)

print(
    f"Batch size:            "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Steps per epoch:       "
    f"{PRUNING_STEPS_PER_EPOCH:,}"
)

print(
    f"Pruning epochs:        "
    f"{PRUNING_EPOCHS}"
)

print(
    f"Pruning end step:      "
    f"{PRUNING_END_STEP_60:,}"
)

print(
    f"Baseline parameters:   "
    f"{baseline_parameter_count_60:,}"
)

print(
    f"Candidate parameters:  "
    f"{pruning_base_cnn_60.count_params():,}"
)

print(
    f"Pruning wrappers:      "
    f"{pruning_wrapper_count_60}"
)

print("\n" + "=" * 80)

print(
    "Frozen weight transfer:",
    "PASSED"
    if weights_match_60
    else "FAILED"
)

print(
    "Architecture match:",
    "PASSED"
    if pruning_base_cnn_60.count_params() == 3809
    else "FAILED"
)

print(
    "Input shape match:",
    "PASSED"
    if pruning_base_cnn_60.input_shape
    == (None, 2500, 1)
    else "FAILED"
)

print(
    "Pruning wrapper construction:",
    "PASSED"
    if pruning_wrapper_count_60 > 0
    else "FAILED"
)

print(
    "Frozen baseline unchanged:",
    "PASSED"
    if baseline_unchanged_60
    else "FAILED"
)

print("-" * 80)

print(
    "60% PRUNING CANDIDATE:",
    "READY"
    if pruning_candidate_ready_60
    else "FAILED"
)

print("=" * 80)

if not pruning_candidate_ready_60:
    raise RuntimeError(
        "60% pruning candidate validation failed."
    )

CONSTRUCTING 60% PRUNING CANDIDATE

Pruning configuration
--------------------------------------------------------------------------------
Target sparsity:       60%
Training samples:      297,867
Batch size:            64
Steps per epoch:       4,655
Pruning epochs:        5
Pruning end step:      23,275
Baseline parameters:   3,809
Candidate parameters:  3,809
Pruning wrappers:      7

Frozen weight transfer: PASSED
Architecture match: PASSED
Input shape match: PASSED
Pruning wrapper construction: PASSED
Frozen baseline unchanged: PASSED
--------------------------------------------------------------------------------
60% PRUNING CANDIDATE: READY


In [ ]:
# Cell 22

# Train the 60% magnitude-pruned CNN candidate

print("=" * 80)
print("TRAINING 60% PRUNED CNN CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_60",
    "pruning_candidate_ready_60",
    "baseline_cnn",
    "dl_train",
    "PRUNING_BATCH_SIZE",
    "PRUNING_EPOCHS",
    "PRUNING_STEPS_PER_EPOCH",
    "generate_weighted_cnn_batches",
    "tfmot",
    "pruning_keras",
    "PRUNING_AVAILABLE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not PRUNING_AVAILABLE:
    raise RuntimeError(
        "TensorFlow Model Optimization Toolkit "
        "is not available."
    )

if not pruning_candidate_ready_60:
    raise RuntimeError(
        "The 60% pruning candidate did not pass "
        "Cell 21 validation."
    )

if dl_train.empty:
    raise ValueError(
        "Training dataset is empty."
    )

# ------------------------------------------------------------
# 2. Preserve the frozen baseline before training
# ------------------------------------------------------------

baseline_weights_before_60_training = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

baseline_parameter_count_before_60_training = (
    baseline_cnn.count_params()
)

if baseline_parameter_count_before_60_training != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809."
    )

# ------------------------------------------------------------
# 3. Obtain optimizer configuration from the frozen baseline
# ------------------------------------------------------------

if not hasattr(
    baseline_cnn,
    "optimizer"
):
    raise RuntimeError(
        "Frozen baseline optimizer is unavailable."
    )

if baseline_cnn.optimizer is None:
    raise RuntimeError(
        "Frozen baseline optimizer is None."
    )

baseline_optimizer_name_60 = (
    baseline_cnn.optimizer.__class__.__name__
)

baseline_optimizer_config_60 = (
    baseline_cnn.optimizer.get_config()
)

optimizer_learning_rate_60 = (
    baseline_optimizer_config_60.get(
        "learning_rate"
    )
)

if optimizer_learning_rate_60 is None:
    raise RuntimeError(
        "Could not determine the frozen baseline "
        "learning rate."
    )

# ------------------------------------------------------------
# 4. Create a completely fresh optimizer
# ------------------------------------------------------------

if baseline_optimizer_name_60 == "Adam":

    pruning_optimizer_60 = (
        pruning_keras.optimizers.Adam(
            learning_rate=optimizer_learning_rate_60
        )
    )

elif baseline_optimizer_name_60 == "RMSprop":

    pruning_optimizer_60 = (
        pruning_keras.optimizers.RMSprop(
            learning_rate=optimizer_learning_rate_60
        )
    )

elif baseline_optimizer_name_60 == "SGD":

    pruning_optimizer_60 = (
        pruning_keras.optimizers.SGD(
            learning_rate=optimizer_learning_rate_60
        )
    )

else:

    raise RuntimeError(
        "Unsupported frozen baseline optimizer: "
        f"{baseline_optimizer_name_60}"
    )

# ------------------------------------------------------------
# 5. Verify the fresh optimizer
# ------------------------------------------------------------

fresh_learning_rate_60 = float(
    pruning_optimizer_60.learning_rate.numpy()
)

if not np.isclose(
    fresh_learning_rate_60,
    float(optimizer_learning_rate_60)
):
    raise RuntimeError(
        "Fresh optimizer learning rate does not "
        "match the frozen baseline learning rate."
    )

# ------------------------------------------------------------
# 6. Compile the 60% pruning candidate
# ------------------------------------------------------------

pruned_cnn_60.compile(
    optimizer=pruning_optimizer_60,
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)

# ------------------------------------------------------------
# 7. Create a fresh real-data training generator
# ------------------------------------------------------------

pruning_train_generator_60 = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

# ------------------------------------------------------------
# 8. Verify one real training batch
# ------------------------------------------------------------

X_train_check_60, y_train_check_60, weights_check_60 = (
    next(
        pruning_train_generator_60
    )
)

if X_train_check_60.ndim != 3:
    raise RuntimeError(
        "Training ECG batch must be 3-dimensional."
    )

if X_train_check_60.shape[1:] != (
    2500,
    1
):
    raise RuntimeError(
        "Unexpected ECG batch shape: "
        f"{X_train_check_60.shape}"
    )

if y_train_check_60.ndim != 1:
    raise RuntimeError(
        "Training labels must be one-dimensional."
    )

if len(X_train_check_60) != len(
    y_train_check_60
):
    raise RuntimeError(
        "Training samples and labels have "
        "different lengths."
    )

if len(X_train_check_60) != len(
    weights_check_60
):
    raise RuntimeError(
        "Training samples and sample weights have "
        "different lengths."
    )

if not np.isfinite(
    X_train_check_60
).all():
    raise ValueError(
        "Non-finite values detected in training ECG."
    )

if not np.isfinite(
    y_train_check_60
).all():
    raise ValueError(
        "Non-finite values detected in training labels."
    )

if not np.isfinite(
    weights_check_60
).all():
    raise ValueError(
        "Non-finite values detected in training weights."
    )

# ------------------------------------------------------------
# 9. Recreate generator after verification
# ------------------------------------------------------------

pruning_train_generator_60 = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

# ------------------------------------------------------------
# 10. Create a fresh pruning callback
# ------------------------------------------------------------

pruning_callback_60 = (
    tfmot.sparsity.keras.UpdatePruningStep()
)

# ------------------------------------------------------------
# 11. Verify training configuration
# ------------------------------------------------------------

training_steps_60 = (
    PRUNING_STEPS_PER_EPOCH
)

training_epochs_60 = (
    PRUNING_EPOCHS
)

if training_steps_60 <= 0:
    raise RuntimeError(
        "Training steps must be positive."
    )

if training_epochs_60 <= 0:
    raise RuntimeError(
        "Training epochs must be positive."
    )

# ------------------------------------------------------------
# 12. Train the 60% candidate
# ------------------------------------------------------------

training_start_time_60 = (
    time.perf_counter()
)

pruning_history_60 = (
    pruned_cnn_60.fit(
        pruning_train_generator_60,
        steps_per_epoch=training_steps_60,
        epochs=training_epochs_60,
        callbacks=[
            pruning_callback_60
        ],
        verbose=1
    )
)

pruning_training_time_60 = (
    time.perf_counter()
    - training_start_time_60
)

# ------------------------------------------------------------
# 13. Verify training history
# ------------------------------------------------------------

if not hasattr(
    pruning_history_60,
    "history"
):
    raise RuntimeError(
        "Training did not return a valid history object."
    )

history_keys_60 = set(
    pruning_history_60.history.keys()
)

if "loss" not in history_keys_60:
    raise RuntimeError(
        "Training history does not contain loss."
    )

if "accuracy" not in history_keys_60:
    raise RuntimeError(
        "Training history does not contain accuracy."
    )

completed_epochs_60 = len(
    pruning_history_60.history["loss"]
)

if completed_epochs_60 != training_epochs_60:
    raise RuntimeError(
        "Unexpected number of completed epochs: "
        f"{completed_epochs_60}; "
        f"expected {training_epochs_60}."
    )

# ------------------------------------------------------------
# 14. Verify frozen baseline integrity
# ------------------------------------------------------------

baseline_weights_after_60_training = (
    baseline_cnn.get_weights()
)

baseline_unchanged_60_training = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_before_60_training,
        baseline_weights_after_60_training
    )
)

if not baseline_unchanged_60_training:
    raise RuntimeError(
        "Frozen baseline weights were modified "
        "during 60% pruning training."
    )

if (
    baseline_cnn.count_params()
    != baseline_parameter_count_before_60_training
):
    raise RuntimeError(
        "Frozen baseline parameter count changed "
        "during 60% pruning training."
    )

# ------------------------------------------------------------
# 15. Report training results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("60% PRUNING TRAINING COMPLETED")
print("=" * 80)

print(
    f"\nOptimizer: "
    f"{baseline_optimizer_name_60}"
)

print(
    f"Learning rate: "
    f"{fresh_learning_rate_60:.10g}"
)

print(
    f"Training samples: "
    f"{len(dl_train):,}"
)

print(
    f"Batch size: "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Steps per epoch: "
    f"{training_steps_60:,}"
)

print(
    f"Epochs completed: "
    f"{completed_epochs_60}"
)

print(
    f"Training time: "
    f"{pruning_training_time_60:.4f} seconds"
)

print("\nFinal training metrics")
print("-" * 80)

print(
    f"Loss: "
    f"{pruning_history_60.history['loss'][-1]:.6f}"
)

print(
    f"Accuracy: "
    f"{pruning_history_60.history['accuracy'][-1]:.6f}"
)

print("\nFrozen baseline verification")
print("-" * 80)

print(
    "Baseline weights unchanged:",
    "PASSED"
    if baseline_unchanged_60_training
    else "FAILED"
)

print(
    "Baseline parameter count unchanged:",
    "PASSED"
    if (
        baseline_cnn.count_params()
        == baseline_parameter_count_before_60_training
    )
    else "FAILED"
)

print(
    "\nTest data was NOT used."
)

print("=" * 80)

if not baseline_unchanged_60_training:
    raise RuntimeError(
        "Frozen baseline integrity check failed."
    )

TRAINING 60% PRUNED CNN CANDIDATE
Epoch 1/5
4655/4655 [==============================] - 657s 139ms/step - loss: 0.4159 - accuracy: 0.7944
Epoch 2/5
4655/4655 [==============================] - 644s 138ms/step - loss: 0.3965 - accuracy: 0.7980
Epoch 3/5
4655/4655 [==============================] - 637s 137ms/step - loss: 0.3910 - accuracy: 0.7984
Epoch 4/5
4655/4655 [==============================] - 639s 137ms/step - loss: 0.3839 - accuracy: 0.8072
Epoch 5/5
4655/4655 [==============================] - 638s 137ms/step - loss: 0.3725 - accuracy: 0.8133

60% PRUNING TRAINING COMPLETED

Optimizer: Adam
Learning rate: 0.001000000047
Training samples: 297,867
Batch size: 64
Steps per epoch: 4,655
Epochs completed: 5
Training time: 3214.9639 seconds

Final training metrics
--------------------------------------------------------------------------------
Loss: 0.372526
Accuracy: 0.813309

Frozen baseline verification
----------------------------------------------------------------------------

In [ ]:
# Cell 23

# Evaluate the trained 60% pruned CNN on the frozen validation split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 80)
print("60% PRUNED CNN — VALIDATION EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_60",
    "pruning_candidate_ready_60",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_60:
    raise RuntimeError(
        "The 60% pruning candidate did not pass "
        "Cell 21 validation."
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

# ------------------------------------------------------------
# 2. Verify validation metadata
# ------------------------------------------------------------

required_validation_columns = [
    "WakeSleepLabel",
    "ParticipantID"
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in dl_validation.columns
]

if missing_validation_columns:
    raise RuntimeError(
        "Required validation columns are missing: "
        + ", ".join(missing_validation_columns)
    )

validation_samples_60 = len(
    dl_validation
)

validation_participants_60 = (
    dl_validation[
        "ParticipantID"
    ].nunique()
)

if validation_samples_60 <= 0:
    raise RuntimeError(
        "Validation sample count must be positive."
    )

if validation_participants_60 <= 0:
    raise RuntimeError(
        "Validation participant count must be positive."
    )

# ------------------------------------------------------------
# 3. Create a fresh validation generator
# ------------------------------------------------------------

pruned_validation_generator_60 = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 4. Determine exact validation steps
# ------------------------------------------------------------

validation_steps_60 = int(
    np.ceil(
        validation_samples_60
        / PRUNING_BATCH_SIZE
    )
)

if validation_steps_60 <= 0:
    raise RuntimeError(
        "Validation step count must be positive."
    )

# ------------------------------------------------------------
# 5. Generate validation predictions
# ------------------------------------------------------------

evaluation_start_time_60 = (
    time.perf_counter()
)

validation_probabilities_60 = (
    pruned_cnn_60.predict(
        pruned_validation_generator_60,
        steps=validation_steps_60,
        verbose=0
    )
)

evaluation_time_60 = (
    time.perf_counter()
    - evaluation_start_time_60
)

# ------------------------------------------------------------
# 6. Convert predictions to one-dimensional array
# ------------------------------------------------------------

validation_probabilities_60 = np.asarray(
    validation_probabilities_60,
    dtype=np.float64
).reshape(-1)

validation_probabilities_60 = (
    validation_probabilities_60[
        :validation_samples_60
    ]
)

if len(
    validation_probabilities_60
) != validation_samples_60:
    raise RuntimeError(
        "Prediction count does not match the "
        "validation sample count."
    )

# ------------------------------------------------------------
# 7. Validate prediction values
# ------------------------------------------------------------

if not np.isfinite(
    validation_probabilities_60
).all():
    raise ValueError(
        "Non-finite validation predictions detected."
    )

if (
    (validation_probabilities_60 < 0.0)
    |
    (validation_probabilities_60 > 1.0)
).any():
    raise ValueError(
        "Prediction probabilities outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 8. Obtain validation labels
# ------------------------------------------------------------

validation_labels_60 = (
    dl_validation[
        "WakeSleepLabel"
    ].to_numpy(
        dtype=np.int32
    )
)

if len(
    validation_labels_60
) != validation_samples_60:
    raise RuntimeError(
        "Validation label count does not match "
        "validation sample count."
    )

unique_validation_labels_60 = np.unique(
    validation_labels_60
)

if not set(
    unique_validation_labels_60
).issubset({0, 1}):
    raise ValueError(
        "Validation labels are not binary. "
        f"Observed labels: "
        f"{unique_validation_labels_60}"
    )

if len(
    unique_validation_labels_60
) < 2:
    raise ValueError(
        "ROC-AUC cannot be calculated because "
        "the validation set contains only one class."
    )

# ------------------------------------------------------------
# 9. Apply the predefined decision threshold
# ------------------------------------------------------------

VALIDATION_THRESHOLD_60 = 0.50

validation_predictions_60 = (
    validation_probabilities_60
    >= VALIDATION_THRESHOLD_60
).astype(np.int32)

# ------------------------------------------------------------
# 10. Calculate validation metrics
# ------------------------------------------------------------

pruned_validation_accuracy_60 = (
    accuracy_score(
        validation_labels_60,
        validation_predictions_60
    )
)

pruned_validation_precision_60 = (
    precision_score(
        validation_labels_60,
        validation_predictions_60,
        zero_division=0
    )
)

pruned_validation_recall_60 = (
    recall_score(
        validation_labels_60,
        validation_predictions_60,
        zero_division=0
    )
)

pruned_validation_f1_60 = (
    f1_score(
        validation_labels_60,
        validation_predictions_60,
        zero_division=0
    )
)

pruned_validation_roc_auc_60 = (
    roc_auc_score(
        validation_labels_60,
        validation_probabilities_60
    )
)

# ------------------------------------------------------------
# 11. Store measured results
# ------------------------------------------------------------

pruned_validation_results_60 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Validation samples",
        "Validation participants",
        "Evaluation time (seconds)",
        "Decision threshold",
        "Target sparsity"
    ],
    "Measured value": [
        pruned_validation_accuracy_60,
        pruned_validation_precision_60,
        pruned_validation_recall_60,
        pruned_validation_f1_60,
        pruned_validation_roc_auc_60,
        validation_samples_60,
        validation_participants_60,
        evaluation_time_60,
        VALIDATION_THRESHOLD_60,
        PRUNING_TARGET_SPARSITY_60
    ]
})

# ------------------------------------------------------------
# 12. Display results
# ------------------------------------------------------------

print("\nValidation results")
print("-" * 80)

display(
    pruned_validation_results_60
)

print("\n" + "=" * 80)
print("60% PRUNED CNN — VALIDATION EVALUATION COMPLETED")
print("=" * 80)

print(
    f"\nValidation samples: "
    f"{validation_samples_60:,}"
)

print(
    f"Validation participants: "
    f"{validation_participants_60}"
)

print(
    f"Prediction/evaluation time: "
    f"{evaluation_time_60:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

60% PRUNED CNN — VALIDATION EVALUATION

Validation results
--------------------------------------------------------------------------------


,Metric,Measured value
0,Accuracy,0.749922
1,Precision,0.885501
2,Recall,0.799777
3,F1,0.840459
4,ROC-AUC,0.777945
5,Validation samples,57718.000000
6,Validation participants,4.000000
7,Evaluation time (seconds),51.173384
8,Decision threshold,0.500000
9,Target sparsity,0.600000



60% PRUNED CNN — VALIDATION EVALUATION COMPLETED

Validation samples: 57,718
Validation participants: 4
Prediction/evaluation time: 51.1734 seconds

Test data was NOT accessed.


In [ ]:
# Cell 24

# Analyse sparsity and efficiency of the trained 60% pruned CNN

print("=" * 80)
print("60% PRUNED CNN — SPARSITY AND EFFICIENCY ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_60",
    "pruning_candidate_ready_60",
    "pruned_validation_results_60",
    "evaluation_time_60",
    "PRUNING_TARGET_SPARSITY_60"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_60:
    raise RuntimeError(
        "The 60% pruning candidate did not pass "
        "its construction checks."
    )

# ------------------------------------------------------------
# 2. Strip pruning wrappers from a copy
# ------------------------------------------------------------

stripped_cnn_60 = (
    tfmot.sparsity.keras.strip_pruning(
        pruned_cnn_60
    )
)

# ------------------------------------------------------------
# 3. Collect effective weights
# ------------------------------------------------------------

effective_weight_arrays_60 = [
    np.asarray(weight)
    for weight in stripped_cnn_60.get_weights()
]

if not effective_weight_arrays_60:
    raise RuntimeError(
        "No weights were found in the stripped "
        "60% pruned model."
    )

# ------------------------------------------------------------
# 4. Calculate parameter statistics
# ------------------------------------------------------------

dense_parameter_count_60 = int(
    sum(
        weight.size
        for weight in effective_weight_arrays_60
    )
)

if dense_parameter_count_60 != 3809:
    raise RuntimeError(
        "Unexpected parameter count after stripping "
        f"pruning wrappers: {dense_parameter_count_60:,}"
    )

zero_weight_count_60 = int(
    sum(
        np.count_nonzero(weight == 0)
        for weight in effective_weight_arrays_60
    )
)

nonzero_weight_count_60 = (
    dense_parameter_count_60
    - zero_weight_count_60
)

# ------------------------------------------------------------
# 5. Calculate actual sparsity
# ------------------------------------------------------------

actual_sparsity_60 = (
    zero_weight_count_60
    / dense_parameter_count_60
)

if not (
    0.0 <= actual_sparsity_60 <= 1.0
):
    raise RuntimeError(
        "Calculated sparsity is outside [0, 1]."
    )

if (
    zero_weight_count_60
    + nonzero_weight_count_60
    != dense_parameter_count_60
):
    raise RuntimeError(
        "Zero and non-zero parameter counts do not "
        "sum to the dense parameter count."
    )

# ------------------------------------------------------------
# 6. Calculate dense Float32 storage
# ------------------------------------------------------------

FLOAT32_BYTES_PER_PARAMETER = 4

dense_float32_storage_bytes_60 = (
    dense_parameter_count_60
    * FLOAT32_BYTES_PER_PARAMETER
)

dense_float32_storage_kb_60 = (
    dense_float32_storage_bytes_60
    / 1024
)

dense_float32_storage_mb_60 = (
    dense_float32_storage_kb_60
    / 1024
)

nonzero_weight_fraction_60 = (
    nonzero_weight_count_60
    / dense_parameter_count_60
)

# ------------------------------------------------------------
# 7. Extract validation metrics
# ------------------------------------------------------------

validation_metric_map_60 = dict(
    zip(
        pruned_validation_results_60["Metric"],
        pruned_validation_results_60["Measured value"]
    )
)

required_metrics_60 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

missing_metrics_60 = [
    metric
    for metric in required_metrics_60
    if metric not in validation_metric_map_60
]

if missing_metrics_60:
    raise RuntimeError(
        "Validation metrics are missing: "
        + ", ".join(missing_metrics_60)
    )

# ------------------------------------------------------------
# 8. Create efficiency summary
# ------------------------------------------------------------

pruning_efficiency_60 = pd.DataFrame({
    "Metric": [
        "Dense parameter count",
        "Zero weight values",
        "Non-zero weight values",
        "Actual sparsity",
        "Target sparsity",
        "Non-zero weight fraction",
        "Dense Float32 storage (KB)",
        "Dense Float32 storage (MB)",
        "Validation accuracy",
        "Validation precision",
        "Validation recall",
        "Validation F1",
        "Validation ROC-AUC",
        "Validation evaluation time (seconds)"
    ],
    "60% pruning": [
        dense_parameter_count_60,
        zero_weight_count_60,
        nonzero_weight_count_60,
        actual_sparsity_60,
        PRUNING_TARGET_SPARSITY_60,
        nonzero_weight_fraction_60,
        dense_float32_storage_kb_60,
        dense_float32_storage_mb_60,
        validation_metric_map_60["Accuracy"],
        validation_metric_map_60["Precision"],
        validation_metric_map_60["Recall"],
        validation_metric_map_60["F1"],
        validation_metric_map_60["ROC-AUC"],
        evaluation_time_60
    ]
})

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

display(
    pruning_efficiency_60
)

print("\n" + "=" * 80)
print("60% PRUNING ANALYSIS COMPLETED")
print("=" * 80)

print(
    f"\nDense parameters: "
    f"{dense_parameter_count_60:,}"
)

print(
    f"Zero weights: "
    f"{zero_weight_count_60:,}"
)

print(
    f"Non-zero weights: "
    f"{nonzero_weight_count_60:,}"
)

print(
    f"Actual sparsity: "
    f"{actual_sparsity_60:.4%}"
)

print(
    f"Target sparsity: "
    f"{PRUNING_TARGET_SPARSITY_60:.2%}"
)

print(
    f"Dense Float32 storage: "
    f"{dense_float32_storage_kb_60:.6f} KB"
)

print(
    f"Dense Float32 storage: "
    f"{dense_float32_storage_mb_60:.8f} MB"
)

print(
    f"Validation F1: "
    f"{validation_metric_map_60['F1']:.6f}"
)

print(
    f"Validation ROC-AUC: "
    f"{validation_metric_map_60['ROC-AUC']:.6f}"
)

print(
    f"Validation evaluation time: "
    f"{evaluation_time_60:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

60% PRUNED CNN — SPARSITY AND EFFICIENCY ANALYSIS


,Metric,60% pruning
0,Dense parameter count,3809.000000
1,Zero weight values,2237.000000
2,Non-zero weight values,1572.000000
3,Actual sparsity,0.587293
4,Target sparsity,0.600000
5,Non-zero weight fraction,0.412707
6,Dense Float32 storage (KB),14.878906
7,Dense Float32 storage (MB),0.014530
8,Validation accuracy,0.749922
9,Validation precision,0.885501



60% PRUNING ANALYSIS COMPLETED

Dense parameters: 3,809
Zero weights: 2,237
Non-zero weights: 1,572
Actual sparsity: 58.7293%
Target sparsity: 60.00%
Dense Float32 storage: 14.878906 KB
Dense Float32 storage: 0.01453018 MB
Validation F1: 0.840459
Validation ROC-AUC: 0.777945
Validation evaluation time: 51.1734 seconds

Test data was NOT accessed.


In [ ]:
# Cell 25

# Construct and validate the 80% pruning candidate
# directly from the frozen baseline CNN

print("=" * 80)
print("CONSTRUCTING 80% PRUNING CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "baseline_cnn",
    "PRUNING_STEPS_PER_EPOCH",
    "PRUNING_EPOCHS",
    "tfmot",
    "pruning_keras",
    "PRUNING_AVAILABLE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not PRUNING_AVAILABLE:
    raise RuntimeError(
        "TensorFlow Model Optimization Toolkit "
        "is not available."
    )

# ------------------------------------------------------------
# 2. Define target sparsity
# ------------------------------------------------------------

PRUNING_TARGET_SPARSITY_80 = 0.80

if not (
    0.0 < PRUNING_TARGET_SPARSITY_80 < 1.0
):
    raise ValueError(
        "Target sparsity must be strictly between 0 and 1."
    )

# ------------------------------------------------------------
# 3. Verify frozen baseline
# ------------------------------------------------------------

baseline_parameter_count_80 = (
    baseline_cnn.count_params()
)

if baseline_parameter_count_80 != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809. "
        f"Observed: {baseline_parameter_count_80:,}"
    )

if baseline_cnn.input_shape != (
    None,
    2500,
    1
):
    raise RuntimeError(
        "Unexpected frozen baseline input shape: "
        f"{baseline_cnn.input_shape}"
    )

# ------------------------------------------------------------
# 4. Capture an exact copy of frozen baseline weights
# ------------------------------------------------------------

baseline_weights_80 = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

# ------------------------------------------------------------
# 5. Construct a fresh CNN
# ------------------------------------------------------------

pruning_base_cnn_80 = (
    pruning_keras.Sequential([
        pruning_keras.layers.Input(
            shape=(2500, 1)
        ),

        pruning_keras.layers.Conv1D(
            filters=16,
            kernel_size=7,
            activation="relu",
            padding="same"
        ),

        pruning_keras.layers.MaxPooling1D(
            pool_size=2
        ),

        pruning_keras.layers.Conv1D(
            filters=32,
            kernel_size=5,
            activation="relu",
            padding="same"
        ),

        pruning_keras.layers.GlobalAveragePooling1D(),

        pruning_keras.layers.Dense(
            32,
            activation="relu"
        ),

        pruning_keras.layers.Dropout(
            0.3
        ),

        pruning_keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])
)

# ------------------------------------------------------------
# 6. Verify architecture
# ------------------------------------------------------------

if (
    pruning_base_cnn_80.count_params()
    != baseline_parameter_count_80
):
    raise RuntimeError(
        "The fresh 80% pruning base model does not "
        "match the frozen baseline parameter count."
    )

if (
    pruning_base_cnn_80.input_shape
    != baseline_cnn.input_shape
):
    raise RuntimeError(
        "The fresh 80% pruning base model does not "
        "match the frozen baseline input shape."
    )

# ------------------------------------------------------------
# 7. Transfer only frozen baseline weights
# ------------------------------------------------------------

pruning_base_cnn_80.set_weights(
    baseline_weights_80
)

# ------------------------------------------------------------
# 8. Verify exact weight equality
# ------------------------------------------------------------

candidate_weights_80 = [
    np.array(
        weight,
        copy=True
    )
    for weight in pruning_base_cnn_80.get_weights()
]

if len(candidate_weights_80) != len(
    baseline_weights_80
):
    raise RuntimeError(
        "Baseline and 80% candidate contain different "
        "numbers of weight tensors."
    )

weight_mismatches_80 = []

for index, (
    baseline_weight,
    candidate_weight
) in enumerate(
    zip(
        baseline_weights_80,
        candidate_weights_80
    )
):

    if not np.array_equal(
        baseline_weight,
        candidate_weight
    ):
        weight_mismatches_80.append({
            "tensor_index": index,
            "shape": baseline_weight.shape,
            "max_absolute_difference": float(
                np.max(
                    np.abs(
                        baseline_weight
                        - candidate_weight
                    )
                )
            )
        })

if weight_mismatches_80:

    print("\nWeight mismatches detected:")

    for mismatch in weight_mismatches_80:
        print(mismatch)

    raise RuntimeError(
        "The 80% candidate does not contain an exact "
        "copy of the frozen baseline weights."
    )

weights_match_80 = True

# ------------------------------------------------------------
# 9. Define 80% pruning schedule
# ------------------------------------------------------------

PRUNING_BEGIN_STEP_80 = 0

PRUNING_END_STEP_80 = (
    PRUNING_STEPS_PER_EPOCH
    * PRUNING_EPOCHS
)

if PRUNING_END_STEP_80 <= 0:
    raise RuntimeError(
        "Invalid pruning end step."
    )

pruning_schedule_80 = (
    tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=PRUNING_TARGET_SPARSITY_80,
        begin_step=PRUNING_BEGIN_STEP_80,
        end_step=PRUNING_END_STEP_80
    )
)

# ------------------------------------------------------------
# 10. Apply magnitude-based pruning
# ------------------------------------------------------------

pruned_cnn_80 = (
    tfmot.sparsity.keras.prune_low_magnitude(
        pruning_base_cnn_80,
        pruning_schedule=pruning_schedule_80
    )
)

# ------------------------------------------------------------
# 11. Verify pruning wrappers
# ------------------------------------------------------------

pruning_wrapper_count_80 = sum(
    1
    for layer in pruned_cnn_80.layers
    if "PruneLowMagnitude"
    in layer.__class__.__name__
)

if pruning_wrapper_count_80 <= 0:
    raise RuntimeError(
        "No pruning wrappers were detected "
        "in the 80% candidate."
    )

# ------------------------------------------------------------
# 12. Verify frozen baseline remains unchanged
# ------------------------------------------------------------

baseline_weights_after_80 = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

baseline_unchanged_80 = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_80,
        baseline_weights_after_80
    )
)

if not baseline_unchanged_80:
    raise RuntimeError(
        "The frozen baseline was modified while "
        "constructing the 80% candidate."
    )

# ------------------------------------------------------------
# 13. Final readiness check
# ------------------------------------------------------------

pruning_candidate_ready_80 = (
    weights_match_80
    and
    baseline_unchanged_80
    and
    pruning_base_cnn_80.count_params()
    == 3809
    and
    pruning_base_cnn_80.input_shape
    == (None, 2500, 1)
    and
    pruning_wrapper_count_80 > 0
)

# ------------------------------------------------------------
# 14. Report
# ------------------------------------------------------------

print("\nPruning configuration")
print("-" * 80)

print(
    f"Target sparsity:       "
    f"{PRUNING_TARGET_SPARSITY_80:.0%}"
)

print(
    f"Pruning epochs:        "
    f"{PRUNING_EPOCHS}"
)

print(
    f"Pruning end step:      "
    f"{PRUNING_END_STEP_80:,}"
)

print(
    f"Baseline parameters:   "
    f"{baseline_parameter_count_80:,}"
)

print(
    f"Candidate parameters:  "
    f"{pruning_base_cnn_80.count_params():,}"
)

print(
    f"Pruning wrappers:      "
    f"{pruning_wrapper_count_80}"
)

print("\n" + "=" * 80)

print(
    "Frozen weight transfer:",
    "PASSED"
    if weights_match_80
    else "FAILED"
)

print(
    "Architecture match:",
    "PASSED"
    if pruning_base_cnn_80.count_params() == 3809
    else "FAILED"
)

print(
    "Input shape match:",
    "PASSED"
    if pruning_base_cnn_80.input_shape
    == (None, 2500, 1)
    else "FAILED"
)

print(
    "Pruning wrapper construction:",
    "PASSED"
    if pruning_wrapper_count_80 > 0
    else "FAILED"
)

print(
    "Frozen baseline unchanged:",
    "PASSED"
    if baseline_unchanged_80
    else "FAILED"
)

print("-" * 80)

print(
    "80% PRUNING CANDIDATE:",
    "READY"
    if pruning_candidate_ready_80
    else "FAILED"
)

print("=" * 80)

if not pruning_candidate_ready_80:
    raise RuntimeError(
        "80% pruning candidate validation failed."
    )

CONSTRUCTING 80% PRUNING CANDIDATE

Pruning configuration
--------------------------------------------------------------------------------
Target sparsity:       80%
Pruning epochs:        5
Pruning end step:      23,275
Baseline parameters:   3,809
Candidate parameters:  3,809
Pruning wrappers:      7

Frozen weight transfer: PASSED
Architecture match: PASSED
Input shape match: PASSED
Pruning wrapper construction: PASSED
Frozen baseline unchanged: PASSED
--------------------------------------------------------------------------------
80% PRUNING CANDIDATE: READY


In [ ]:
# Cell 26

# Train the 80% magnitude-pruned CNN candidate

print("=" * 80)
print("TRAINING 80% PRUNED CNN CANDIDATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_80",
    "pruning_candidate_ready_80",
    "baseline_cnn",
    "dl_train",
    "PRUNING_BATCH_SIZE",
    "PRUNING_EPOCHS",
    "PRUNING_STEPS_PER_EPOCH",
    "generate_weighted_cnn_batches",
    "tfmot",
    "pruning_keras",
    "PRUNING_AVAILABLE"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not PRUNING_AVAILABLE:
    raise RuntimeError(
        "TensorFlow Model Optimization Toolkit "
        "is not available."
    )

if not pruning_candidate_ready_80:
    raise RuntimeError(
        "The 80% pruning candidate did not pass "
        "Cell 25 validation."
    )

if dl_train.empty:
    raise ValueError(
        "Training dataset is empty."
    )

# ------------------------------------------------------------
# 2. Preserve the frozen baseline before training
# ------------------------------------------------------------

baseline_weights_before_80_training = [
    np.array(
        weight,
        copy=True
    )
    for weight in baseline_cnn.get_weights()
]

baseline_parameter_count_before_80_training = (
    baseline_cnn.count_params()
)

if baseline_parameter_count_before_80_training != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count is not 3,809."
    )

# ------------------------------------------------------------
# 3. Obtain optimizer configuration from the baseline
# ------------------------------------------------------------

if not hasattr(
    baseline_cnn,
    "optimizer"
):
    raise RuntimeError(
        "Frozen baseline optimizer is unavailable."
    )

if baseline_cnn.optimizer is None:
    raise RuntimeError(
        "Frozen baseline optimizer is None."
    )

baseline_optimizer_name_80 = (
    baseline_cnn.optimizer.__class__.__name__
)

baseline_optimizer_config_80 = (
    baseline_cnn.optimizer.get_config()
)

optimizer_learning_rate_80 = (
    baseline_optimizer_config_80.get(
        "learning_rate"
    )
)

if optimizer_learning_rate_80 is None:
    raise RuntimeError(
        "Could not determine the frozen baseline "
        "learning rate."
    )

# ------------------------------------------------------------
# 4. Create a fresh optimizer
# ------------------------------------------------------------

if baseline_optimizer_name_80 == "Adam":

    pruning_optimizer_80 = (
        pruning_keras.optimizers.Adam(
            learning_rate=optimizer_learning_rate_80
        )
    )

elif baseline_optimizer_name_80 == "RMSprop":

    pruning_optimizer_80 = (
        pruning_keras.optimizers.RMSprop(
            learning_rate=optimizer_learning_rate_80
        )
    )

elif baseline_optimizer_name_80 == "SGD":

    pruning_optimizer_80 = (
        pruning_keras.optimizers.SGD(
            learning_rate=optimizer_learning_rate_80
        )
    )

else:

    raise RuntimeError(
        "Unsupported frozen baseline optimizer: "
        f"{baseline_optimizer_name_80}"
    )

# ------------------------------------------------------------
# 5. Verify optimizer learning rate
# ------------------------------------------------------------

fresh_learning_rate_80 = float(
    pruning_optimizer_80.learning_rate.numpy()
)

if not np.isclose(
    fresh_learning_rate_80,
    float(optimizer_learning_rate_80)
):
    raise RuntimeError(
        "Fresh optimizer learning rate does not "
        "match the frozen baseline learning rate."
    )

# ------------------------------------------------------------
# 6. Compile the 80% pruning candidate
# ------------------------------------------------------------

pruned_cnn_80.compile(
    optimizer=pruning_optimizer_80,
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)

# ------------------------------------------------------------
# 7. Create a fresh real-data training generator
# ------------------------------------------------------------

pruning_train_generator_80 = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

# ------------------------------------------------------------
# 8. Verify one real training batch
# ------------------------------------------------------------

X_train_check_80, y_train_check_80, weights_check_80 = (
    next(
        pruning_train_generator_80
    )
)

if X_train_check_80.ndim != 3:
    raise RuntimeError(
        "Training ECG batch must be 3-dimensional."
    )

if X_train_check_80.shape[1:] != (
    2500,
    1
):
    raise RuntimeError(
        "Unexpected ECG batch shape: "
        f"{X_train_check_80.shape}"
    )

if y_train_check_80.ndim != 1:
    raise RuntimeError(
        "Training labels must be one-dimensional."
    )

if len(X_train_check_80) != len(
    y_train_check_80
):
    raise RuntimeError(
        "Training samples and labels have "
        "different lengths."
    )

if len(X_train_check_80) != len(
    weights_check_80
):
    raise RuntimeError(
        "Training samples and sample weights have "
        "different lengths."
    )

if not np.isfinite(
    X_train_check_80
).all():
    raise ValueError(
        "Non-finite values detected in training ECG."
    )

if not np.isfinite(
    y_train_check_80
).all():
    raise ValueError(
        "Non-finite values detected in training labels."
    )

if not np.isfinite(
    weights_check_80
).all():
    raise ValueError(
        "Non-finite values detected in training weights."
    )

# ------------------------------------------------------------
# 9. Recreate generator after verification
# ------------------------------------------------------------

pruning_train_generator_80 = (
    generate_weighted_cnn_batches(
        metadata_df=dl_train,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=True
    )
)

# ------------------------------------------------------------
# 10. Create fresh pruning callback
# ------------------------------------------------------------

pruning_callback_80 = (
    tfmot.sparsity.keras.UpdatePruningStep()
)

# ------------------------------------------------------------
# 11. Verify training configuration
# ------------------------------------------------------------

training_steps_80 = (
    PRUNING_STEPS_PER_EPOCH
)

training_epochs_80 = (
    PRUNING_EPOCHS
)

if training_steps_80 <= 0:
    raise RuntimeError(
        "Training steps must be positive."
    )

if training_epochs_80 <= 0:
    raise RuntimeError(
        "Training epochs must be positive."
    )

# ------------------------------------------------------------
# 12. Train the 80% candidate
# ------------------------------------------------------------

training_start_time_80 = (
    time.perf_counter()
)

pruning_history_80 = (
    pruned_cnn_80.fit(
        pruning_train_generator_80,
        steps_per_epoch=training_steps_80,
        epochs=training_epochs_80,
        callbacks=[
            pruning_callback_80
        ],
        verbose=1
    )
)

pruning_training_time_80 = (
    time.perf_counter()
    - training_start_time_80
)

# ------------------------------------------------------------
# 13. Verify training history
# ------------------------------------------------------------

if not hasattr(
    pruning_history_80,
    "history"
):
    raise RuntimeError(
        "Training did not return a valid history object."
    )

history_keys_80 = set(
    pruning_history_80.history.keys()
)

if "loss" not in history_keys_80:
    raise RuntimeError(
        "Training history does not contain loss."
    )

if "accuracy" not in history_keys_80:
    raise RuntimeError(
        "Training history does not contain accuracy."
    )

completed_epochs_80 = len(
    pruning_history_80.history["loss"]
)

if completed_epochs_80 != training_epochs_80:
    raise RuntimeError(
        "Unexpected number of completed epochs: "
        f"{completed_epochs_80}; "
        f"expected {training_epochs_80}."
    )

# ------------------------------------------------------------
# 14. Verify frozen baseline integrity
# ------------------------------------------------------------

baseline_weights_after_80_training = (
    baseline_cnn.get_weights()
)

baseline_unchanged_80_training = all(
    np.array_equal(
        before,
        after
    )
    for before, after in zip(
        baseline_weights_before_80_training,
        baseline_weights_after_80_training
    )
)

if not baseline_unchanged_80_training:
    raise RuntimeError(
        "Frozen baseline weights were modified "
        "during 80% pruning training."
    )

if (
    baseline_cnn.count_params()
    != baseline_parameter_count_before_80_training
):
    raise RuntimeError(
        "Frozen baseline parameter count changed "
        "during 80% pruning training."
    )

# ------------------------------------------------------------
# 15. Report training results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("80% PRUNING TRAINING COMPLETED")
print("=" * 80)

print(
    f"\nOptimizer: "
    f"{baseline_optimizer_name_80}"
)

print(
    f"Learning rate: "
    f"{fresh_learning_rate_80:.10g}"
)

print(
    f"Training samples: "
    f"{len(dl_train):,}"
)

print(
    f"Batch size: "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Steps per epoch: "
    f"{training_steps_80:,}"
)

print(
    f"Epochs completed: "
    f"{completed_epochs_80}"
)

print(
    f"Training time: "
    f"{pruning_training_time_80:.4f} seconds"
)

print("\nFinal training metrics")
print("-" * 80)

print(
    f"Loss: "
    f"{pruning_history_80.history['loss'][-1]:.6f}"
)

print(
    f"Accuracy: "
    f"{pruning_history_80.history['accuracy'][-1]:.6f}"
)

print("\nFrozen baseline verification")
print("-" * 80)

print(
    "Baseline weights unchanged:",
    "PASSED"
    if baseline_unchanged_80_training
    else "FAILED"
)

print(
    "Baseline parameter count unchanged:",
    "PASSED"
    if (
        baseline_cnn.count_params()
        == baseline_parameter_count_before_80_training
    )
    else "FAILED"
)

print(
    "\nTest data was NOT used."
)

print("=" * 80)

if not baseline_unchanged_80_training:
    raise RuntimeError(
        "Frozen baseline integrity check failed."
    )

TRAINING 80% PRUNED CNN CANDIDATE
Epoch 1/5
4655/4655 [==============================] - 645s 137ms/step - loss: 0.4174 - accuracy: 0.7936
Epoch 2/5
4655/4655 [==============================] - 643s 138ms/step - loss: 0.4128 - accuracy: 0.7877
Epoch 3/5
4655/4655 [==============================] - 638s 137ms/step - loss: 0.4284 - accuracy: 0.7797
Epoch 4/5
4655/4655 [==============================] - 639s 137ms/step - loss: 0.4372 - accuracy: 0.7713
Epoch 5/5
4655/4655 [==============================] - 636s 137ms/step - loss: 0.4246 - accuracy: 0.7785

80% PRUNING TRAINING COMPLETED

Optimizer: Adam
Learning rate: 0.001000000047
Training samples: 297,867
Batch size: 64
Steps per epoch: 4,655
Epochs completed: 5
Training time: 3209.0960 seconds

Final training metrics
--------------------------------------------------------------------------------
Loss: 0.424634
Accuracy: 0.778451

Frozen baseline verification
----------------------------------------------------------------------------

In [ ]:
# Cell 27

# Evaluate the trained 80% pruned CNN on the frozen validation split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 80)
print("80% PRUNED CNN — VALIDATION EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_80",
    "pruning_candidate_ready_80",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE",
    "PRUNING_TARGET_SPARSITY_80"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_80:
    raise RuntimeError(
        "The 80% pruning candidate did not pass "
        "its construction checks."
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

# ------------------------------------------------------------
# 2. Verify validation metadata
# ------------------------------------------------------------

required_validation_columns = [
    "WakeSleepLabel",
    "ParticipantID"
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in dl_validation.columns
]

if missing_validation_columns:
    raise RuntimeError(
        "Required validation columns are missing: "
        + ", ".join(missing_validation_columns)
    )

validation_samples_80 = len(
    dl_validation
)

validation_participants_80 = (
    dl_validation["ParticipantID"].nunique()
)

if validation_samples_80 <= 0:
    raise RuntimeError(
        "Validation sample count must be positive."
    )

if validation_participants_80 <= 0:
    raise RuntimeError(
        "Validation participant count must be positive."
    )

# ------------------------------------------------------------
# 3. Create a fresh validation generator
# ------------------------------------------------------------

pruned_validation_generator_80 = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 4. Determine exact validation steps
# ------------------------------------------------------------

validation_steps_80 = int(
    np.ceil(
        validation_samples_80
        / PRUNING_BATCH_SIZE
    )
)

if validation_steps_80 <= 0:
    raise RuntimeError(
        "Validation step count must be positive."
    )

# ------------------------------------------------------------
# 5. Generate validation predictions
# ------------------------------------------------------------

evaluation_start_time_80 = (
    time.perf_counter()
)

validation_probabilities_80 = (
    pruned_cnn_80.predict(
        pruned_validation_generator_80,
        steps=validation_steps_80,
        verbose=0
    )
)

evaluation_time_80 = (
    time.perf_counter()
    - evaluation_start_time_80
)

# ------------------------------------------------------------
# 6. Convert predictions to one-dimensional array
# ------------------------------------------------------------

validation_probabilities_80 = np.asarray(
    validation_probabilities_80,
    dtype=np.float64
).reshape(-1)

validation_probabilities_80 = (
    validation_probabilities_80[
        :validation_samples_80
    ]
)

if len(
    validation_probabilities_80
) != validation_samples_80:
    raise RuntimeError(
        "Prediction count does not match the "
        "validation sample count."
    )

# ------------------------------------------------------------
# 7. Validate prediction values
# ------------------------------------------------------------

if not np.isfinite(
    validation_probabilities_80
).all():
    raise ValueError(
        "Non-finite validation predictions detected."
    )

if (
    (validation_probabilities_80 < 0.0)
    |
    (validation_probabilities_80 > 1.0)
).any():
    raise ValueError(
        "Prediction probabilities outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 8. Obtain validation labels
# ------------------------------------------------------------

validation_labels_80 = (
    dl_validation[
        "WakeSleepLabel"
    ].to_numpy(
        dtype=np.int32
    )
)

if len(
    validation_labels_80
) != validation_samples_80:
    raise RuntimeError(
        "Validation label count does not match "
        "validation sample count."
    )

unique_validation_labels_80 = np.unique(
    validation_labels_80
)

if not set(
    unique_validation_labels_80
).issubset({0, 1}):
    raise ValueError(
        "Validation labels are not binary. "
        f"Observed labels: "
        f"{unique_validation_labels_80}"
    )

if len(
    unique_validation_labels_80
) < 2:
    raise ValueError(
        "ROC-AUC cannot be calculated because "
        "the validation set contains only one class."
    )

# ------------------------------------------------------------
# 9. Apply the predefined decision threshold
# ------------------------------------------------------------

VALIDATION_THRESHOLD_80 = 0.50

validation_predictions_80 = (
    validation_probabilities_80
    >= VALIDATION_THRESHOLD_80
).astype(np.int32)

# ------------------------------------------------------------
# 10. Calculate validation metrics
# ------------------------------------------------------------

pruned_validation_accuracy_80 = (
    accuracy_score(
        validation_labels_80,
        validation_predictions_80
    )
)

pruned_validation_precision_80 = (
    precision_score(
        validation_labels_80,
        validation_predictions_80,
        zero_division=0
    )
)

pruned_validation_recall_80 = (
    recall_score(
        validation_labels_80,
        validation_predictions_80,
        zero_division=0
    )
)

pruned_validation_f1_80 = (
    f1_score(
        validation_labels_80,
        validation_predictions_80,
        zero_division=0
    )
)

pruned_validation_roc_auc_80 = (
    roc_auc_score(
        validation_labels_80,
        validation_probabilities_80
    )
)

# ------------------------------------------------------------
# 11. Store measured results
# ------------------------------------------------------------

pruned_validation_results_80 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Validation samples",
        "Validation participants",
        "Evaluation time (seconds)",
        "Decision threshold",
        "Target sparsity"
    ],
    "Measured value": [
        pruned_validation_accuracy_80,
        pruned_validation_precision_80,
        pruned_validation_recall_80,
        pruned_validation_f1_80,
        pruned_validation_roc_auc_80,
        validation_samples_80,
        validation_participants_80,
        evaluation_time_80,
        VALIDATION_THRESHOLD_80,
        PRUNING_TARGET_SPARSITY_80
    ]
})

# ------------------------------------------------------------
# 12. Display results
# ------------------------------------------------------------

print("\nValidation results")
print("-" * 80)

display(
    pruned_validation_results_80
)

print("\n" + "=" * 80)
print("80% PRUNED CNN — VALIDATION EVALUATION COMPLETED")
print("=" * 80)

print(
    f"\nValidation samples: "
    f"{validation_samples_80:,}"
)

print(
    f"Validation participants: "
    f"{validation_participants_80}"
)

print(
    f"Prediction/evaluation time: "
    f"{evaluation_time_80:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

80% PRUNED CNN — VALIDATION EVALUATION

Validation results
--------------------------------------------------------------------------------


,Metric,Measured value
0,Accuracy,0.744811
1,Precision,0.884007
2,Recall,0.794392
3,F1,0.836807
4,ROC-AUC,0.780374
5,Validation samples,57718.000000
6,Validation participants,4.000000
7,Evaluation time (seconds),82.414184
8,Decision threshold,0.500000
9,Target sparsity,0.800000



80% PRUNED CNN — VALIDATION EVALUATION COMPLETED

Validation samples: 57,718
Validation participants: 4
Prediction/evaluation time: 82.4142 seconds

Test data was NOT accessed.


In [ ]:
# Cell 28

# Analyse sparsity and efficiency of the trained 80% pruned CNN

print("=" * 80)
print("80% PRUNED CNN — SPARSITY AND EFFICIENCY ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "pruned_cnn_80",
    "pruning_candidate_ready_80",
    "pruned_validation_results_80",
    "evaluation_time_80",
    "PRUNING_TARGET_SPARSITY_80",
    "tfmot"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if not pruning_candidate_ready_80:
    raise RuntimeError(
        "The 80% pruning candidate did not pass "
        "its construction checks."
    )

# ------------------------------------------------------------
# 2. Strip pruning wrappers from a copy
# ------------------------------------------------------------

stripped_cnn_80 = (
    tfmot.sparsity.keras.strip_pruning(
        pruned_cnn_80
    )
)

# ------------------------------------------------------------
# 3. Collect effective weights
# ------------------------------------------------------------

effective_weight_arrays_80 = [
    np.asarray(weight)
    for weight in stripped_cnn_80.get_weights()
]

if not effective_weight_arrays_80:
    raise RuntimeError(
        "No weights were found in the stripped "
        "80% pruned model."
    )

# ------------------------------------------------------------
# 4. Calculate parameter statistics
# ------------------------------------------------------------

dense_parameter_count_80 = int(
    sum(
        weight.size
        for weight in effective_weight_arrays_80
    )
)

if dense_parameter_count_80 != 3809:
    raise RuntimeError(
        "Unexpected parameter count after stripping "
        f"pruning wrappers: {dense_parameter_count_80:,}"
    )

zero_weight_count_80 = int(
    sum(
        np.count_nonzero(weight == 0)
        for weight in effective_weight_arrays_80
    )
)

nonzero_weight_count_80 = (
    dense_parameter_count_80
    - zero_weight_count_80
)

# ------------------------------------------------------------
# 5. Calculate actual sparsity
# ------------------------------------------------------------

actual_sparsity_80 = (
    zero_weight_count_80
    / dense_parameter_count_80
)

if not (
    0.0 <= actual_sparsity_80 <= 1.0
):
    raise RuntimeError(
        "Calculated sparsity is outside [0, 1]."
    )

if (
    zero_weight_count_80
    + nonzero_weight_count_80
    != dense_parameter_count_80
):
    raise RuntimeError(
        "Zero and non-zero parameter counts do not "
        "sum to the dense parameter count."
    )

# ------------------------------------------------------------
# 6. Calculate dense Float32 storage
# ------------------------------------------------------------

FLOAT32_BYTES_PER_PARAMETER = 4

dense_float32_storage_bytes_80 = (
    dense_parameter_count_80
    * FLOAT32_BYTES_PER_PARAMETER
)

dense_float32_storage_kb_80 = (
    dense_float32_storage_bytes_80
    / 1024
)

dense_float32_storage_mb_80 = (
    dense_float32_storage_kb_80
    / 1024
)

nonzero_weight_fraction_80 = (
    nonzero_weight_count_80
    / dense_parameter_count_80
)

# ------------------------------------------------------------
# 7. Extract validation metrics
# ------------------------------------------------------------

validation_metric_map_80 = dict(
    zip(
        pruned_validation_results_80["Metric"],
        pruned_validation_results_80["Measured value"]
    )
)

required_metrics_80 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

missing_metrics_80 = [
    metric
    for metric in required_metrics_80
    if metric not in validation_metric_map_80
]

if missing_metrics_80:
    raise RuntimeError(
        "Validation metrics are missing: "
        + ", ".join(missing_metrics_80)
    )

# ------------------------------------------------------------
# 8. Create efficiency summary
# ------------------------------------------------------------

pruning_efficiency_80 = pd.DataFrame({
    "Metric": [
        "Dense parameter count",
        "Zero weight values",
        "Non-zero weight values",
        "Actual sparsity",
        "Target sparsity",
        "Non-zero weight fraction",
        "Dense Float32 storage (KB)",
        "Dense Float32 storage (MB)",
        "Validation accuracy",
        "Validation precision",
        "Validation recall",
        "Validation F1",
        "Validation ROC-AUC",
        "Validation evaluation time (seconds)"
    ],
    "80% pruning": [
        dense_parameter_count_80,
        zero_weight_count_80,
        nonzero_weight_count_80,
        actual_sparsity_80,
        PRUNING_TARGET_SPARSITY_80,
        nonzero_weight_fraction_80,
        dense_float32_storage_kb_80,
        dense_float32_storage_mb_80,
        validation_metric_map_80["Accuracy"],
        validation_metric_map_80["Precision"],
        validation_metric_map_80["Recall"],
        validation_metric_map_80["F1"],
        validation_metric_map_80["ROC-AUC"],
        evaluation_time_80
    ]
})

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

display(
    pruning_efficiency_80
)

print("\n" + "=" * 80)
print("80% PRUNING ANALYSIS COMPLETED")
print("=" * 80)

print(
    f"\nDense parameters: "
    f"{dense_parameter_count_80:,}"
)

print(
    f"Zero weights: "
    f"{zero_weight_count_80:,}"
)

print(
    f"Non-zero weights: "
    f"{nonzero_weight_count_80:,}"
)

print(
    f"Actual sparsity: "
    f"{actual_sparsity_80:.4%}"
)

print(
    f"Target sparsity: "
    f"{PRUNING_TARGET_SPARSITY_80:.2%}"
)

print(
    f"Dense Float32 storage: "
    f"{dense_float32_storage_kb_80:.6f} KB"
)

print(
    f"Dense Float32 storage: "
    f"{dense_float32_storage_mb_80:.8f} MB"
)

print(
    f"Validation F1: "
    f"{validation_metric_map_80['F1']:.6f}"
)

print(
    f"Validation ROC-AUC: "
    f"{validation_metric_map_80['ROC-AUC']:.6f}"
)

print(
    f"Validation evaluation time: "
    f"{evaluation_time_80:.4f} seconds"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

80% PRUNED CNN — SPARSITY AND EFFICIENCY ANALYSIS


,Metric,80% pruning
0,Dense parameter count,3809.000000
1,Zero weight values,2984.000000
2,Non-zero weight values,825.000000
3,Actual sparsity,0.783408
4,Target sparsity,0.800000
5,Non-zero weight fraction,0.216592
6,Dense Float32 storage (KB),14.878906
7,Dense Float32 storage (MB),0.014530
8,Validation accuracy,0.744811
9,Validation precision,0.884007



80% PRUNING ANALYSIS COMPLETED

Dense parameters: 3,809
Zero weights: 2,984
Non-zero weights: 825
Actual sparsity: 78.3408%
Target sparsity: 80.00%
Dense Float32 storage: 14.878906 KB
Dense Float32 storage: 0.01453018 MB
Validation F1: 0.836807
Validation ROC-AUC: 0.780374
Validation evaluation time: 82.4142 seconds

Test data was NOT accessed.


In [ ]:
# Cell 29

# Compare the frozen baseline with all pruning levels
# using only measured validation and efficiency results.

print("=" * 100)
print("FROZEN BASELINE vs PRUNING — COMPLETE VALIDATION COMPARISON")
print("=" * 100)



required_names = [
    "pruned_validation_results_20",
    "pruned_validation_results_40",
    "pruned_validation_results_60",
    "pruned_validation_results_80",
    "pruning_efficiency_20",
    "pruning_efficiency_40",
    "pruning_efficiency_60",
    "pruning_efficiency_80"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required measured results are missing: "
        + ", ".join(missing_names)
    )

# ------------------------------------------------------------
# 2. Helper: safely convert measured numeric values
# ------------------------------------------------------------

def measured_float(value, field_name):
    """
    Convert a measured value to float.

    Percentage strings such as '58.7293%' are converted
    to their decimal representation 0.587293.

    Numeric values are returned unchanged.
    """

    if isinstance(value, str):

        cleaned_value = value.strip()

        if cleaned_value.endswith("%"):

            numeric_part = (
                cleaned_value[:-1].strip()
            )

            try:
                return float(numeric_part) / 100.0

            except ValueError as error:
                raise ValueError(
                    f"Could not convert {field_name} "
                    f"value '{value}' to a percentage."
                ) from error

        try:
            return float(cleaned_value)

        except ValueError as error:
            raise ValueError(
                f"Could not convert {field_name} "
                f"value '{value}' to a numeric value."
            ) from error

    try:
        return float(value)

    except (TypeError, ValueError) as error:
        raise ValueError(
            f"Could not convert {field_name} "
            f"value '{value}' to a numeric value."
        ) from error


# ------------------------------------------------------------
# 3. Helper: extract validation metrics
# ------------------------------------------------------------

def extract_validation_metrics(
    results_df,
    model_name
):

    required_metrics = [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]

    required_columns = [
        "Metric",
        "Measured value"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in results_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"{model_name} is missing columns: "
            + ", ".join(missing_columns)
        )

    metric_map = dict(
        zip(
            results_df["Metric"],
            results_df["Measured value"]
        )
    )

    missing_metrics = [
        metric
        for metric in required_metrics
        if metric not in metric_map
    ]

    if missing_metrics:
        raise RuntimeError(
            f"{model_name} is missing validation metrics: "
            + ", ".join(missing_metrics)
        )

    extracted_metrics = {
        "Model": model_name,
        "Accuracy": measured_float(
            metric_map["Accuracy"],
            f"{model_name} Accuracy"
        ),
        "Precision": measured_float(
            metric_map["Precision"],
            f"{model_name} Precision"
        ),
        "Recall": measured_float(
            metric_map["Recall"],
            f"{model_name} Recall"
        ),
        "F1": measured_float(
            metric_map["F1"],
            f"{model_name} F1"
        ),
        "ROC-AUC": measured_float(
            metric_map["ROC-AUC"],
            f"{model_name} ROC-AUC"
        )
    }

    return extracted_metrics


# ------------------------------------------------------------
# 4. Extract frozen baseline validation metrics
# ------------------------------------------------------------

if "baseline_validation_results" in globals():

    baseline_metrics_29 = (
        extract_validation_metrics(
            baseline_validation_results,
            "Frozen baseline"
        )
    )

else:

    # These values were independently measured in the
    # completed frozen-baseline validation evaluation.

    baseline_metrics_29 = {
        "Model": "Frozen baseline",
        "Accuracy": 0.697027,
        "Precision": 0.905643,
        "Recall": 0.705661,
        "F1": 0.793242,
        "ROC-AUC": 0.746448
    }


# ------------------------------------------------------------
# 5. Extract all pruning validation metrics
# ------------------------------------------------------------

metrics_20_29 = extract_validation_metrics(
    pruned_validation_results_20,
    "20% pruning"
)

metrics_40_29 = extract_validation_metrics(
    pruned_validation_results_40,
    "40% pruning"
)

metrics_60_29 = extract_validation_metrics(
    pruned_validation_results_60,
    "60% pruning"
)

metrics_80_29 = extract_validation_metrics(
    pruned_validation_results_80,
    "80% pruning"
)


# ------------------------------------------------------------
# 6. Build validation-performance table
# ------------------------------------------------------------

pruning_validation_comparison = pd.DataFrame([
    baseline_metrics_29,
    metrics_20_29,
    metrics_40_29,
    metrics_60_29,
    metrics_80_29
])


# ------------------------------------------------------------
# 7. Calculate changes relative to frozen baseline
# ------------------------------------------------------------

baseline_row_29 = (
    pruning_validation_comparison.iloc[0]
)

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]:

    pruning_validation_comparison[
        f"{metric} change vs baseline"
    ] = (
        pruning_validation_comparison[metric]
        - float(baseline_row_29[metric])
    )


# ------------------------------------------------------------
# 8. Validate predictive-performance values
# ------------------------------------------------------------

metric_columns_29 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

metric_values_29 = (
    pruning_validation_comparison[
        metric_columns_29
    ].to_numpy(
        dtype=np.float64
    )
)

if not np.isfinite(
    metric_values_29
).all():

    raise RuntimeError(
        "Non-finite validation metric detected."
    )

if (
    (metric_values_29 < 0.0)
    |
    (metric_values_29 > 1.0)
).any():

    raise RuntimeError(
        "Validation metric outside the expected "
        "[0, 1] range detected."
    )


# ------------------------------------------------------------
# 9. Helper: extract efficiency measurements
# ------------------------------------------------------------

def extract_efficiency_measurements(
    efficiency_df,
    column_name,
    model_name
):

    required_metrics = [
        "Dense parameter count",
        "Zero weight values",
        "Non-zero weight values",
        "Actual sparsity",
        "Target sparsity"
    ]

    required_columns = [
        "Metric",
        column_name
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in efficiency_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"{model_name} efficiency results are missing: "
            + ", ".join(missing_columns)
        )

    metric_map = dict(
        zip(
            efficiency_df["Metric"],
            efficiency_df[column_name]
        )
    )

    missing_metrics = [
        metric
        for metric in required_metrics
        if metric not in metric_map
    ]

    if missing_metrics:
        raise RuntimeError(
            f"{model_name} efficiency results are missing: "
            + ", ".join(missing_metrics)
        )

    return {
        "Model": model_name,
        "Target sparsity": measured_float(
            metric_map["Target sparsity"],
            f"{model_name} target sparsity"
        ),
        "Actual sparsity": measured_float(
            metric_map["Actual sparsity"],
            f"{model_name} actual sparsity"
        ),
        "Zero weights": int(
            round(
                measured_float(
                    metric_map["Zero weight values"],
                    f"{model_name} zero weights"
                )
            )
        ),
        "Non-zero weights": int(
            round(
                measured_float(
                    metric_map["Non-zero weight values"],
                    f"{model_name} non-zero weights"
                )
            )
        ),
        "Dense parameters": int(
            round(
                measured_float(
                    metric_map["Dense parameter count"],
                    f"{model_name} dense parameters"
                )
            )
        )
    }


# ------------------------------------------------------------
# 10. Extract pruning efficiency measurements
# ------------------------------------------------------------

efficiency_20_29 = (
    extract_efficiency_measurements(
        pruning_efficiency_20,
        "20% pruning",
        "20% pruning"
    )
)

efficiency_40_29 = (
    extract_efficiency_measurements(
        pruning_efficiency_40,
        "40% pruning",
        "40% pruning"
    )
)

efficiency_60_29 = (
    extract_efficiency_measurements(
        pruning_efficiency_60,
        "60% pruning",
        "60% pruning"
    )
)

efficiency_80_29 = (
    extract_efficiency_measurements(
        pruning_efficiency_80,
        "80% pruning",
        "80% pruning"
    )
)


# ------------------------------------------------------------
# 11. Create frozen-baseline efficiency record
# ------------------------------------------------------------

baseline_zero_weights_29 = int(
    sum(
        np.count_nonzero(
            weight == 0
        )
        for weight in baseline_cnn.get_weights()
    )
)

baseline_nonzero_weights_29 = int(
    baseline_cnn.count_params()
    - baseline_zero_weights_29
)

baseline_dense_parameters_29 = int(
    baseline_cnn.count_params()
)

baseline_actual_sparsity_29 = (
    baseline_zero_weights_29
    / baseline_dense_parameters_29
)

baseline_efficiency_record_29 = {
    "Model": "Frozen baseline",
    "Target sparsity": 0.0,
    "Actual sparsity": baseline_actual_sparsity_29,
    "Zero weights": baseline_zero_weights_29,
    "Non-zero weights": baseline_nonzero_weights_29,
    "Dense parameters": baseline_dense_parameters_29
}


# ------------------------------------------------------------
# 12. Build complete efficiency comparison
# ------------------------------------------------------------

pruning_efficiency_comparison = pd.DataFrame([
    baseline_efficiency_record_29,
    efficiency_20_29,
    efficiency_40_29,
    efficiency_60_29,
    efficiency_80_29
])


# ------------------------------------------------------------
# 13. Verify parameter consistency
# ------------------------------------------------------------

expected_parameter_count_29 = (
    baseline_dense_parameters_29
)

observed_parameter_counts_29 = (
    pruning_efficiency_comparison[
        "Dense parameters"
    ].to_numpy(
        dtype=np.int64
    )
)

if not np.all(
    observed_parameter_counts_29
    == expected_parameter_count_29
):

    raise RuntimeError(
        "Parameter count mismatch detected across "
        "the baseline and pruning candidates."
    )


# ------------------------------------------------------------
# 14. Verify sparsity values
# ------------------------------------------------------------

sparsity_values_29 = (
    pruning_efficiency_comparison[
        [
            "Target sparsity",
            "Actual sparsity"
        ]
    ].to_numpy(
        dtype=np.float64
    )
)

if not np.isfinite(
    sparsity_values_29
).all():

    raise RuntimeError(
        "Non-finite sparsity value detected."
    )

if (
    (sparsity_values_29 < 0.0)
    |
    (sparsity_values_29 > 1.0)
).any():

    raise RuntimeError(
        "Sparsity value outside the expected "
        "[0, 1] range detected."
    )


# ------------------------------------------------------------
# 15. Add validation performance to efficiency table
# ------------------------------------------------------------

validation_lookup_29 = {
    "Frozen baseline": baseline_metrics_29,
    "20% pruning": metrics_20_29,
    "40% pruning": metrics_40_29,
    "60% pruning": metrics_60_29,
    "80% pruning": metrics_80_29
}

pruning_efficiency_comparison[
    "Validation F1"
] = (
    pruning_efficiency_comparison["Model"]
    .map(
        lambda model:
        validation_lookup_29[model]["F1"]
    )
)

pruning_efficiency_comparison[
    "Validation ROC-AUC"
] = (
    pruning_efficiency_comparison["Model"]
    .map(
        lambda model:
        validation_lookup_29[model]["ROC-AUC"]
    )
)


# ------------------------------------------------------------
# 16. Display predictive-performance comparison
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VALIDATION PERFORMANCE")
print("=" * 100)

display(
    pruning_validation_comparison.round(6)
)


# ------------------------------------------------------------
# 17. Display sparsity and efficiency comparison
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SPARSITY AND EFFICIENCY")
print("=" * 100)

display(
    pruning_efficiency_comparison.round(6)
)


# ------------------------------------------------------------
# 18. Identify highest observed validation F1
# ------------------------------------------------------------

best_f1_index_29 = (
    pruning_validation_comparison[
        "F1"
    ].idxmax()
)

best_f1_model_29 = (
    pruning_validation_comparison.loc[
        best_f1_index_29,
        "Model"
    ]
)

best_f1_value_29 = (
    pruning_validation_comparison.loc[
        best_f1_index_29,
        "F1"
    ]
)


# ------------------------------------------------------------
# 19. Identify highest observed validation ROC-AUC
# ------------------------------------------------------------

best_auc_index_29 = (
    pruning_validation_comparison[
        "ROC-AUC"
    ].idxmax()
)

best_auc_model_29 = (
    pruning_validation_comparison.loc[
        best_auc_index_29,
        "Model"
    ]
)

best_auc_value_29 = (
    pruning_validation_comparison.loc[
        best_auc_index_29,
        "ROC-AUC"
    ]
)


# ------------------------------------------------------------
# 20. Report observed findings
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("OBSERVED VALIDATION RESULTS")
print("=" * 100)

print(
    f"\nHighest observed validation F1: "
    f"{best_f1_model_29}"
)

print(
    f"F1: "
    f"{best_f1_value_29:.6f}"
)

print(
    f"\nHighest observed validation ROC-AUC: "
    f"{best_auc_model_29}"
)

print(
    f"ROC-AUC: "
    f"{best_auc_value_29:.6f}"
)

print(
    "\nAll pruning levels were evaluated using "
    "the validation data only."
)

print(
    "The held-out test data was NOT accessed."
)

print(
    "\nNo final model-selection decision is made "
    "automatically in this cell."
)

print("=" * 100)

FROZEN BASELINE vs PRUNING — COMPLETE VALIDATION COMPARISON

VALIDATION PERFORMANCE


,Model,Accuracy,Precision,Recall,F1,ROC-AUC,Accuracy change vs baseline,Precision change vs baseline,Recall change vs baseline,F1 change vs baseline,ROC-AUC change vs baseline
0,Frozen baseline,0.697027,0.905643,0.705661,0.793242,0.746448,0.000000,0.000000,0.000000,0.000000,0.000000
1,20% pruning,0.714387,0.868789,0.769422,0.816092,0.745819,0.017360,-0.036854,0.063761,0.022850,-0.000629
2,40% pruning,0.732007,0.878663,0.782696,0.827908,0.764577,0.034980,-0.026979,0.077035,0.034666,0.018129
3,60% pruning,0.749922,0.885501,0.799777,0.840459,0.777945,0.052895,-0.020141,0.094116,0.047217,0.031497
4,80% pruning,0.744811,0.884007,0.794392,0.836807,0.780374,0.047784,-0.021636,0.088731,0.043565,0.033926



SPARSITY AND EFFICIENCY


,Model,Target sparsity,Actual sparsity,Zero weights,Non-zero weights,Dense parameters,Validation F1,Validation ROC-AUC
0,Frozen baseline,0.0,0.000263,1,3808,3809,0.793242,0.746448
1,20% pruning,0.2,0.195852,746,3063,3809,0.816092,0.745819
2,40% pruning,0.4,0.391966,1493,2316,3809,0.827908,0.764577
3,60% pruning,0.6,0.587293,2237,1572,3809,0.840459,0.777945
4,80% pruning,0.8,0.783408,2984,825,3809,0.836807,0.780374



OBSERVED VALIDATION RESULTS

Highest observed validation F1: 60% pruning
F1: 0.840459

Highest observed validation ROC-AUC: 80% pruning
ROC-AUC: 0.780374

All pruning levels were evaluated using the validation data only.
The held-out test data was NOT accessed.

No final model-selection decision is made automatically in this cell.


## 5. Quantisation Experiment

Post-training quantisation is investigated as the second software-based optimisation technique. Unlike pruning, which introduces sparsity into the model weights, quantisation aims to reduce the numerical representation and storage requirements of the trained model.

The frozen Lightweight CNN is converted to TensorFlow Lite using post-training dynamic-range quantisation. This approach is applied without retraining the model and therefore provides a direct assessment of the effect of quantisation on model size and inference efficiency.

The quantised model is evaluated on the same validation split used throughout Notebook 5. The comparison considers predictive performance alongside measured model size and prediction time.

The objective is to determine whether quantisation can provide a useful reduction in computational and storage requirements while maintaining acceptable ECG classification performance.

The held-out test set remains completely isolated during this optimisation stage and is not accessed until the final model has been selected using validation evidence.

In [ ]:
# Cell 30


print("=" * 80)
print("QUANTISATION EXPERIMENT — FROZEN BASELINE CNN")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names = [
    "baseline_cnn",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE",
    "WINDOW_SAMPLES"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names)
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

if baseline_cnn.count_params() <= 0:
    raise RuntimeError(
        "Frozen baseline CNN contains no parameters."
    )

# ------------------------------------------------------------
# 2. Verify frozen baseline architecture
# ------------------------------------------------------------

baseline_input_shape = (
    tuple(baseline_cnn.input_shape[1:])
)

expected_input_shape = (
    WINDOW_SAMPLES,
    1
)

if baseline_input_shape != expected_input_shape:
    raise RuntimeError(
        "Baseline CNN input shape does not match "
        f"the expected ECG shape. "
        f"Observed: {baseline_input_shape}; "
        f"Expected: {expected_input_shape}"
    )

baseline_parameter_count_30 = int(
    baseline_cnn.count_params()
)

if baseline_parameter_count_30 != 3809:
    raise RuntimeError(
        "Frozen baseline parameter count has changed. "
        f"Observed: {baseline_parameter_count_30:,}; "
        "Expected: 3,809."
    )

# ------------------------------------------------------------
# 3. Create TensorFlow Lite converter
# ------------------------------------------------------------

print("\nCreating TensorFlow Lite converter...")

converter_30 = (
    tf.lite.TFLiteConverter.from_keras_model(
        baseline_cnn
    )
)

# Post-training dynamic-range quantisation.
#
# This quantises eligible model weights to reduced precision
# while retaining floating-point model inputs/outputs.
#
# No test data is required or accessed.
converter_30.optimizations = [
    tf.lite.Optimize.DEFAULT
]

# ------------------------------------------------------------
# 4. Convert the model
# ------------------------------------------------------------

print(
    "Applying post-training dynamic-range quantisation..."
)

quantized_cnn_tflite_30 = (
    converter_30.convert()
)

if not quantized_cnn_tflite_30:
    raise RuntimeError(
        "TensorFlow Lite conversion returned "
        "an empty model."
    )

if not isinstance(
    quantized_cnn_tflite_30,
    (bytes, bytearray)
):
    raise RuntimeError(
        "Unexpected TensorFlow Lite model type: "
        f"{type(quantized_cnn_tflite_30)}"
    )

quantized_model_size_bytes_30 = int(
    len(quantized_cnn_tflite_30)
)

if quantized_model_size_bytes_30 <= 0:
    raise RuntimeError(
        "Quantised TensorFlow Lite model has "
        "zero bytes."
    )

# ------------------------------------------------------------
# 5. Create an interpreter for verification
# ------------------------------------------------------------

print(
    "Creating TensorFlow Lite interpreter..."
)

quantized_interpreter_30 = (
    tf.lite.Interpreter(
        model_content=quantized_cnn_tflite_30
    )
)

quantized_interpreter_30.allocate_tensors()

input_details_30 = (
    quantized_interpreter_30.get_input_details()
)

output_details_30 = (
    quantized_interpreter_30.get_output_details()
)

if len(input_details_30) != 1:
    raise RuntimeError(
        "Expected exactly one TensorFlow Lite "
        "input tensor."
    )

if len(output_details_30) != 1:
    raise RuntimeError(
        "Expected exactly one TensorFlow Lite "
        "output tensor."
    )

# ------------------------------------------------------------
# 6. Verify TensorFlow Lite input shape
# ------------------------------------------------------------

tflite_input_shape_30 = tuple(
    input_details_30[0]["shape"]
)

expected_tflite_input_shape_30 = (
    1,
    WINDOW_SAMPLES,
    1
)

if tflite_input_shape_30 != (
    expected_tflite_input_shape_30
):
    raise RuntimeError(
        "Quantised model input shape is unexpected. "
        f"Observed: {tflite_input_shape_30}; "
        f"Expected: {expected_tflite_input_shape_30}"
    )

# ------------------------------------------------------------
# 7. Verify input and output data types
# ------------------------------------------------------------

tflite_input_dtype_30 = (
    input_details_30[0]["dtype"]
)

tflite_output_dtype_30 = (
    output_details_30[0]["dtype"]
)

if tflite_input_dtype_30 != np.float32:
    raise RuntimeError(
        "This dynamic-range quantisation experiment "
        "expected a float32 input interface. "
        f"Observed: {tflite_input_dtype_30}"
    )

if tflite_output_dtype_30 != np.float32:
    raise RuntimeError(
        "This dynamic-range quantisation experiment "
        "expected a float32 output interface. "
        f"Observed: {tflite_output_dtype_30}"
    )

# ------------------------------------------------------------
# 8. Measure original frozen Keras model storage
# ------------------------------------------------------------

baseline_weight_bytes_30 = int(
    sum(
        weight.nbytes
        for weight in baseline_cnn.get_weights()
    )
)

baseline_weight_size_kb_30 = (
    baseline_weight_bytes_30 / 1024
)

baseline_weight_size_mb_30 = (
    baseline_weight_size_kb_30 / 1024
)

quantized_model_size_kb_30 = (
    quantized_model_size_bytes_30 / 1024
)

quantized_model_size_mb_30 = (
    quantized_model_size_kb_30 / 1024
)

# ------------------------------------------------------------
# 9. Calculate measured size ratio
# ------------------------------------------------------------

quantized_to_baseline_size_ratio_30 = (
    quantized_model_size_bytes_30
    / baseline_weight_bytes_30
)

if not np.isfinite(
    quantized_to_baseline_size_ratio_30
):
    raise RuntimeError(
        "Non-finite quantised-to-baseline "
        "size ratio detected."
    )

# ------------------------------------------------------------
# 10. Store quantisation measurements
# ------------------------------------------------------------

quantization_baseline_results_30 = pd.DataFrame({
    "Metric": [
        "Baseline trainable parameters",
        "Baseline Float32 weight storage (bytes)",
        "Baseline Float32 weight storage (KB)",
        "Baseline Float32 weight storage (MB)",
        "Quantised TFLite model size (bytes)",
        "Quantised TFLite model size (KB)",
        "Quantised TFLite model size (MB)",
        "Quantised / baseline size ratio",
        "TFLite input dtype",
        "TFLite output dtype"
    ],
    "Measured value": [
        baseline_parameter_count_30,
        baseline_weight_bytes_30,
        baseline_weight_size_kb_30,
        baseline_weight_size_mb_30,
        quantized_model_size_bytes_30,
        quantized_model_size_kb_30,
        quantized_model_size_mb_30,
        quantized_to_baseline_size_ratio_30,
        str(tflite_input_dtype_30),
        str(tflite_output_dtype_30)
    ]
})

# ------------------------------------------------------------
# 11. Display verified quantisation configuration
# ------------------------------------------------------------

display(
    quantization_baseline_results_30
)

print("\n" + "=" * 80)
print("QUANTISATION CONVERSION COMPLETED")
print("=" * 80)

print(
    f"\nBaseline trainable parameters: "
    f"{baseline_parameter_count_30:,}"
)

print(
    f"Baseline Float32 weight storage: "
    f"{baseline_weight_size_kb_30:.4f} KB"
)

print(
    f"Quantised TFLite model size: "
    f"{quantized_model_size_kb_30:.4f} KB"
)

print(
    f"Quantised / baseline size ratio: "
    f"{quantized_to_baseline_size_ratio_30:.4f}"
)

print(
    f"TFLite input dtype: "
    f"{tflite_input_dtype_30}"
)

print(
    f"TFLite output dtype: "
    f"{tflite_output_dtype_30}"
)

print(
    "\nQuantisation method: "
    "post-training dynamic-range quantisation."
)

print(
    "No training data was modified."
)

print(
    "Validation data was NOT used for conversion."
)

print(
    "Test data was NOT accessed."
)

print("=" * 80)

QUANTISATION EXPERIMENT — FROZEN BASELINE CNN

Creating TensorFlow Lite converter...
Applying post-training dynamic-range quantisation...
Saved artifact at '/tmp/tmpqqsqisq9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2500, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134583640403472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134583640404432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134583640406544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134583640405200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134583640405968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134583640407312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134583640407120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134586292396752: TensorSpec(shape=(), dtype=tf.resource, name=None)
Creating TensorFlow Li

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Metric,Measured value
0,Baseline trainable parameters,3809
1,Baseline Float32 weight storage (bytes),15236
2,Baseline Float32 weight storage (KB),14.878906
3,Baseline Float32 weight storage (MB),0.01453
4,Quantised TFLite model size (bytes),10040
5,Quantised TFLite model size (KB),9.804688
6,Quantised TFLite model size (MB),0.009575
7,Quantised / baseline size ratio,0.658966
8,TFLite input dtype,<class 'numpy.float32'>
9,TFLite output dtype,<class 'numpy.float32'>



QUANTISATION CONVERSION COMPLETED

Baseline trainable parameters: 3,809
Baseline Float32 weight storage: 14.8789 KB
Quantised TFLite model size: 9.8047 KB
Quantised / baseline size ratio: 0.6590
TFLite input dtype: <class 'numpy.float32'>
TFLite output dtype: <class 'numpy.float32'>

Quantisation method: post-training dynamic-range quantisation.
No training data was modified.
Validation data was NOT used for conversion.
Test data was NOT accessed.


In [ ]:
# Cell 31

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 80)
print("QUANTISATION EVALUATION — VALIDATION SET")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_names_31 = [
    "baseline_cnn",
    "dl_validation",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE",
    "quantized_interpreter_30",
    "input_details_30",
    "output_details_30"
]

missing_names_31 = [
    name
    for name in required_names_31
    if name not in globals()
]

if missing_names_31:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names_31)
    )

if dl_validation.empty:
    raise ValueError(
        "Validation dataset is empty."
    )

# ------------------------------------------------------------
# 2. Verify validation metadata
# ------------------------------------------------------------

required_validation_columns_31 = [
    "WakeSleepLabel",
    "ParticipantID"
]

missing_validation_columns_31 = [
    column
    for column in required_validation_columns_31
    if column not in dl_validation.columns
]

if missing_validation_columns_31:
    raise RuntimeError(
        "Required validation columns are missing: "
        + ", ".join(missing_validation_columns_31)
    )

validation_samples_31 = len(
    dl_validation
)

validation_participants_31 = (
    dl_validation["ParticipantID"].nunique()
)

if validation_samples_31 <= 0:
    raise RuntimeError(
        "Validation sample count must be positive."
    )

if validation_participants_31 <= 0:
    raise RuntimeError(
        "Validation participant count must be positive."
    )

# ------------------------------------------------------------
# 3. Obtain validation labels
# ------------------------------------------------------------

validation_labels_31 = (
    dl_validation[
        "WakeSleepLabel"
    ].to_numpy(
        dtype=np.int32
    )
)

if len(validation_labels_31) != validation_samples_31:
    raise RuntimeError(
        "Validation label count does not match "
        "validation sample count."
    )

unique_validation_labels_31 = np.unique(
    validation_labels_31
)

if not set(
    unique_validation_labels_31
).issubset({0, 1}):
    raise ValueError(
        "Validation labels are not binary. "
        f"Observed labels: "
        f"{unique_validation_labels_31}"
    )

if len(unique_validation_labels_31) < 2:
    raise ValueError(
        "Validation set must contain both classes "
        "for ROC-AUC calculation."
    )

# ------------------------------------------------------------
# 4. Create deterministic validation generator
# ------------------------------------------------------------

validation_generator_31 = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 5. Determine exact validation steps
# ------------------------------------------------------------

validation_steps_31 = int(
    np.ceil(
        validation_samples_31
        / PRUNING_BATCH_SIZE
    )
)

if validation_steps_31 <= 0:
    raise RuntimeError(
        "Validation step count must be positive."
    )

print(
    f"\nValidation samples: "
    f"{validation_samples_31:,}"
)

print(
    f"Validation participants: "
    f"{validation_participants_31}"
)

print(
    f"Validation batch size: "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Validation steps: "
    f"{validation_steps_31}"
)

# ------------------------------------------------------------
# 6. Evaluate frozen baseline CNN
# ------------------------------------------------------------

print("\nEvaluating frozen baseline CNN...")

baseline_validation_start_31 = (
    time.perf_counter()
)

baseline_validation_probabilities_31 = (
    baseline_cnn.predict(
        validation_generator_31,
        steps=validation_steps_31,
        verbose=0
    )
)

baseline_validation_time_31 = (
    time.perf_counter()
    - baseline_validation_start_31
)

baseline_validation_probabilities_31 = (
    np.asarray(
        baseline_validation_probabilities_31,
        dtype=np.float64
    )
    .reshape(-1)
)

baseline_validation_probabilities_31 = (
    baseline_validation_probabilities_31[
        :validation_samples_31
    ]
)

if len(
    baseline_validation_probabilities_31
) != validation_samples_31:
    raise RuntimeError(
        "Baseline prediction count does not match "
        "validation sample count."
    )

if not np.isfinite(
    baseline_validation_probabilities_31
).all():
    raise RuntimeError(
        "Baseline predictions contain non-finite values."
    )

if (
    (baseline_validation_probabilities_31 < 0.0)
    |
    (baseline_validation_probabilities_31 > 1.0)
).any():
    raise RuntimeError(
        "Baseline predictions outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 7. Recreate deterministic validation generator
# ------------------------------------------------------------

quantized_validation_generator_31 = (
    generate_ecg_batches_fast(
        metadata_df=dl_validation,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

# ------------------------------------------------------------
# 8. Configure TFLite interpreter for batch processing
# ------------------------------------------------------------

quantized_input_index_31 = (
    input_details_30[0]["index"]
)

quantized_output_index_31 = (
    output_details_30[0]["index"]
)

quantized_input_shape_31 = (
    input_details_30[0]["shape"]
)

if len(quantized_input_shape_31) != 3:
    raise RuntimeError(
        "Unexpected TFLite input rank. "
        f"Observed shape: "
        f"{quantized_input_shape_31}"
    )

if (
    int(quantized_input_shape_31[1])
    != int(WINDOW_SAMPLES)
):
    raise RuntimeError(
        "TFLite ECG window length does not match "
        "WINDOW_SAMPLES."
    )

# The converted model has a dynamic batch dimension.
# Resize it to the established validation batch size.
tflite_batch_shape_31 = [
    PRUNING_BATCH_SIZE,
    WINDOW_SAMPLES,
    1
]

quantized_interpreter_30.resize_tensor_input(
    quantized_input_index_31,
    tflite_batch_shape_31,
    strict=False
)

quantized_interpreter_30.allocate_tensors()

# Refresh tensor details after allocation.
input_details_31 = (
    quantized_interpreter_30.get_input_details()
)

output_details_31 = (
    quantized_interpreter_30.get_output_details()
)

if len(input_details_31) != 1:
    raise RuntimeError(
        "Expected exactly one TFLite input tensor."
    )

if len(output_details_31) != 1:
    raise RuntimeError(
        "Expected exactly one TFLite output tensor."
    )

quantized_input_index_31 = (
    input_details_31[0]["index"]
)

quantized_output_index_31 = (
    output_details_31[0]["index"]
)

actual_tflite_input_shape_31 = tuple(
    input_details_31[0]["shape"]
)

expected_tflite_input_shape_31 = (
    PRUNING_BATCH_SIZE,
    WINDOW_SAMPLES,
    1
)

if actual_tflite_input_shape_31 != (
    expected_tflite_input_shape_31
):
    raise RuntimeError(
        "TFLite input shape after resizing is incorrect. "
        f"Observed: {actual_tflite_input_shape_31}; "
        f"Expected: {expected_tflite_input_shape_31}"
    )

# ------------------------------------------------------------
# 9. Evaluate quantised TFLite CNN batch-by-batch
# ------------------------------------------------------------

print("Evaluating quantised TFLite CNN...")

quantized_validation_probabilities_31 = []

quantized_validation_start_31 = (
    time.perf_counter()
)

for step_31 in range(
    validation_steps_31
):

    X_batch_31, y_batch_31 = (
        next(
            quantized_validation_generator_31
        )
    )

    X_batch_31 = np.asarray(
        X_batch_31,
        dtype=np.float32
    )

    if X_batch_31.ndim != 3:
        raise RuntimeError(
            "Unexpected validation batch shape: "
            f"{X_batch_31.shape}"
        )

    actual_batch_size_31 = (
        X_batch_31.shape[0]
    )

    if actual_batch_size_31 <= 0:
        raise RuntimeError(
            "Encountered an empty validation batch."
        )

    # Pad only the final batch to the fixed TFLite
    # batch size. Predictions corresponding to the
    # padding are discarded.
    if actual_batch_size_31 < PRUNING_BATCH_SIZE:

        padding_count_31 = (
            PRUNING_BATCH_SIZE
            - actual_batch_size_31
        )

        X_batch_31 = np.concatenate(
            [
                X_batch_31,
                np.zeros(
                    (
                        padding_count_31,
                        WINDOW_SAMPLES,
                        1
                    ),
                    dtype=np.float32
                )
            ],
            axis=0
        )

    if X_batch_31.shape != (
        PRUNING_BATCH_SIZE,
        WINDOW_SAMPLES,
        1
    ):
        raise RuntimeError(
            "Prepared TFLite batch has incorrect shape: "
            f"{X_batch_31.shape}"
        )

    if not np.isfinite(
        X_batch_31
    ).all():
        raise RuntimeError(
            "Non-finite values detected in "
            "validation ECG batch."
        )

    quantized_interpreter_30.set_tensor(
        quantized_input_index_31,
        X_batch_31
    )

    quantized_interpreter_30.invoke()

    batch_output_31 = (
        quantized_interpreter_30.get_tensor(
            quantized_output_index_31
        )
    )

    batch_probabilities_31 = (
        np.asarray(
            batch_output_31,
            dtype=np.float64
        )
        .reshape(-1)
    )

    if len(
        batch_probabilities_31
    ) != PRUNING_BATCH_SIZE:
        raise RuntimeError(
            "Unexpected number of TFLite predictions "
            f"in batch {step_31 + 1}."
        )

    # Retain only real validation samples.
    batch_probabilities_31 = (
        batch_probabilities_31[
            :actual_batch_size_31
        ]
    )

    quantized_validation_probabilities_31.extend(
        batch_probabilities_31.tolist()
    )

quantized_validation_time_31 = (
    time.perf_counter()
    - quantized_validation_start_31
)

quantized_validation_probabilities_31 = (
    np.asarray(
        quantized_validation_probabilities_31,
        dtype=np.float64
    )
)

# ------------------------------------------------------------
# 10. Validate quantised predictions
# ------------------------------------------------------------

if len(
    quantized_validation_probabilities_31
) != validation_samples_31:
    raise RuntimeError(
        "Quantised prediction count does not match "
        "validation sample count. "
        f"Predictions: "
        f"{len(quantized_validation_probabilities_31):,}; "
        f"Expected: {validation_samples_31:,}"
    )

if not np.isfinite(
    quantized_validation_probabilities_31
).all():
    raise RuntimeError(
        "Quantised predictions contain non-finite values."
    )

if (
    (quantized_validation_probabilities_31 < 0.0)
    |
    (quantized_validation_probabilities_31 > 1.0)
).any():
    raise RuntimeError(
        "Quantised predictions outside [0, 1] detected."
    )

# ------------------------------------------------------------
# 11. Convert probabilities to binary predictions
# ------------------------------------------------------------

baseline_validation_predictions_31 = (
    baseline_validation_probabilities_31 >= 0.5
).astype(np.int32)

quantized_validation_predictions_31 = (
    quantized_validation_probabilities_31 >= 0.5
).astype(np.int32)

# ------------------------------------------------------------
# 12. Calculate baseline metrics
# ------------------------------------------------------------

baseline_accuracy_31 = accuracy_score(
    validation_labels_31,
    baseline_validation_predictions_31
)

baseline_precision_31 = precision_score(
    validation_labels_31,
    baseline_validation_predictions_31,
    zero_division=0
)

baseline_recall_31 = recall_score(
    validation_labels_31,
    baseline_validation_predictions_31,
    zero_division=0
)

baseline_f1_31 = f1_score(
    validation_labels_31,
    baseline_validation_predictions_31,
    zero_division=0
)

baseline_roc_auc_31 = roc_auc_score(
    validation_labels_31,
    baseline_validation_probabilities_31
)

# ------------------------------------------------------------
# 13. Calculate quantised metrics
# ------------------------------------------------------------

quantized_accuracy_31 = accuracy_score(
    validation_labels_31,
    quantized_validation_predictions_31
)

quantized_precision_31 = precision_score(
    validation_labels_31,
    quantized_validation_predictions_31,
    zero_division=0
)

quantized_recall_31 = recall_score(
    validation_labels_31,
    quantized_validation_predictions_31,
    zero_division=0
)

quantized_f1_31 = f1_score(
    validation_labels_31,
    quantized_validation_predictions_31,
    zero_division=0
)

quantized_roc_auc_31 = roc_auc_score(
    validation_labels_31,
    quantized_validation_probabilities_31
)

# ------------------------------------------------------------
# 14. Calculate changes relative to baseline
# ------------------------------------------------------------

accuracy_change_31 = (
    quantized_accuracy_31
    - baseline_accuracy_31
)

precision_change_31 = (
    quantized_precision_31
    - baseline_precision_31
)

recall_change_31 = (
    quantized_recall_31
    - baseline_recall_31
)

f1_change_31 = (
    quantized_f1_31
    - baseline_f1_31
)

roc_auc_change_31 = (
    quantized_roc_auc_31
    - baseline_roc_auc_31
)

# ------------------------------------------------------------
# 15. Create validation comparison table
# ------------------------------------------------------------

quantization_validation_results_31 = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC",
            "Prediction time (seconds)"
        ],
        "Frozen baseline": [
            baseline_accuracy_31,
            baseline_precision_31,
            baseline_recall_31,
            baseline_f1_31,
            baseline_roc_auc_31,
            baseline_validation_time_31
        ],
        "Quantised CNN": [
            quantized_accuracy_31,
            quantized_precision_31,
            quantized_recall_31,
            quantized_f1_31,
            quantized_roc_auc_31,
            quantized_validation_time_31
        ],
        "Change": [
            accuracy_change_31,
            precision_change_31,
            recall_change_31,
            f1_change_31,
            roc_auc_change_31,
            (
                quantized_validation_time_31
                - baseline_validation_time_31
            )
        ]
    }
)

display(
    quantization_validation_results_31
)

# ------------------------------------------------------------
# 16. Store results for subsequent analysis
# ------------------------------------------------------------

quantization_validation_summary_31 = {
    "accuracy": quantized_accuracy_31,
    "precision": quantized_precision_31,
    "recall": quantized_recall_31,
    "f1": quantized_f1_31,
    "roc_auc": quantized_roc_auc_31,
    "prediction_time": quantized_validation_time_31,
    "model_size_bytes": quantized_model_size_bytes_30,
    "model_size_kb": quantized_model_size_kb_30,
    "model_size_mb": quantized_model_size_mb_30,
    "size_ratio": quantized_to_baseline_size_ratio_30
}

print("\n" + "=" * 80)
print("QUANTISATION VALIDATION EVALUATION COMPLETED")
print("=" * 80)

print(
    f"\nValidation samples: "
    f"{validation_samples_31:,}"
)

print(
    f"Validation participants: "
    f"{validation_participants_31}"
)

print(
    f"\nBaseline F1: "
    f"{baseline_f1_31:.6f}"
)

print(
    f"Quantised F1: "
    f"{quantized_f1_31:.6f}"
)

print(
    f"F1 change: "
    f"{f1_change_31:+.6f}"
)

print(
    f"\nBaseline ROC-AUC: "
    f"{baseline_roc_auc_31:.6f}"
)

print(
    f"Quantised ROC-AUC: "
    f"{quantized_roc_auc_31:.6f}"
)

print(
    f"ROC-AUC change: "
    f"{roc_auc_change_31:+.6f}"
)

print(
    f"\nQuantised TFLite model size: "
    f"{quantized_model_size_kb_30:.4f} KB"
)

print(
    f"Quantised / baseline size ratio: "
    f"{quantized_to_baseline_size_ratio_30:.6f}"
)

print(
    "\nTest data was NOT accessed."
)

print("=" * 80)

QUANTISATION EVALUATION — VALIDATION SET

Validation samples: 57,718
Validation participants: 4
Validation batch size: 64
Validation steps: 902

Evaluating frozen baseline CNN...
Evaluating quantised TFLite CNN...


,Metric,Frozen baseline,Quantised CNN,Change
0,Accuracy,0.697027,0.692436,-0.004591
1,Precision,0.905643,0.907891,0.002248
2,Recall,0.705661,0.697309,-0.008351
3,F1,0.793242,0.788787,-0.004454
4,ROC-AUC,0.746448,0.744670,-0.001778
5,Prediction time (seconds),51.093595,27.691437,-23.402158



QUANTISATION VALIDATION EVALUATION COMPLETED

Validation samples: 57,718
Validation participants: 4

Baseline F1: 0.793242
Quantised F1: 0.788787
F1 change: -0.004454

Baseline ROC-AUC: 0.746448
Quantised ROC-AUC: 0.744670
ROC-AUC change: -0.001778

Quantised TFLite model size: 9.8047 KB
Quantised / baseline size ratio: 0.658966

Test data was NOT accessed.


### Quantisation Validation Results

The quantised CNN is compared with the frozen Float32 baseline using the validation set.

The experiment evaluates both predictive performance and computational efficiency. In particular, model-size reduction and measured prediction-time reduction are treated as important efficiency indicators for the lightweight deployment objective.

The quantised CNN achieved a measured **34.10% reduction in model size** and a **45.80% reduction in prediction time**, corresponding to an approximately **1.845× measured inference speed-up**.

The changes in predictive performance were comparatively small on the validation set. The F1-score changed from **0.7932** for the frozen baseline to **0.7888** after quantisation, while ROC-AUC changed from **0.7464** to **0.7447**.

These measurements provide the basis for comparing quantisation against the pruning candidates in the subsequent optimisation analysis.

In [ ]:
# Cell 32

# Analyse the measured effect of quantisation relative to
# the frozen baseline CNN using validation-only evidence.

print("=" * 80)
print("QUANTISATION — BASELINE COMPARISON AND EFFICIENCY ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify required measured results
# ------------------------------------------------------------

required_names_32 = [
    "quantization_validation_results_31",
    "quantization_validation_summary_31",
    "baseline_parameter_count_30",
    "baseline_weight_size_kb_30",
    "baseline_weight_size_mb_30",
    "quantized_model_size_kb_30",
    "quantized_model_size_mb_30",
    "quantized_model_size_bytes_30",
    "quantized_to_baseline_size_ratio_30"
]

missing_names_32 = [
    name
    for name in required_names_32
    if name not in globals()
]

if missing_names_32:
    raise RuntimeError(
        "Required quantisation results are missing: "
        + ", ".join(missing_names_32)
    )

# ------------------------------------------------------------
# 2. Extract measured validation results
# ------------------------------------------------------------

quantisation_results_map_32 = dict(
    zip(
        quantization_validation_results_31["Metric"],
        quantization_validation_results_31["Quantised CNN"]
    )
)

baseline_results_map_32 = dict(
    zip(
        quantization_validation_results_31["Metric"],
        quantization_validation_results_31["Frozen baseline"]
    )
)

required_metrics_32 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "Prediction time (seconds)"
]

for metric_32 in required_metrics_32:

    if metric_32 not in quantisation_results_map_32:
        raise RuntimeError(
            f"Quantised result is missing metric: "
            f"{metric_32}"
        )

    if metric_32 not in baseline_results_map_32:
        raise RuntimeError(
            f"Baseline result is missing metric: "
            f"{metric_32}"
        )

# ------------------------------------------------------------
# 3. Calculate measured efficiency changes
# ------------------------------------------------------------

baseline_time_32 = float(
    baseline_results_map_32[
        "Prediction time (seconds)"
    ]
)

quantized_time_32 = float(
    quantisation_results_map_32[
        "Prediction time (seconds)"
    ]
)

if baseline_time_32 <= 0:
    raise RuntimeError(
        "Baseline prediction time must be positive."
    )

if quantized_time_32 <= 0:
    raise RuntimeError(
        "Quantised prediction time must be positive."
    )

prediction_time_change_32 = (
    quantized_time_32
    - baseline_time_32
)

prediction_time_reduction_32 = (
    baseline_time_32
    - quantized_time_32
)

prediction_time_reduction_percent_32 = (
    prediction_time_reduction_32
    / baseline_time_32
)

inference_speedup_32 = (
    baseline_time_32
    / quantized_time_32
)

# ------------------------------------------------------------
# 4. Calculate measured model-size reduction
# ------------------------------------------------------------

if baseline_weight_size_kb_30 <= 0:
    raise RuntimeError(
        "Baseline model storage must be positive."
    )

if quantized_model_size_kb_30 <= 0:
    raise RuntimeError(
        "Quantised model size must be positive."
    )

model_size_reduction_kb_32 = (
    baseline_weight_size_kb_30
    - quantized_model_size_kb_30
)

model_size_reduction_percent_32 = (
    model_size_reduction_kb_32
    / baseline_weight_size_kb_30
)

# ------------------------------------------------------------
# 5. Extract predictive-performance changes
# ------------------------------------------------------------

accuracy_change_32 = (
    float(
        quantisation_results_map_32["Accuracy"]
    )
    - float(
        baseline_results_map_32["Accuracy"]
    )
)

precision_change_32 = (
    float(
        quantisation_results_map_32["Precision"]
    )
    - float(
        baseline_results_map_32["Precision"]
    )
)

recall_change_32 = (
    float(
        quantisation_results_map_32["Recall"]
    )
    - float(
        baseline_results_map_32["Recall"]
    )
)

f1_change_32 = (
    float(
        quantisation_results_map_32["F1"]
    )
    - float(
        baseline_results_map_32["F1"]
    )
)

roc_auc_change_32 = (
    float(
        quantisation_results_map_32["ROC-AUC"]
    )
    - float(
        baseline_results_map_32["ROC-AUC"]
    )
)

# ------------------------------------------------------------
# 6. Verify all calculated values are finite
# ------------------------------------------------------------

calculated_values_32 = [
    prediction_time_change_32,
    prediction_time_reduction_32,
    prediction_time_reduction_percent_32,
    inference_speedup_32,
    model_size_reduction_kb_32,
    model_size_reduction_percent_32,
    accuracy_change_32,
    precision_change_32,
    recall_change_32,
    f1_change_32,
    roc_auc_change_32
]

if not np.isfinite(
    np.asarray(calculated_values_32)
).all():
    raise RuntimeError(
        "Non-finite value detected in the "
        "quantisation analysis."
    )

# ------------------------------------------------------------
# 7. Create efficiency summary
# ------------------------------------------------------------

quantisation_efficiency_summary_32 = pd.DataFrame({
    "Metric": [
        "Baseline Float32 storage (KB)",
        "Quantised TFLite size (KB)",
        "Model-size reduction (KB)",
        "Model-size reduction (%)",
        "Baseline validation prediction time (s)",
        "Quantised validation prediction time (s)",
        "Prediction-time reduction (s)",
        "Prediction-time reduction (%)",
        "Inference speed-up (x)",
        "Accuracy change",
        "Precision change",
        "Recall change",
        "F1 change",
        "ROC-AUC change"
    ],
    "Measured value": [
        baseline_weight_size_kb_30,
        quantized_model_size_kb_30,
        model_size_reduction_kb_32,
        model_size_reduction_percent_32,
        baseline_time_32,
        quantized_time_32,
        prediction_time_reduction_32,
        prediction_time_reduction_percent_32,
        inference_speedup_32,
        accuracy_change_32,
        precision_change_32,
        recall_change_32,
        f1_change_32,
        roc_auc_change_32
    ]
})

display(
    quantisation_efficiency_summary_32
)

# ------------------------------------------------------------
# 8. Print measured findings
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("QUANTISATION ANALYSIS COMPLETED")
print("=" * 80)

print(
    f"\nBaseline Float32 storage: "
    f"{baseline_weight_size_kb_30:.4f} KB"
)

print(
    f"Quantised TFLite model size: "
    f"{quantized_model_size_kb_30:.4f} KB"
)

print(
    f"Measured model-size reduction: "
    f"{model_size_reduction_percent_32 * 100:.2f}%"
)

print(
    f"\nBaseline validation prediction time: "
    f"{baseline_time_32:.4f} seconds"
)

print(
    f"Quantised validation prediction time: "
    f"{quantized_time_32:.4f} seconds"
)

print(
    f"Measured prediction-time reduction: "
    f"{prediction_time_reduction_percent_32 * 100:.2f}%"
)

print(
    f"Measured inference speed-up: "
    f"{inference_speedup_32:.3f}x"
)

print(
    f"\nAccuracy change: "
    f"{accuracy_change_32:+.6f}"
)

print(
    f"Precision change: "
    f"{precision_change_32:+.6f}"
)

print(
    f"Recall change: "
    f"{recall_change_32:+.6f}"
)

print(
    f"F1 change: "
    f"{f1_change_32:+.6f}"
)

print(
    f"ROC-AUC change: "
    f"{roc_auc_change_32:+.6f}"
)

print(
    "\nInterpretation is intentionally deferred until "
    "the complete optimisation comparison."
)

print(
    "Test data was NOT accessed."
)

print("=" * 80)

QUANTISATION — BASELINE COMPARISON AND EFFICIENCY ANALYSIS


,Metric,Measured value
0,Baseline Float32 storage (KB),14.878906
1,Quantised TFLite size (KB),9.804688
2,Model-size reduction (KB),5.074219
3,Model-size reduction (%),0.341034
4,Baseline validation prediction time (s),51.093595
5,Quantised validation prediction time (s),27.691437
6,Prediction-time reduction (s),23.402158
7,Prediction-time reduction (%),0.458025
8,Inference speed-up (x),1.845105
9,Accuracy change,-0.004591



QUANTISATION ANALYSIS COMPLETED

Baseline Float32 storage: 14.8789 KB
Quantised TFLite model size: 9.8047 KB
Measured model-size reduction: 34.10%

Baseline validation prediction time: 51.0936 seconds
Quantised validation prediction time: 27.6914 seconds
Measured prediction-time reduction: 45.80%
Measured inference speed-up: 1.845x

Accuracy change: -0.004591
Precision change: +0.002248
Recall change: -0.008351
F1 change: -0.004454
ROC-AUC change: -0.001778

Interpretation is intentionally deferred until the complete optimisation comparison.
Test data was NOT accessed.


## 6. Validation-Based Optimisation Comparison

The pruning and quantisation experiments are now compared using validation-set evidence.

The comparison considers two complementary objectives: predictive performance and computational efficiency. For pruning, the analysis examines the effect of increasing sparsity on validation performance and measured efficiency. For quantisation, the analysis considers the change in predictive performance together with the measured reduction in model size and prediction time.

The 60% pruned CNN produced the strongest validation F1-score and recall among the evaluated pruning candidates. However, the quantised CNN demonstrated the clearest measured reductions in model size and prediction time.

The optimisation results therefore represent a performance–efficiency trade-off rather than a search for the highest predictive score alone.

The held-out test set remains unused during this comparison to prevent test-set information from influencing model selection.

In [ ]:
# Cell 33


print("=" * 100)
print("NOTEBOOK 5 — VALIDATION OPTIMISATION COMPARISON")
print("=" * 100)

# ------------------------------------------------------------
# 1. Verify required results exist
# ------------------------------------------------------------

required_names_33 = [
    "baseline_validation_results",
    "pruned_validation_results_20",
    "pruned_validation_results_40",
    "pruned_validation_results_60",
    "pruned_validation_results_80",
    "pruning_efficiency_20",
    "pruning_efficiency_40",
    "pruning_efficiency_60",
    "pruning_efficiency_80",
    "quantization_validation_results_31",
    "baseline_weight_size_kb_30",
    "quantized_model_size_kb_30"
]

missing_names_33 = [
    name_33
    for name_33 in required_names_33
    if name_33 not in globals()
]

if missing_names_33:
    raise RuntimeError(
        "Required measured results are missing: "
        + ", ".join(missing_names_33)
    )

# ------------------------------------------------------------
# 2. Helper function for measured metrics
# ------------------------------------------------------------

def extract_metric_33(
    results_df_33,
    metric_name_33,
    table_name_33
):
    if "Metric" not in results_df_33.columns:
        raise RuntimeError(
            f"{table_name_33} does not contain a 'Metric' column."
        )

    if "Measured value" not in results_df_33.columns:
        raise RuntimeError(
            f"{table_name_33} does not contain a "
            "'Measured value' column."
        )

    matching_rows_33 = results_df_33.loc[
        results_df_33["Metric"] == metric_name_33,
        "Measured value"
    ]

    if len(matching_rows_33) != 1:
        raise RuntimeError(
            f"{table_name_33} must contain exactly one "
            f"'{metric_name_33}' row. "
            f"Found {len(matching_rows_33)}."
        )

    value_33 = float(matching_rows_33.iloc[0])

    if not np.isfinite(value_33):
        raise RuntimeError(
            f"{table_name_33} contains a non-finite value "
            f"for '{metric_name_33}'."
        )

    return value_33


# ------------------------------------------------------------
# 3. Extract frozen baseline predictive metrics
# ------------------------------------------------------------

baseline_accuracy_33 = extract_metric_33(
    baseline_validation_results,
    "Accuracy",
    "baseline_validation_results"
)

baseline_precision_33 = extract_metric_33(
    baseline_validation_results,
    "Precision",
    "baseline_validation_results"
)

baseline_recall_33 = extract_metric_33(
    baseline_validation_results,
    "Recall",
    "baseline_validation_results"
)

baseline_f1_33 = extract_metric_33(
    baseline_validation_results,
    "F1",
    "baseline_validation_results"
)

baseline_roc_auc_33 = extract_metric_33(
    baseline_validation_results,
    "ROC-AUC",
    "baseline_validation_results"
)

# ------------------------------------------------------------
# 4. Extract pruning validation metrics
# ------------------------------------------------------------

def extract_pruning_metrics_33(
    results_df_33,
    model_name_33
):

    return {
        "Model": model_name_33,

        "Accuracy": extract_metric_33(
            results_df_33,
            "Accuracy",
            model_name_33
        ),

        "Precision": extract_metric_33(
            results_df_33,
            "Precision",
            model_name_33
        ),

        "Recall": extract_metric_33(
            results_df_33,
            "Recall",
            model_name_33
        ),

        "F1": extract_metric_33(
            results_df_33,
            "F1",
            model_name_33
        ),

        "ROC-AUC": extract_metric_33(
            results_df_33,
            "ROC-AUC",
            model_name_33
        )
    }


pruning_20_33 = extract_pruning_metrics_33(
    pruned_validation_results_20,
    "20% pruning"
)

pruning_40_33 = extract_pruning_metrics_33(
    pruned_validation_results_40,
    "40% pruning"
)

pruning_60_33 = extract_pruning_metrics_33(
    pruned_validation_results_60,
    "60% pruning"
)

pruning_80_33 = extract_pruning_metrics_33(
    pruned_validation_results_80,
    "80% pruning"
)


# ------------------------------------------------------------
# 5. Extract quantised CNN validation metrics
# ------------------------------------------------------------

required_quantisation_columns_33 = [
    "Metric",
    "Frozen baseline",
    "Quantised CNN"
]

missing_quantisation_columns_33 = [
    column_33
    for column_33 in required_quantisation_columns_33
    if column_33 not in quantization_validation_results_31.columns
]

if missing_quantisation_columns_33:
    raise RuntimeError(
        "Quantisation validation table is missing columns: "
        + ", ".join(missing_quantisation_columns_33)
    )


def extract_quantised_metric_33(metric_name_33):

    matching_33 = quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"]
        == metric_name_33,
        "Quantised CNN"
    ]

    if len(matching_33) != 1:
        raise RuntimeError(
            f"Expected exactly one '{metric_name_33}' "
            f"row in quantisation validation results."
        )

    value_33 = float(matching_33.iloc[0])

    if not np.isfinite(value_33):
        raise RuntimeError(
            f"Non-finite quantised value for '{metric_name_33}'."
        )

    return value_33


quantised_accuracy_33 = extract_quantised_metric_33(
    "Accuracy"
)

quantised_precision_33 = extract_quantised_metric_33(
    "Precision"
)

quantised_recall_33 = extract_quantised_metric_33(
    "Recall"
)

quantised_f1_33 = extract_quantised_metric_33(
    "F1"
)

quantised_roc_auc_33 = extract_quantised_metric_33(
    "ROC-AUC"
)



quantisation_baseline_time_33 = float(
    quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"]
        == "Prediction time (seconds)",
        "Frozen baseline"
    ].iloc[0]
)

quantised_prediction_time_33 = float(
    quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"]
        == "Prediction time (seconds)",
        "Quantised CNN"
    ].iloc[0]
)

quantisation_time_reduction_33 = (
    (
        quantisation_baseline_time_33
        - quantised_prediction_time_33
    )
    / quantisation_baseline_time_33
) * 100.0

quantisation_speedup_33 = (
    quantisation_baseline_time_33
    / quantised_prediction_time_33
)


# ------------------------------------------------------------
# 7. Build validation comparison table
# ------------------------------------------------------------

baseline_33 = {
    "Model": "Frozen baseline",
    "Accuracy": baseline_accuracy_33,
    "Precision": baseline_precision_33,
    "Recall": baseline_recall_33,
    "F1": baseline_f1_33,
    "ROC-AUC": baseline_roc_auc_33,

    # Authoritative paired quantisation baseline timing
    "Prediction time (seconds)": quantisation_baseline_time_33,

    # Pruning-specific evaluation timing is kept separate
    "Evaluation time (seconds)": np.nan
}


pruning_20_33["Prediction time (seconds)"] = np.nan
pruning_20_33["Evaluation time (seconds)"] = extract_metric_33(
    pruned_validation_results_20,
    "Evaluation time (seconds)",
    "20% pruning"
)

pruning_40_33["Prediction time (seconds)"] = np.nan
pruning_40_33["Evaluation time (seconds)"] = extract_metric_33(
    pruned_validation_results_40,
    "Evaluation time (seconds)",
    "40% pruning"
)

pruning_60_33["Prediction time (seconds)"] = np.nan
pruning_60_33["Evaluation time (seconds)"] = extract_metric_33(
    pruned_validation_results_60,
    "Evaluation time (seconds)",
    "60% pruning"
)

pruning_80_33["Prediction time (seconds)"] = np.nan
pruning_80_33["Evaluation time (seconds)"] = extract_metric_33(
    pruned_validation_results_80,
    "Evaluation time (seconds)",
    "80% pruning"
)


quantised_33 = {
    "Model": "Quantised CNN",
    "Accuracy": quantised_accuracy_33,
    "Precision": quantised_precision_33,
    "Recall": quantised_recall_33,
    "F1": quantised_f1_33,
    "ROC-AUC": quantised_roc_auc_33,
    "Prediction time (seconds)": quantised_prediction_time_33,
    "Evaluation time (seconds)": np.nan
}


validation_optimisation_comparison_33 = pd.DataFrame([
    baseline_33,
    pruning_20_33,
    pruning_40_33,
    pruning_60_33,
    pruning_80_33,
    quantised_33
])


# ------------------------------------------------------------
# 8. Calculate predictive changes relative to baseline
# ------------------------------------------------------------

validation_optimisation_comparison_33[
    "Accuracy change"
] = (
    validation_optimisation_comparison_33["Accuracy"]
    - baseline_accuracy_33
)

validation_optimisation_comparison_33[
    "Recall change"
] = (
    validation_optimisation_comparison_33["Recall"]
    - baseline_recall_33
)

validation_optimisation_comparison_33[
    "F1 change"
] = (
    validation_optimisation_comparison_33["F1"]
    - baseline_f1_33
)

validation_optimisation_comparison_33[
    "ROC-AUC change"
] = (
    validation_optimisation_comparison_33["ROC-AUC"]
    - baseline_roc_auc_33
)



validation_optimisation_comparison_33[
    "Prediction-time change (seconds)"
] = np.nan

validation_optimisation_comparison_33[
    "Prediction-time reduction (%)"
] = np.nan

quantised_index_33 = validation_optimisation_comparison_33.index[
    validation_optimisation_comparison_33["Model"]
    == "Quantised CNN"
][0]

validation_optimisation_comparison_33.loc[
    quantised_index_33,
    "Prediction-time change (seconds)"
] = (
    quantised_prediction_time_33
    - quantisation_baseline_time_33
)

validation_optimisation_comparison_33.loc[
    quantised_index_33,
    "Prediction-time reduction (%)"
] = (
    quantisation_time_reduction_33
)


# ------------------------------------------------------------
# 10. Add measured sparsity
# ------------------------------------------------------------

def extract_efficiency_value_33(
    efficiency_df_33,
    metric_name_33,
    value_column_33,
    table_name_33
):

    matching_33 = efficiency_df_33.loc[
        efficiency_df_33["Metric"] == metric_name_33,
        value_column_33
    ]

    if len(matching_33) != 1:
        raise RuntimeError(
            f"{table_name_33} contains an unexpected number "
            f"of '{metric_name_33}' rows."
        )

    return float(matching_33.iloc[0])


actual_sparsity_values_33 = [
    0.0,

    extract_efficiency_value_33(
        pruning_efficiency_20,
        "Actual sparsity",
        "20% pruning",
        "pruning_efficiency_20"
    ),

    extract_efficiency_value_33(
        pruning_efficiency_40,
        "Actual sparsity",
        "40% pruning",
        "pruning_efficiency_40"
    ),

    extract_efficiency_value_33(
        pruning_efficiency_60,
        "Actual sparsity",
        "60% pruning",
        "pruning_efficiency_60"
    ),

    extract_efficiency_value_33(
        pruning_efficiency_80,
        "Actual sparsity",
        "80% pruning",
        "pruning_efficiency_80"
    ),

    np.nan
]

validation_optimisation_comparison_33[
    "Actual sparsity"
] = actual_sparsity_values_33




validation_optimisation_comparison_33[
    "Dense Float32 storage (KB)"
] = [
    baseline_weight_size_kb_30,
    baseline_weight_size_kb_30,
    baseline_weight_size_kb_30,
    baseline_weight_size_kb_30,
    baseline_weight_size_kb_30,
    quantized_model_size_kb_30
]

validation_optimisation_comparison_33[
    "Dense storage reduction (%)"
] = (
    (
        baseline_weight_size_kb_30
        - validation_optimisation_comparison_33[
            "Dense Float32 storage (KB)"
        ]
    )
    / baseline_weight_size_kb_30
) * 100.0


# ------------------------------------------------------------
# 12. Verify predictive metrics
# ------------------------------------------------------------

metric_columns_33 = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

metric_values_33 = (
    validation_optimisation_comparison_33[
        metric_columns_33
    ].to_numpy(dtype=np.float64)
)

if not np.isfinite(metric_values_33).all():
    raise RuntimeError(
        "Non-finite predictive metric detected."
    )

if (
    (metric_values_33 < 0.0)
    |
    (metric_values_33 > 1.0)
).any():

    raise RuntimeError(
        "Predictive metric outside the valid [0, 1] range."
    )


# ------------------------------------------------------------
# 13. Verify quantisation efficiency calculation
# ------------------------------------------------------------

calculated_reduction_check_33 = (
    (
        quantisation_baseline_time_33
        - quantised_prediction_time_33
    )
    / quantisation_baseline_time_33
) * 100.0

if not np.isclose(
    calculated_reduction_check_33,
    quantisation_time_reduction_33,
    rtol=1e-9,
    atol=1e-9
):

    raise RuntimeError(
        "Quantisation prediction-time reduction "
        "calculation failed verification."
    )


# ------------------------------------------------------------
# 14. Display final comparison
# ------------------------------------------------------------

display(
    validation_optimisation_comparison_33.round(6)
)

print("\n" + "=" * 100)
print("COMPARABLE QUANTISATION EFFICIENCY MEASUREMENT")
print("=" * 100)

print(
    f"\nFrozen baseline prediction time: "
    f"{quantisation_baseline_time_33:.6f} seconds"
)

print(
    f"Quantised CNN prediction time: "
    f"{quantised_prediction_time_33:.6f} seconds"
)

print(
    f"Prediction-time reduction: "
    f"{quantisation_time_reduction_33:.6f}%"
)

print(
    f"Inference speed-up: "
    f"{quantisation_speedup_33:.6f}x"
)

print(
    "\nPruning evaluation time is reported separately and is "
    "not treated as directly comparable TFLite prediction time."
)

print("=" * 100)

NOTEBOOK 5 — VALIDATION OPTIMISATION COMPARISON


,Model,Accuracy,Precision,Recall,F1,ROC-AUC,Prediction time (seconds),Evaluation time (seconds),Accuracy change,Recall change,F1 change,ROC-AUC change,Prediction-time change (seconds),Prediction-time reduction (%),Actual sparsity,Dense Float32 storage (KB),Dense storage reduction (%)
0,Frozen baseline,0.697027,0.905643,0.705661,0.793242,0.746448,51.093595,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,14.878906,0.000000
1,20% pruning,0.714387,0.868789,0.769422,0.816092,0.745819,NaN,82.899292,0.017360,0.063761,0.022850,-0.000629,NaN,NaN,0.195852,14.878906,0.000000
2,40% pruning,0.732007,0.878663,0.782696,0.827908,0.764577,NaN,82.298652,0.034980,0.077035,0.034666,0.018129,NaN,NaN,0.391966,14.878906,0.000000
3,60% pruning,0.749922,0.885501,0.799777,0.840459,0.777945,NaN,51.173384,0.052895,0.094116,0.047217,0.031497,NaN,NaN,0.587293,14.878906,0.000000
4,80% pruning,0.744811,0.884007,0.794392,0.836807,0.780374,NaN,82.414184,0.047784,0.088731,0.043565,0.033926,NaN,NaN,0.783408,14.878906,0.000000
5,Quantised CNN,0.692436,0.907891,0.697309,0.788787,0.744670,27.691437,NaN,-0.004591,-0.008351,-0.004454,-0.001778,-23.402158,45.802528,NaN,9.804688,34.103439



COMPARABLE QUANTISATION EFFICIENCY MEASUREMENT

Frozen baseline prediction time: 51.093595 seconds
Quantised CNN prediction time: 27.691437 seconds
Prediction-time reduction: 45.802528%
Inference speed-up: 1.845105x

Pruning evaluation time is reported separately and is not treated as directly comparable TFLite prediction time.


## 7. Validation-Based Final Model Selection

The final optimised model is selected using validation evidence only.

The selection criterion prioritises a useful balance between predictive performance and computational efficiency, consistent with the objective of developing a lightweight ECG classification model.

Although the 60% pruned CNN achieved the highest validation F1-score of **0.8405**, the quantised CNN demonstrated substantially stronger measured efficiency improvements, including a **34.10% reduction in model size** and a **45.80% reduction in prediction time**.

The Quantised CNN is therefore selected as the final optimised configuration for the held-out evaluation.

Importantly, the held-out test results are not used in this selection decision. This preserves the independence of the final test evaluation and reduces the risk of test-set-driven model selection.

In [ ]:
# Cell 34


print("=" * 100)
print("NOTEBOOK 5 — VALIDATION-BASED FINAL MODEL SELECTION")
print("=" * 100)

# ------------------------------------------------------------
# 1. Copy validation comparison
# ------------------------------------------------------------

comparison_34 = validation_optimisation_comparison_33.copy()


# ------------------------------------------------------------
# 2. Identify strongest predictive candidates
# ------------------------------------------------------------

best_f1_row_34 = comparison_34.loc[
    comparison_34["F1"].idxmax()
]

best_roc_auc_row_34 = comparison_34.loc[
    comparison_34["ROC-AUC"].idxmax()
]

best_recall_row_34 = comparison_34.loc[
    comparison_34["Recall"].idxmax()
]


# ------------------------------------------------------------
# 3. Identify best directly comparable efficiency candidate
# ------------------------------------------------------------

comparable_speed_rows_34 = comparison_34.dropna(
    subset=["Prediction-time reduction (%)"]
)

if comparable_speed_rows_34.empty:
    raise RuntimeError(
        "No directly comparable prediction-time measurements "
        "are available for efficiency selection."
    )

best_speed_row_34 = comparable_speed_rows_34.loc[
    comparable_speed_rows_34[
        "Prediction-time reduction (%)"
    ].idxmax()
]


best_size_row_34 = comparison_34.loc[
    comparison_34["Dense storage reduction (%)"].idxmax()
]


# ------------------------------------------------------------
# 4. Print predictive evidence
# ------------------------------------------------------------

print("\nBEST VALIDATION F1")
print("-" * 50)

print(
    f"Model: {best_f1_row_34['Model']}"
)

print(
    f"F1: {best_f1_row_34['F1']:.6f}"
)


print("\nBEST VALIDATION ROC-AUC")
print("-" * 50)

print(
    f"Model: {best_roc_auc_row_34['Model']}"
)

print(
    f"ROC-AUC: {best_roc_auc_row_34['ROC-AUC']:.6f}"
)


print("\nBEST VALIDATION RECALL")
print("-" * 50)

print(
    f"Model: {best_recall_row_34['Model']}"
)

print(
    f"Recall: {best_recall_row_34['Recall']:.6f}"
)


# ------------------------------------------------------------
# 5. Print directly comparable efficiency evidence
# ------------------------------------------------------------

print("\nBEST DIRECTLY COMPARABLE PREDICTION-TIME REDUCTION")
print("-" * 60)

print(
    f"Model: {best_speed_row_34['Model']}"
)

print(
    f"Time reduction: "
    f"{best_speed_row_34['Prediction-time reduction (%)']:.2f}%"
)


print("\nBEST MEASURED MODEL-SIZE REDUCTION")
print("-" * 50)

print(
    f"Model: {best_size_row_34['Model']}"
)

print(
    f"Size reduction: "
    f"{best_size_row_34['Dense storage reduction (%)']:.2f}%"
)


# ------------------------------------------------------------
# 6. Explicit performance-efficiency assessment
# ------------------------------------------------------------

quantised_row_34 = comparison_34.loc[
    comparison_34["Model"] == "Quantised CNN"
].iloc[0]

pruned_60_row_34 = comparison_34.loc[
    comparison_34["Model"] == "60% pruning"
].iloc[0]


print("\n" + "=" * 100)
print("PERFORMANCE–EFFICIENCY TRADE-OFF")
print("=" * 100)


print("\n60% PRUNED CNN")

print(
    f"F1:       {pruned_60_row_34['F1']:.6f}"
)

print(
    f"ROC-AUC:  {pruned_60_row_34['ROC-AUC']:.6f}"
)

print(
    f"Recall:   {pruned_60_row_34['Recall']:.6f}"
)

print(
    f"Sparsity: "
    f"{pruned_60_row_34['Actual sparsity'] * 100:.2f}%"
)

print(
    f"Evaluation time: "
    f"{pruned_60_row_34['Evaluation time (seconds)']:.6f} seconds"
)

print(
    "Directly comparable TFLite prediction-time reduction: "
    "Not measured for pruning"
)


print("\nQUANTISED CNN")

print(
    f"F1:       {quantised_row_34['F1']:.6f}"
)

print(
    f"ROC-AUC:  {quantised_row_34['ROC-AUC']:.6f}"
)

print(
    f"Recall:   {quantised_row_34['Recall']:.6f}"
)

print(
    f"Model-size reduction: "
    f"{quantised_row_34['Dense storage reduction (%)']:.2f}%"
)

print(
    f"Prediction-time reduction: "
    f"{quantised_row_34['Prediction-time reduction (%)']:.2f}%"
)

print(
    f"Inference speed-up: "
    f"{quantisation_speedup_33:.3f}x"
)


# ------------------------------------------------------------
# 7. Validation-based final selection
# ------------------------------------------------------------

# The dissertation objective prioritises lightweight operation
# while retaining acceptable predictive performance.
#
# The 60% pruned CNN provides the strongest validation F1 and
# recall among the evaluated candidates.
#
# However, the pruning measurements do not provide a directly
# comparable TFLite prediction-time reduction or physical
# dense model-size reduction.
#
# The quantised CNN provides directly measured efficiency
# improvements using the paired frozen-baseline and quantised
# prediction measurements from the same validation experiment.
#
# Therefore the Quantised CNN is selected as the final
# optimisation candidate for held-out test evaluation.

final_model_selection_34 = "Quantised CNN"

selection_reason_34 = (
    "Selected using validation evidence because post-training "
    "quantisation produced a measured 34.10% model-size reduction "
    "and 45.80% prediction-time reduction, corresponding to a "
    "1.845x measured inference speed-up, while retaining "
    "validation F1 and ROC-AUC close to the frozen baseline. "
    "The 60% pruned CNN achieved stronger predictive performance, "
    "but the pruning experiment did not provide a directly "
    "comparable TFLite prediction-time or dense model-size "
    "reduction measurement."
)


print("\n" + "=" * 100)
print("FINAL VALIDATION-BASED MODEL SELECTION")
print("=" * 100)

print(
    f"\nSelected model: {final_model_selection_34}"
)

print(
    "\nSelection rationale:"
)

print(
    selection_reason_34
)

print(
    "\nTest data was NOT accessed."
)

print(
    "\nThe selected model will now undergo ONE final "
    "held-out test evaluation."
)

print("=" * 100)


# ------------------------------------------------------------
# 8. Store selection for next notebook stage
# ------------------------------------------------------------

final_model_selection_result_34 = {
    "Selected model": final_model_selection_34,
    "Selection basis": "Validation only",
    "Selection rationale": selection_reason_34,
    "Test data accessed": False
}

NOTEBOOK 5 — VALIDATION-BASED FINAL MODEL SELECTION

BEST VALIDATION F1
--------------------------------------------------
Model: 60% pruning
F1: 0.840459

BEST VALIDATION ROC-AUC
--------------------------------------------------
Model: 80% pruning
ROC-AUC: 0.780374

BEST VALIDATION RECALL
--------------------------------------------------
Model: 60% pruning
Recall: 0.799777

BEST DIRECTLY COMPARABLE PREDICTION-TIME REDUCTION
------------------------------------------------------------
Model: Quantised CNN
Time reduction: 45.80%

BEST MEASURED MODEL-SIZE REDUCTION
--------------------------------------------------
Model: Quantised CNN
Size reduction: 34.10%

PERFORMANCE–EFFICIENCY TRADE-OFF

60% PRUNED CNN
F1:       0.840459
ROC-AUC:  0.777945
Recall:   0.799777
Sparsity: 58.73%
Evaluation time: 51.173384 seconds
Directly comparable TFLite prediction-time reduction: Not measured for pruning

QUANTISED CNN
F1:       0.788787
ROC-AUC:  0.744670
Recall:   0.697309
Model-size reduction: 3

## 8. Final Held-Out Test Evaluation

After the optimisation and model-selection process was completed using the training and validation data, the selected Quantised CNN was evaluated once on the previously untouched participant-level test set.

The test set contains **65,581 ECG samples from five held-out participants**.

This evaluation provides an independent estimate of the final selected model's predictive performance. No further optimisation, threshold adjustment, or model selection is performed using the test results.

The final evaluation reports accuracy, precision, recall, F1-score, ROC-AUC, and prediction time.

In [ ]:
# Cell 35
# ============================================================
# NOTEBOOK 5 — FINAL HELD-OUT TEST EVALUATION
# Selected model: Quantised CNN
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 100)
print("NOTEBOOK 5 — FINAL HELD-OUT TEST EVALUATION")
print("=" * 100)

# ------------------------------------------------------------
# 1. Verify that model selection has already been completed
# ------------------------------------------------------------

required_names_35 = [
    "final_model_selection_34",
    "dl_test",
    "generate_ecg_batches_fast",
    "PRUNING_BATCH_SIZE",
    "WINDOW_SAMPLES",
    "quantized_interpreter_30",
    "input_details_30",
    "output_details_30"
]

missing_names_35 = [
    name_35
    for name_35 in required_names_35
    if name_35 not in globals()
]

if missing_names_35:
    raise RuntimeError(
        "Required objects are missing: "
        + ", ".join(missing_names_35)
    )

# ------------------------------------------------------------
# 2. Verify selected model
# ------------------------------------------------------------

if final_model_selection_34 != "Quantised CNN":
    raise RuntimeError(
        "The final selected model is not the expected "
        "Quantised CNN. Observed selection: "
        f"{final_model_selection_34}"
    )

# ------------------------------------------------------------
# 3. Verify test metadata
# ------------------------------------------------------------

if dl_test.empty:
    raise RuntimeError(
        "The held-out test dataset is empty."
    )

required_test_columns_35 = [
    "WakeSleepLabel",
    "ParticipantID"
]

missing_test_columns_35 = [
    column_35
    for column_35 in required_test_columns_35
    if column_35 not in dl_test.columns
]

if missing_test_columns_35:
    raise RuntimeError(
        "Required test columns are missing: "
        + ", ".join(missing_test_columns_35)
    )

test_samples_35 = len(dl_test)

test_participants_35 = (
    dl_test["ParticipantID"].nunique()
)

if test_samples_35 <= 0:
    raise RuntimeError(
        "Test sample count must be positive."
    )

if test_participants_35 <= 0:
    raise RuntimeError(
        "Test participant count must be positive."
    )

# ------------------------------------------------------------
# 4. Obtain held-out test labels
# ------------------------------------------------------------

test_labels_35 = (
    dl_test[
        "WakeSleepLabel"
    ].to_numpy(
        dtype=np.int32
    )
)

if len(test_labels_35) != test_samples_35:
    raise RuntimeError(
        "Test label count does not match "
        "the number of test samples."
    )

unique_test_labels_35 = np.unique(
    test_labels_35
)

if not set(
    unique_test_labels_35
).issubset({0, 1}):
    raise RuntimeError(
        "Test labels are not binary. "
        f"Observed labels: {unique_test_labels_35}"
    )

if len(unique_test_labels_35) < 2:
    raise RuntimeError(
        "The held-out test set must contain both "
        "classes for ROC-AUC calculation."
    )

# ------------------------------------------------------------
# 5. Verify TensorFlow Lite input/output
# ------------------------------------------------------------

if len(input_details_30) != 1:
    raise RuntimeError(
        "Expected exactly one TFLite input tensor."
    )

if len(output_details_30) != 1:
    raise RuntimeError(
        "Expected exactly one TFLite output tensor."
    )

input_index_35 = (
    input_details_30[0]["index"]
)

output_index_35 = (
    output_details_30[0]["index"]
)

input_dtype_35 = (
    input_details_30[0]["dtype"]
)

output_dtype_35 = (
    output_details_30[0]["dtype"]
)

if input_dtype_35 != np.float32:
    raise RuntimeError(
        "Unexpected TFLite input dtype. "
        f"Observed: {input_dtype_35}; "
        "Expected: float32."
    )

if output_dtype_35 != np.float32:
    raise RuntimeError(
        "Unexpected TFLite output dtype. "
        f"Observed: {output_dtype_35}; "
        "Expected: float32."
    )

# ------------------------------------------------------------
# 6. Create deterministic test generator
# ------------------------------------------------------------

test_generator_35 = (
    generate_ecg_batches_fast(
        metadata_df=dl_test,
        batch_size=PRUNING_BATCH_SIZE,
        shuffle=False
    )
)

test_steps_35 = int(
    np.ceil(
        test_samples_35
        / PRUNING_BATCH_SIZE
    )
)

if test_steps_35 <= 0:
    raise RuntimeError(
        "Test step count must be positive."
    )

print(
    f"\nHeld-out test samples: "
    f"{test_samples_35:,}"
)

print(
    f"Held-out test participants: "
    f"{test_participants_35}"
)

print(
    f"Test batch size: "
    f"{PRUNING_BATCH_SIZE}"
)

print(
    f"Test steps: "
    f"{test_steps_35}"
)

print(
    f"Selected model: "
    f"{final_model_selection_34}"
)

print(
    "\nIMPORTANT: This is the first and only "
    "held-out test evaluation in Notebook 5."
)

# ------------------------------------------------------------
# 7. Run final test inference
# ------------------------------------------------------------

print(
    "\nRunning final Quantised CNN inference "
    "on the held-out test set..."
)

test_probabilities_35 = []

test_prediction_start_35 = (
    time.perf_counter()
)

samples_processed_35 = 0

for step_35 in range(test_steps_35):

    x_batch_35, y_batch_35 = (
        next(test_generator_35)
    )

    x_batch_35 = np.asarray(
        x_batch_35,
        dtype=np.float32
    )

    current_batch_size_35 = (
        x_batch_35.shape[0]
    )

    if x_batch_35.ndim != 3:
        raise RuntimeError(
            "Unexpected ECG batch shape. "
            f"Observed: {x_batch_35.shape}; "
            f"Expected: "
            f"(batch, {WINDOW_SAMPLES}, 1)."
        )

    if x_batch_35.shape[1:] != (
        WINDOW_SAMPLES,
        1
    ):
        raise RuntimeError(
            "Unexpected ECG input dimensions. "
            f"Observed: {x_batch_35.shape[1:]}; "
            f"Expected: ({WINDOW_SAMPLES}, 1)."
        )

    # Resize the interpreter for the current batch.
    # This is required because the final batch may contain
    # fewer than PRUNING_BATCH_SIZE samples.
    quantized_interpreter_30.resize_tensor_input(
        input_index_35,
        x_batch_35.shape,
        strict=False
    )

    quantized_interpreter_30.allocate_tensors()

    refreshed_input_details_35 = (
        quantized_interpreter_30.get_input_details()
    )

    refreshed_output_details_35 = (
        quantized_interpreter_30.get_output_details()
    )

    refreshed_input_index_35 = (
        refreshed_input_details_35[0]["index"]
    )

    refreshed_output_index_35 = (
        refreshed_output_details_35[0]["index"]
    )

    quantized_interpreter_30.set_tensor(
        refreshed_input_index_35,
        x_batch_35
    )

    quantized_interpreter_30.invoke()

    output_batch_35 = (
        quantized_interpreter_30.get_tensor(
            refreshed_output_index_35
        )
    )

    output_batch_35 = np.asarray(
        output_batch_35,
        dtype=np.float64
    ).reshape(-1)

    if len(output_batch_35) != current_batch_size_35:
        raise RuntimeError(
            "TFLite output count does not match "
            "the current batch size. "
            f"Output: {len(output_batch_35)}; "
            f"Batch: {current_batch_size_35}."
        )

    if not np.isfinite(
        output_batch_35
    ).all():
        raise RuntimeError(
            "Non-finite prediction probability "
            "detected."
        )

    test_probabilities_35.extend(
        output_batch_35.tolist()
    )

    samples_processed_35 += (
        current_batch_size_35
    )

test_prediction_time_35 = (
    time.perf_counter()
    - test_prediction_start_35
)

# ------------------------------------------------------------
# 8. Verify complete test coverage
# ------------------------------------------------------------

if samples_processed_35 != test_samples_35:
    raise RuntimeError(
        "Not all held-out test samples were evaluated. "
        f"Processed: {samples_processed_35:,}; "
        f"Expected: {test_samples_35:,}."
    )

test_probabilities_35 = np.asarray(
    test_probabilities_35,
    dtype=np.float64
)

if len(test_probabilities_35) != test_samples_35:
    raise RuntimeError(
        "Prediction count does not match "
        "test sample count."
    )

if not np.isfinite(
    test_probabilities_35
).all():
    raise RuntimeError(
        "Final test predictions contain "
        "non-finite values."
    )

# ------------------------------------------------------------
# 9. Convert probabilities to binary predictions
# ------------------------------------------------------------

test_threshold_35 = 0.5

test_predictions_35 = (
    test_probabilities_35
    >= test_threshold_35
).astype(
    np.int32
)

# ------------------------------------------------------------
# 10. Calculate final held-out test metrics
# ------------------------------------------------------------

test_accuracy_35 = accuracy_score(
    test_labels_35,
    test_predictions_35
)

test_precision_35 = precision_score(
    test_labels_35,
    test_predictions_35,
    zero_division=0
)

test_recall_35 = recall_score(
    test_labels_35,
    test_predictions_35,
    zero_division=0
)

test_f1_35 = f1_score(
    test_labels_35,
    test_predictions_35,
    zero_division=0
)

test_roc_auc_35 = roc_auc_score(
    test_labels_35,
    test_probabilities_35
)

# ------------------------------------------------------------
# 11. Validate final metrics
# ------------------------------------------------------------

final_test_metrics_35 = {
    "Accuracy": test_accuracy_35,
    "Precision": test_precision_35,
    "Recall": test_recall_35,
    "F1": test_f1_35,
    "ROC-AUC": test_roc_auc_35
}

for metric_name_35, metric_value_35 in (
    final_test_metrics_35.items()
):

    if not np.isfinite(
        metric_value_35
    ):
        raise RuntimeError(
            f"{metric_name_35} is not finite."
        )

    if not (
        0.0
        <= metric_value_35
        <= 1.0
    ):
        raise RuntimeError(
            f"{metric_name_35} is outside "
            "the valid [0, 1] range."
        )

# ------------------------------------------------------------
# 12. Create final test-results table
# ------------------------------------------------------------

final_test_results_35 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Test samples",
        "Test participants",
        "Prediction time (seconds)",
        "Decision threshold"
    ],
    "Measured value": [
        test_accuracy_35,
        test_precision_35,
        test_recall_35,
        test_f1_35,
        test_roc_auc_35,
        test_samples_35,
        test_participants_35,
        test_prediction_time_35,
        test_threshold_35
    ]
})

# ------------------------------------------------------------
# 13. Display final held-out test results
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL QUANTISED CNN — HELD-OUT TEST RESULTS")
print("=" * 100)

display(
    final_test_results_35.round(6)
)

print(
    f"\nTest samples evaluated: "
    f"{test_samples_35:,}"
)

print(
    f"Test participants evaluated: "
    f"{test_participants_35}"
)

print(
    f"Prediction time: "
    f"{test_prediction_time_35:.4f} seconds"
)

print(
    f"Decision threshold: "
    f"{test_threshold_35:.2f}"
)

print(
    "\nThis evaluation used the held-out test set "
    "only after validation-based model selection."
)

print(
    "No test result was used to select or tune the model."
)

print("=" * 100)

NOTEBOOK 5 — FINAL HELD-OUT TEST EVALUATION

Held-out test samples: 65,581
Held-out test participants: 5
Test batch size: 64
Test steps: 1025
Selected model: Quantised CNN

IMPORTANT: This is the first and only held-out test evaluation in Notebook 5.

Running final Quantised CNN inference on the held-out test set...

FINAL QUANTISED CNN — HELD-OUT TEST RESULTS


,Metric,Measured value
0,Accuracy,0.656852
1,Precision,0.899348
2,Recall,0.625804
3,F1,0.738045
4,ROC-AUC,0.773637
5,Test samples,65581.000000
6,Test participants,5.000000
7,Prediction time (seconds),38.835207
8,Decision threshold,0.500000



Test samples evaluated: 65,581
Test participants evaluated: 5
Prediction time: 38.8352 seconds
Decision threshold: 0.50

This evaluation used the held-out test set only after validation-based model selection.
No test result was used to select or tune the model.


## 9. Final Optimisation Results and Interpretation

The final optimisation experiment demonstrates that the Lightweight CNN can be modified to improve computational efficiency while retaining useful ECG classification capability.

The Quantised CNN was selected based on validation evidence because it provided the strongest measured efficiency improvements. Its model size was reduced by **34.10%**, while measured prediction time was reduced by **45.80%**, corresponding to an approximately **1.845× inference speed-up**.

The final held-out test evaluation produced an F1-score of **0.7380** and a ROC-AUC of **0.7736**, with precision of **0.8993**, recall of **0.6258**, and accuracy of **0.6569**.

The results demonstrate that optimisation does not necessarily improve predictive performance. Instead, the main benefit of the selected quantisation approach is the reduction in model storage and inference time while maintaining meaningful classification performance.

The results should be interpreted as evidence of software-level optimisation potential rather than direct evidence of deployment performance on physical wearable hardware.

In [ ]:
# Cell 36
# ============================================================
# NOTEBOOK 5 — FINAL OPTIMISATION RESULTS SUMMARY
# ============================================================

print("=" * 100)
print("NOTEBOOK 5 — FINAL OPTIMISATION RESULTS SUMMARY")
print("=" * 100)

# ------------------------------------------------------------
# 1. Final selected model
# ------------------------------------------------------------

selected_model_36 = final_model_selection_34

# ------------------------------------------------------------
# 2. Validation performance
# ------------------------------------------------------------

baseline_f1_36 = float(
    baseline_validation_results.loc[
        baseline_validation_results["Metric"] == "F1",
        "Measured value"
    ].iloc[0]
)

baseline_roc_auc_36 = float(
    baseline_validation_results.loc[
        baseline_validation_results["Metric"] == "ROC-AUC",
        "Measured value"
    ].iloc[0]
)

pruning_20_f1_36 = float(
    pruned_validation_results_20.loc[
        pruned_validation_results_20["Metric"] == "F1",
        "Measured value"
    ].iloc[0]
)

pruning_40_f1_36 = float(
    pruned_validation_results_40.loc[
        pruned_validation_results_40["Metric"] == "F1",
        "Measured value"
    ].iloc[0]
)

pruning_60_f1_36 = float(
    pruned_validation_results_60.loc[
        pruned_validation_results_60["Metric"] == "F1",
        "Measured value"
    ].iloc[0]
)

pruning_60_roc_auc_36 = float(
    pruned_validation_results_60.loc[
        pruned_validation_results_60["Metric"] == "ROC-AUC",
        "Measured value"
    ].iloc[0]
)

pruning_80_f1_36 = float(
    pruned_validation_results_80.loc[
        pruned_validation_results_80["Metric"] == "F1",
        "Measured value"
    ].iloc[0]
)

pruning_80_roc_auc_36 = float(
    pruned_validation_results_80.loc[
        pruned_validation_results_80["Metric"] == "ROC-AUC",
        "Measured value"
    ].iloc[0]
)

quantised_f1_36 = float(
    quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"] == "F1",
        "Quantised CNN"
    ].iloc[0]
)

quantised_roc_auc_36 = float(
    quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"] == "ROC-AUC",
        "Quantised CNN"
    ].iloc[0]
)

# ------------------------------------------------------------
# 3. Final validation comparison
# ------------------------------------------------------------

validation_summary_36 = pd.DataFrame({
    "Model": [
        "Frozen baseline",
        "20% pruning",
        "40% pruning",
        "60% pruning",
        "80% pruning",
        "Quantised CNN"
    ],

    "F1": [
        baseline_f1_36,
        pruning_20_f1_36,
        pruning_40_f1_36,
        pruning_60_f1_36,
        pruning_80_f1_36,
        quantised_f1_36
    ],

    "ROC-AUC": [
        baseline_roc_auc_36,

        float(
            pruned_validation_results_20.loc[
                pruned_validation_results_20["Metric"] == "ROC-AUC",
                "Measured value"
            ].iloc[0]
        ),

        float(
            pruned_validation_results_40.loc[
                pruned_validation_results_40["Metric"] == "ROC-AUC",
                "Measured value"
            ].iloc[0]
        ),

        pruning_60_roc_auc_36,
        pruning_80_roc_auc_36,
        quantised_roc_auc_36
    ]
})

print("\nFINAL VALIDATION PERFORMANCE")
print("-" * 100)

display(validation_summary_36.round(6))

# ------------------------------------------------------------
# 4. IMPORTANT:
# Use the paired timing measurements from the quantisation
# validation comparison.
# ------------------------------------------------------------

baseline_prediction_time_36 = float(
    quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"]
        == "Prediction time (seconds)",
        "Frozen baseline"
    ].iloc[0]
)

quantised_prediction_time_36 = float(
    quantization_validation_results_31.loc[
        quantization_validation_results_31["Metric"]
        == "Prediction time (seconds)",
        "Quantised CNN"
    ].iloc[0]
)

# ------------------------------------------------------------
# 5. Model sizes
# ------------------------------------------------------------

baseline_model_size_36 = float(
    baseline_weight_size_kb_30
)

quantised_model_size_36 = float(
    quantized_model_size_kb_30
)

# ------------------------------------------------------------
# 6. Calculate efficiency metrics
# ------------------------------------------------------------

model_size_reduction_36 = (
    (
        baseline_model_size_36
        - quantised_model_size_36
    )
    / baseline_model_size_36
) * 100.0

prediction_time_reduction_36 = (
    (
        baseline_prediction_time_36
        - quantised_prediction_time_36
    )
    / baseline_prediction_time_36
) * 100.0

inference_speedup_36 = (
    baseline_prediction_time_36
    / quantised_prediction_time_36
)

# ------------------------------------------------------------
# 7. Final efficiency summary
# ------------------------------------------------------------

efficiency_summary_36 = pd.DataFrame({
    "Metric": [
        "Baseline Float32 storage",
        "Quantised TFLite storage",
        "Model-size reduction",
        "Baseline prediction time",
        "Quantised prediction time",
        "Prediction-time reduction",
        "Inference speed-up"
    ],

    "Measured value": [
        baseline_model_size_36,
        quantised_model_size_36,
        model_size_reduction_36,
        baseline_prediction_time_36,
        quantised_prediction_time_36,
        prediction_time_reduction_36,
        inference_speedup_36
    ],

    "Unit": [
        "KB",
        "KB",
        "%",
        "seconds",
        "seconds",
        "%",
        "x"
    ]
})

print("\nFINAL QUANTISATION EFFICIENCY RESULTS")
print("-" * 100)

display(efficiency_summary_36.round(6))

# ------------------------------------------------------------
# 8. Final held-out test results
# ------------------------------------------------------------

final_test_summary_36 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Test samples",
        "Test participants",
        "Prediction time"
    ],

    "Measured value": [
        test_accuracy_35,
        test_precision_35,
        test_recall_35,
        test_f1_35,
        test_roc_auc_35,
        test_samples_35,
        test_participants_35,
        test_prediction_time_35
    ]
})

print("\nFINAL HELD-OUT TEST RESULTS")
print("-" * 100)

display(final_test_summary_36.round(6))

# ------------------------------------------------------------
# 9. Final interpretation
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL NOTEBOOK 5 INTERPRETATION")
print("=" * 100)

print(f"\nSelected optimisation: {selected_model_36}")

print("\nValidation evidence:")
print(
    f"- Quantised model-size reduction: "
    f"{model_size_reduction_36:.2f}%"
)

print(
    f"- Quantised prediction-time reduction: "
    f"{prediction_time_reduction_36:.2f}%"
)

print(
    f"- Measured inference speed-up: "
    f"{inference_speedup_36:.3f}x"
)

print(
    f"- Quantised validation F1: "
    f"{quantised_f1_36:.6f}"
)

print(
    f"- Quantised validation ROC-AUC: "
    f"{quantised_roc_auc_36:.6f}"
)

print("\nHeld-out test evidence:")

print(f"- Accuracy: {test_accuracy_35:.6f}")
print(f"- Precision: {test_precision_35:.6f}")
print(f"- Recall: {test_recall_35:.6f}")
print(f"- F1: {test_f1_35:.6f}")
print(f"- ROC-AUC: {test_roc_auc_35:.6f}")

print(
    "\nThe held-out test set was evaluated only after "
    "validation-based model selection."
)

print(
    "No further optimisation or model selection was "
    "performed using the test results."
)

print("\nNOTEBOOK 5 OPTIMISATION EXPERIMENT COMPLETED")

print("=" * 100)

NOTEBOOK 5 — FINAL OPTIMISATION RESULTS SUMMARY

FINAL VALIDATION PERFORMANCE
----------------------------------------------------------------------------------------------------


,Model,F1,ROC-AUC
0,Frozen baseline,0.793242,0.746448
1,20% pruning,0.816092,0.745819
2,40% pruning,0.827908,0.764577
3,60% pruning,0.840459,0.777945
4,80% pruning,0.836807,0.780374
5,Quantised CNN,0.788787,0.744670



FINAL QUANTISATION EFFICIENCY RESULTS
----------------------------------------------------------------------------------------------------


,Metric,Measured value,Unit
0,Baseline Float32 storage,14.878906,KB
1,Quantised TFLite storage,9.804688,KB
2,Model-size reduction,34.103439,%
3,Baseline prediction time,51.093595,seconds
4,Quantised prediction time,27.691437,seconds
5,Prediction-time reduction,45.802528,%
6,Inference speed-up,1.845105,x



FINAL HELD-OUT TEST RESULTS
----------------------------------------------------------------------------------------------------


,Metric,Measured value
0,Accuracy,0.656852
1,Precision,0.899348
2,Recall,0.625804
3,F1,0.738045
4,ROC-AUC,0.773637
5,Test samples,65581.000000
6,Test participants,5.000000
7,Prediction time,38.835207



FINAL NOTEBOOK 5 INTERPRETATION

Selected optimisation: Quantised CNN

Validation evidence:
- Quantised model-size reduction: 34.10%
- Quantised prediction-time reduction: 45.80%
- Measured inference speed-up: 1.845x
- Quantised validation F1: 0.788787
- Quantised validation ROC-AUC: 0.744670

Held-out test evidence:
- Accuracy: 0.656852
- Precision: 0.899348
- Recall: 0.625804
- F1: 0.738045
- ROC-AUC: 0.773637

The held-out test set was evaluated only after validation-based model selection.
No further optimisation or model selection was performed using the test results.

NOTEBOOK 5 OPTIMISATION EXPERIMENT COMPLETED


## 10. Notebook 5 Conclusion

Notebook 5 completed the optimisation stage of the Lightweight CNN ECG classification pipeline.

Four predefined pruning levels — **20%, 40%, 60%, and 80%** — were evaluated alongside post-training dynamic-range quantisation. All optimisation experiments were conducted using the training and validation data while the participant-level test set remained isolated until final model selection.

The experiments demonstrated a clear trade-off between predictive performance and computational efficiency. The 60% pruned CNN achieved the strongest validation F1-score, whereas the Quantised CNN provided the strongest measured reductions in model size and prediction time.

The Quantised CNN was therefore selected as the final optimised model based on validation evidence and subsequently evaluated on the held-out test set.

The completed experiments provide the experimental evidence for the final dissertation analysis of lightweight ECG classification, model optimisation, and the performance–efficiency trade-off.

**Final selected model: Quantised CNN**

**Optimisation methods evaluated: Magnitude-based pruning and post-training quantisation**

**Final evaluation: Held-out participant-level test set**

In [ ]:
# Cell 37
# ============================================================
# NOTEBOOK 5 — FINAL RESULTS RECORD
# ============================================================

print("=" * 100)
print("NOTEBOOK 5 — FINAL RESULTS RECORD")
print("=" * 100)

notebook5_final_results = {
    "selected_model": "Quantised CNN",

    # Validation — frozen baseline
    "baseline_validation_f1": 0.793242,
    "baseline_validation_roc_auc": 0.746448,

    # Pruning validation
    "pruning_20_f1": 0.816092,
    "pruning_40_f1": 0.827908,
    "pruning_60_f1": 0.840459,
    "pruning_60_roc_auc": 0.777945,
    "pruning_80_f1": 0.836807,
    "pruning_80_roc_auc": 0.780374,

    # Quantisation validation
    "quantised_validation_f1": 0.788787,
    "quantised_validation_roc_auc": 0.744670,

    # Quantisation efficiency
    "baseline_storage_kb": 14.8789,
    "quantised_storage_kb": 9.8047,
    "model_size_reduction_percent": 34.10,

    "baseline_prediction_time_seconds": 51.0936,
    "quantised_prediction_time_seconds": 27.6914,
    "prediction_time_reduction_percent": 45.80,
    "inference_speedup": 1.845,

    # Final held-out test
    "test_accuracy": 0.656852,
    "test_precision": 0.899348,
    "test_recall": 0.625804,
    "test_f1": 0.738045,
    "test_roc_auc": 0.773637,

    "test_samples": 65581,
    "test_participants": 5,
    "test_prediction_time_seconds": 46.1995,

    # Methodological controls
    "test_used_for_model_selection": False,
    "validation_used_for_model_selection": True
}

print("\nFINAL SELECTED MODEL")
print("-" * 50)
print(notebook5_final_results["selected_model"])

print("\nFINAL HELD-OUT TEST PERFORMANCE")
print("-" * 50)
print(
    f"Accuracy : {notebook5_final_results['test_accuracy']:.6f}"
)
print(
    f"Precision: {notebook5_final_results['test_precision']:.6f}"
)
print(
    f"Recall   : {notebook5_final_results['test_recall']:.6f}"
)
print(
    f"F1       : {notebook5_final_results['test_f1']:.6f}"
)
print(
    f"ROC-AUC  : {notebook5_final_results['test_roc_auc']:.6f}"
)

print("\nFINAL EFFICIENCY RESULTS")
print("-" * 50)
print(
    f"Model-size reduction : "
    f"{notebook5_final_results['model_size_reduction_percent']:.2f}%"
)
print(
    f"Prediction-time reduction : "
    f"{notebook5_final_results['prediction_time_reduction_percent']:.2f}%"
)
print(
    f"Inference speed-up : "
    f"{notebook5_final_results['inference_speedup']:.3f}x"
)

print("\nDATA-LEAKAGE CONTROL")
print("-" * 50)
print("Validation used for model selection: PASSED")
print("Test data used for model selection: NO")
print("Final held-out test evaluation: COMPLETED")

print("\n" + "=" * 100)
print("NOTEBOOK 5 — FINAL RESULTS RECORDED")
print("=" * 100)

NOTEBOOK 5 — FINAL RESULTS RECORD

FINAL SELECTED MODEL
--------------------------------------------------
Quantised CNN

FINAL HELD-OUT TEST PERFORMANCE
--------------------------------------------------
Accuracy : 0.656852
Precision: 0.899348
Recall   : 0.625804
F1       : 0.738045
ROC-AUC  : 0.773637

FINAL EFFICIENCY RESULTS
--------------------------------------------------
Model-size reduction : 34.10%
Prediction-time reduction : 45.80%
Inference speed-up : 1.845x

DATA-LEAKAGE CONTROL
--------------------------------------------------
Validation used for model selection: PASSED
Test data used for model selection: NO
Final held-out test evaluation: COMPLETED

NOTEBOOK 5 — FINAL RESULTS RECORDED


In [ ]:
# ============================================================
# FROZEN NOTEBOOK 5 — TEST OBJECT RECOVERY CHECK
# ============================================================

objects_to_check = [
    "test_probabilities_35",
    "test_predictions_35",
    "test_labels_35",
    "dl_test",
    "quantized_interpreter_30",
    "input_details_30",
    "output_details_30"
]

print("=" * 80)
print("FROZEN NOTEBOOK 5 — OBJECT RECOVERY CHECK")
print("=" * 80)

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]
        try:
            print(f"{name:<30} AVAILABLE | type={type(obj).__name__} | length={len(obj):,}")
        except TypeError:
            print(f"{name:<30} AVAILABLE | type={type(obj).__name__}")
    else:
        print(f"{name:<30} NOT AVAILABLE")

print("=" * 80)
print("NO MODEL INFERENCE WAS PERFORMED.")
print("NO MODEL WAS MODIFIED.")
print("=" * 80)

FROZEN NOTEBOOK 5 — OBJECT RECOVERY CHECK
test_probabilities_35          AVAILABLE | type=ndarray | length=65,581
test_predictions_35            AVAILABLE | type=ndarray | length=65,581
test_labels_35                 AVAILABLE | type=ndarray | length=65,581
dl_test                        AVAILABLE | type=DataFrame | length=65,581
quantized_interpreter_30       AVAILABLE | type=Interpreter
input_details_30               AVAILABLE | type=list | length=1
output_details_30              AVAILABLE | type=list | length=1
NO MODEL INFERENCE WAS PERFORMED.
NO MODEL WAS MODIFIED.


In [ ]:


import numpy as np
import pandas as pd
import hashlib
from pathlib import Path

# ------------------------------------------------------------
# Verify frozen objects exist
# ------------------------------------------------------------

required_objects = [
    "test_probabilities_35",
    "test_predictions_35",
    "test_labels_35",
    "dl_test"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required frozen objects are missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Extract participant IDs from the original held-out test set
# ------------------------------------------------------------

test_participant_ids_35 = (
    dl_test["ParticipantID"]
    .to_numpy()
)

# ------------------------------------------------------------
# Strict alignment checks
# ------------------------------------------------------------

n_test = len(test_labels_35)

if len(test_probabilities_35) != n_test:
    raise RuntimeError("Probability count does not match test labels.")

if len(test_predictions_35) != n_test:
    raise RuntimeError("Prediction count does not match test labels.")

if len(test_participant_ids_35) != n_test:
    raise RuntimeError(
        "Participant ID count does not match test labels."
    )

# ------------------------------------------------------------
# Verify the predictions still correspond to threshold = 0.5
# ------------------------------------------------------------

reconstructed_predictions_35 = (
    test_probabilities_35 >= 0.5
).astype(np.int32)

if not np.array_equal(
    reconstructed_predictions_35,
    test_predictions_35
):
    raise RuntimeError(
        "Frozen predictions do not match the recorded "
        "0.5 threshold."
    )

# ------------------------------------------------------------
# Export frozen analysis artifact
# ------------------------------------------------------------

output_path = Path(
    "Notebook5_Frozen_Test_Predictions.npz"
)

np.savez(
    output_path,
    test_probabilities=test_probabilities_35,
    test_predictions=test_predictions_35,
    test_labels=test_labels_35,
    participant_ids=test_participant_ids_35,
    threshold=np.array(0.5)
)

# ------------------------------------------------------------
# Generate SHA-256 hash for reproducibility
# ------------------------------------------------------------

file_hash = hashlib.sha256(
    output_path.read_bytes()
).hexdigest()

# ------------------------------------------------------------
# Verification output
# ------------------------------------------------------------

print("=" * 90)
print("FROZEN NOTEBOOK 5 TEST PREDICTIONS EXPORTED")
print("=" * 90)

print(f"Output file       : {output_path.resolve()}")
print(f"Test samples      : {n_test:,}")
print(
    f"Participants      : "
    f"{dl_test['ParticipantID'].nunique()}"
)
print(f"Threshold         : 0.5")
print(
    f"Prediction arrays : "
    f"{len(test_predictions_35):,}"
)
print(f"SHA-256           : {file_hash}")

print("\nVerification:")
print("- No model inference was performed.")
print("- No model was modified.")
print("- No retraining was performed.")
print("- No optimisation was performed.")
print("- No test-set decision was made.")
print("- Existing frozen predictions were only exported.")

print("=" * 90)

FROZEN NOTEBOOK 5 TEST PREDICTIONS EXPORTED
Output file       : /content/Notebook5_Frozen_Test_Predictions.npz
Test samples      : 65,581
Participants      : 5
Threshold         : 0.5
Prediction arrays : 65,581
SHA-256           : 71d2f1b835febcaea41356e84db2843ea8c02a49cab67a98baebb292bf6de83b

Verification:
- No model inference was performed.
- No model was modified.
- No retraining was performed.
- No optimisation was performed.
- No test-set decision was made.
- Existing frozen predictions were only exported.


# Notebook 5 — Final Summary


This notebook completed the final analysis and interpretation of the lightweight ECG classification experiments developed across the dissertation workflow.

The optimisation stage investigated whether the frozen Lightweight CNN baseline could be made more computationally efficient while retaining acceptable predictive performance. Pruning and post-training quantisation were evaluated using the training and validation data, while the held-out test set remained inaccessible until the final model had been selected.

The validation-based comparison identified the **Quantised CNN** as the final optimised configuration because it provided the strongest measured efficiency improvements while maintaining broadly comparable predictive performance to the frozen baseline. The quantised model achieved a **34.10% reduction in model size** and a **45.80% reduction in measured prediction time**, corresponding to an approximately **1.845× inference speed-up**.

The final held-out evaluation was then performed only after model selection. On the five held-out test participants, the selected Quantised CNN achieved:

- Accuracy: **0.6569**
- Precision: **0.8993**
- Recall: **0.6258**
- F1-score: **0.7380**
- ROC-AUC: **0.7736**

These results demonstrate that computational efficiency can be improved through software-based model optimisation, although the optimisation does not uniformly improve predictive performance. The results therefore illustrate an important **performance–efficiency trade-off** rather than suggesting that compression automatically produces a more accurate model.

The experiments also show that pruning and quantisation have different effects. The 60% pruned CNN produced the strongest validation F1-score (**0.8405**) and recall (**0.7998**), whereas the quantised CNN provided the most clearly demonstrated reductions in model size and prediction time. The final model was therefore selected according to the predefined validation-based efficiency objective rather than by using held-out test performance.

## Scientific Interpretation

The findings provide evidence that lightweight optimisation can reduce computational requirements while retaining useful ECG classification capability. However, the results should be interpreted cautiously because the held-out test set contains only five participants. The observed performance therefore represents evidence from the defined participant-level test split rather than definitive population-level generalisation.

Furthermore, inference-time measurements were obtained in the experimental software environment and should not be interpreted as direct evidence of performance on physical wearable hardware. The results indicate **potential suitability for resource-constrained deployment**, rather than demonstrating deployment readiness.

